# PeerBERT: Explainable Classification of AI-Generated Peer Reviews

**This notebook contains the whole project.** It writes the code into the runtime, runs every
experiment on the **original datasets** with the **original models**, and shows the results
with figures and a short analysis after each experiment.

| # | Experiment | What it produces |
|---|---|---|
| 1 | Data loading, paper-level splits, leakage audit | dataset statistics, audit verdict, length-shortcut check |
| 2 | PeerBERT training (ModernBERT-base, 3 seeds per dataset) | training curves |
| 3 | PeerBERT evaluation | metrics with 95% CIs, confusion matrices, ROC/PR, per-generator, per-venue, length terciles, calibration, errors |
| 4 | Eight baselines on the shared test subset | comparison table, ROC overlay, TPR at low FPR, paired significance tests |
| 5 | Ablation (loss variants, R-Drop, dropout, CoCoDet-style margin loss) | validation Macro-F1 per variant |
| 6 | Cross-dataset transfer (no retraining) | in-domain vs. transfer performance |
| 7 | Explainability (Integrated Gradients + attention) | faithfulness vs. random, top tokens, highlighted reviews |
| 8 | Final report | all tables (CSV/LaTeX) and figures (PNG/PDF) |

Datasets: [IntelLabs/AI-Peer-Review-Detection-Benchmark](https://huggingface.co/datasets/IntelLabs/AI-Peer-Review-Detection-Benchmark)
(binary) and [Sem-Detect/ML_Conferences-Peer-Reviews](https://huggingface.co/datasets/Sem-Detect/ML_Conferences-Peer-Reviews)
(human / ai / rewrite). Model: [answerdotai/ModernBERT-base](https://huggingface.co/answerdotai/ModernBERT-base).

## How to run
1. **Runtime -> Change runtime type -> T4 GPU** (an A100/L4 is faster).
2. **Runtime -> Run all**, or run the cells one by one.
3. If Colab disconnects, reconnect and **Run all** again. Finished work is skipped, and training
   resumes from its last finished epoch, because everything is saved to Google Drive.

The full run takes many GPU hours (training 2 datasets x 3 seeds, plus 8 baselines). You can
switch experiments on and off in the **Settings** cell, and run them across several sessions.

## 0. Settings

In [ ]:
# ---- what to run ------------------------------------------------------------------------
RUN = {
    "prepare":   True,   # 1. load datasets, split by paper, leakage audit
    "smoke":     True,   # quick 1-epoch training check before the real training
    "train":     True,   # 2. PeerBERT, all seeds, both datasets
    "evaluate":  True,   # 3. PeerBERT evaluation
    "baselines": True,   # 4. eight baselines
    "ablation":  True,   # 5. ablation variants (single seed, validation selection)
    "cross":     True,   # 6. cross-dataset transfer
    "xai":       True,   # 7. explainability
}
DATASETS = ["intellabs", "semdetect"]
SEEDS = None          # None = seeds from configs/train.yaml ([42, 43, 44]); e.g. [42] for a faster first run
N_BOOT = 1000         # bootstrap resamples for confidence intervals
XAI_PER_GROUP = 25    # explained reviews per (true class, correct/incorrect) group

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    WORK = "/kaggle/working/PeerBERT_outputs"
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/PeerBERT_outputs"
    except ImportError:                       # local Jupyter
        WORK = os.path.abspath("PeerBERT_outputs")
CODE = "/content/PeerBERT" if os.path.exists("/content") else os.path.abspath("PeerBERT_code")
os.makedirs(WORK, exist_ok=True)
os.makedirs(CODE, exist_ok=True)
os.environ["PRAD_WORK_DIR"] = WORK
print("code    ->", CODE)
print("outputs ->", WORK)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "no GPU found: select a GPU runtime")

### Install dependencies
torch is preinstalled on Colab. Optionally, add a Hugging Face token as a Colab secret named
`HF_TOKEN` (key icon on the left) for faster downloads without rate limits.

In [ ]:
!pip install -q "transformers>=4.48.0" "huggingface_hub>=0.22" "accelerate>=0.26" "captum>=0.7.0" pyarrow scikit-learn scipy matplotlib pyyaml packaging
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("no HF_TOKEN secret: anonymous downloads")
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())

## Project code
The next cells write the project code (configs, `src/`, `scripts/`) into the runtime.

**To change a setting, edit the config cells** (`configs/config.yaml`, `configs/train.yaml`,
`configs/baselines.yaml`) and re-run them before the experiment.

In [ ]:
for d in ["configs", "scripts", "src/data", "src/models", "src/evaluation", "src/xai", "src/baselines/statistical",
          "src/baselines/zero_shot", "src/baselines/neural_detectors"]:
    os.makedirs(os.path.join(CODE, d), exist_ok=True)
%cd {CODE}

In [ ]:
%%writefile configs/baselines.yaml
# Baselines (scripts/06_run_baselines.py). All methods are compared on the manifests'
# test_baseline subset (nested inside the model test set). Sem-Detect: human/ai rows only
# for metrics; rewrite rows give a descriptive "fraction flagged as AI" diagnostic.
# Thresholds are calibrated on a whole-paper VALIDATION sample only.

run: [logrank, gltr, detectllm_lrr, mfd, fast_detectgpt, binoculars, radar, dnagpt]

common:
  dtype: float16              # LM inference dtype on GPU (float32 on CPU)
  max_tokens: 512             # tokens per review scored by every LM-based detector
  batch_size: 4
  val_rows: 600               # whole-paper validation sample for threshold calibration
  train_rows: 2000            # whole-paper train sample used to FIT gltr/mfd (disclosed)
  threshold_objective: macro_f1
  gpu_id: 0

# One shared forward pass of this LM gives LogRank, GLTR, DetectLLM-LRR, MFD features and
# (same reference = scoring model) the analytic Fast-DetectGPT criterion.
token_stats:
  model: EleutherAI/gpt-neo-2.7B   # a scoring model used in the Fast-DetectGPT repo (model.py)

gltr:
  rank_buckets: [10, 100, 1000]    # top-10 / 10-100 / 100-1000 / >1000 (1-indexed ranks)

mfd:
  head: logreg                     # logreg ("MFD") | mlp ("MFD-M", 4-16-16-16-1)
  mlp_hidden: [16, 16, 16]

fast_detectgpt:
  reference_model: same            # "same" = token_stats.model (white-box setting)
  scoring_model: same

binoculars:
  # Paper/official pair: tiiuae/falcon-7b (observer) + tiiuae/falcon-7b-instruct (performer).
  # Two 7B models do not fit a 16 GB T4 in fp16, so a smaller same-tokenizer base/instruct
  # pair is used by default (DEVIATION, disclosed). SmolLM2 is not one of the IntelLabs
  # generator families (claude, gemini, gpt-4o, llama, qwen).
  use_paper_pair: false
  paper_observer: tiiuae/falcon-7b
  paper_performer: tiiuae/falcon-7b-instruct
  observer: HuggingFaceTB/SmolLM2-1.7B
  performer: HuggingFaceTB/SmolLM2-1.7B-Instruct
  batch_size: 2

radar:
  model: TrustSafeAI/RADAR-Vicuna-7B   # RoBERTa-based detector checkpoint
  max_length: 512
  # The released label order could not be verified, so no index is assumed: the output
  # column that tracks "AI" is chosen on the VALIDATION sample only and recorded.
  batch_size: 16

dnagpt:
  regenerator: EleutherAI/gpt-neo-2.7B  # LOCAL LM (deviation: paper regenerates with the source LLM)
  gamma: 0.5                  # truncation ratio (characters)
  k: 5                        # regenerations per review (official demo uses 30; compute-capped)
  max_prompt_tokens: 384
  max_new_tokens: 200
  temperature: 0.7
  ngram_min: 4
  ngram_max: 25
  wscore: false               # optional probability-based score
  batch_size: 2

In [ ]:
%%writefile configs/config.yaml
project:
  name: peer-review-ai-detection
  seed: 42

paths:
  # Local: "." . On Kaggle: set env var PRAD_WORK_DIR=/kaggle/working (overrides this).
  work_dir: "."
  audit_dir: outputs/audit
  manifest_dir: outputs/manifests
  model_dir: outputs/models
  results_dir: outputs/results
  cache_dir: outputs/cache            # restart-safe parquet cache of loaded subsets

# Config keys use `semdetect`; output folders use `sem_detect`.
dataset_dirs:
  intellabs: intellabs
  semdetect: sem_detect

data:
  intellabs:
    repo_id: IntelLabs/AI-Peer-Review-Detection-Benchmark
    train_source_subset: calibration   # dataset card calls it "Training"
    test_subset: test
    extended_subset: extended          # robustness only, never used for training
    use_extended: false
    # NOT assumed: confirm with scripts/00_inspect_datasets.py before using.
    ai_label_value: 1
    val_fraction_papers: 0.15
    train_max_rows: 24000              # null = use all; compute-driven cap
    val_max_rows: null                 # null = full validation split
    model_eval_max_rows: 30000         # null = full official test
    baseline_eval_max_rows: 600
    extended_max_rows: 30000           # null = full extended (after overlap exclusion)
    # Narrative review fields, joined with a blank line in THIS order.
    text_fields_allow:
      - review
      - summary_of_the_paper
      - summary
      - strength_and_weaknesses          # ICLR 2023 form
      - strengths_and_weaknesses         # NeurIPS 2022 form
      - strengths
      - weaknesses
      - main_review
      - clarity_quality_novelty_and_reproducibility
      - questions
      - limitations
      - limitations_and_societal_impact
      - summary_of_the_review
    # Never used as text (scores, flags, metadata, label-leaking fields).
    text_fields_never:
      - label
      - custom_id
      - PaperId
      - review-id
      - conference
      - title                           # review title in older ICLR forms (short, not review body)
      - rating
      - confidence
      - correctness
      - technical_novelty_and_significance
      - empirical_novelty_and_significance
      - recommendation
      - soundness
      - presentation
      - contribution
      - flag_for_ethics_review
      - needs_ethics_review
      - ethics_flag
      - ethics_review_area
      - time_spent_reviewing
      - code_of_conduct               # leaks the label (length differs by label)
    # Flattened sub-fields (e.g. strengths.strength_1) exist only in some AI files: excluded
    # from text and reported. Any "<allow-listed field>.<name>" column is treated the same way.
    text_field_never_prefixes: [strengths., weaknesses., questions., limitations.]
    # Schema guard: an unlisted string column with mean length above this is an error.
    unknown_narrative_min_mean_chars: 150
  semdetect:
    repo_id: Sem-Detect/ML_Conferences-Peer-Reviews
    config_name: reviews
    label_map: {human: 0, ai: 1, rewrite: 2}
    text_field: clean_review
    # Columns the loader cannot work without (it raises if any is missing in the real files).
    required_columns: [paper_id, conference, year, class, clean_review, split]
    # Used when present: review_id (row ids), submission_id (overlap cross-check),
    # og_review_id (metadata only). Missing ones are reported, never invented.
    optional_columns: [review_id, submission_id, og_review_id]
    val_fraction_papers: 0.10
    train_max_rows: null
    val_max_rows: null
    model_eval_max_rows: null
    baseline_eval_max_rows: 600

audit:
  # Fraction of eval rows whose exact text also occurs in train: above -> FAIL, else WARN.
  exact_dup_cross_fail_fraction: 0.001
  near_dup_sample_size: 2000          # per partition, TF-IDF cosine (information only)
  near_dup_cosine: 0.90
  length_auroc_warn: 0.75             # max(AUROC, 1-AUROC) of length-only detector

inspection:
  sample_conf_years: [ICLR2021, ICLR2022, ICLR2023, ICLR2024, NeurIPS2021, NeurIPS2024]
  generator_folder: gpt4o
  compare_generator_folder: claude

In [ ]:
%%writefile configs/train.yaml
# PeerBERT (ModernBERT-base) training configuration. Used by scripts 03, 04, 05, 07, 08.
model:
  name: answerdotai/ModernBERT-base
  max_length: 1024            # ModernBERT supports up to 8192; truncation rate is reported
  truncation: head            # head | head_tail (keep start and end of long reviews)
  head_tokens: 256            # head_tail only: tokens kept from the start
  # ModernBERT's default dropouts are 0.0 (verified at runtime from the loaded config).
  # dropout_on=true sets every *_dropout field that exists in the config to `dropout`.
  dropout_on: true
  dropout: 0.1
  head: linear                # linear | cosine (cosine is required by the csm loss)
  cosine_scale: 30.0          # s in CoCoDet's cosine head

training:
  seeds: [42, 43, 44]         # final configuration: mean +/- std over these seeds
  epochs: 3
  lr: 3.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.06
  batch_size: 4               # per step; effective batch = batch_size * grad_accum = 32
  grad_accum: 8               # sized for a 16 GB T4 with R-Drop (two forward passes)
  eval_batch_size: 16
  gradient_checkpointing: false   # set true if a GPU runs out of memory (slower, less memory)
  max_grad_norm: 1.0
  early_stopping_patience: 2  # epochs without val Macro-F1 improvement
  max_bad_steps: 20           # consecutive non-finite losses before aborting
  num_workers: 2
  amp: auto                   # auto: bf16 if supported, else fp16 + GradScaler; "off" for fp32
  gpu_id: 0                   # single GPU; set before torch is imported

loss:
  type: weighted_ce           # ce | weighted_ce | label_smoothing | focal | csm
  label_smoothing: 0.05
  focal_gamma: 2.0
  csm_base_margin: 0.25       # CoCoDet m_base
  csm_cost_margin: 0.25       # CoCoDet m_cost (human <-> ai confusions)
  rdrop_alpha: 1.0            # 0 disables R-Drop (L_cons)

threshold:
  objective: macro_f1         # val-calibrated threshold objective: macro_f1 | youden

# Ablations (scripts/07_run_ablation.py), selected on VALIDATION Macro-F1, one seed.
ablation:
  seed: 42
  variants:
    - {name: ce_unweighted,     loss: {type: ce,              rdrop_alpha: 0.0}}
    - {name: ce_weighted,       loss: {type: weighted_ce,     rdrop_alpha: 0.0}}
    - {name: label_smoothing,   loss: {type: label_smoothing, rdrop_alpha: 0.0}}
    - {name: focal_weighted,    loss: {type: focal,           rdrop_alpha: 0.0}}
    - {name: rdrop_0.5,         loss: {type: weighted_ce,     rdrop_alpha: 0.5}}
    - {name: rdrop_1.0,         loss: {type: weighted_ce,     rdrop_alpha: 1.0}}
    - {name: dropout_off,       loss: {type: weighted_ce,     rdrop_alpha: 0.0}, model: {dropout_on: false}}
    - {name: csm_cosine,        loss: {type: csm,             rdrop_alpha: 0.0}, model: {head: cosine}}

In [ ]:
%%writefile src/baselines/neural_detectors/radar.py
"""
RADAR (Hu et al., NeurIPS 2023), checkpoint TrustSafeAI/RADAR-Vicuna-7B: a RoBERTa-based
sequence classifier released by the authors. Pretrained detector: no training on our data.

Label order: the model card does not document it and the official demo could not be
fetched -> UNVERIFIED, so NO index is assumed. The output column whose probability ranks
validation AI reviews above human reviews (higher validation AUROC) is used; this is
decided on the VALIDATION sample only, recorded, and disclosed.
"""

import numpy as np


def load(model_name):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()
    return model, tok, device


def probabilities(model, tok, texts, device, max_length=512, batch_size=16):
    import torch
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = None
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True, truncation=True,
                  max_length=max_length).to(device)
        with torch.inference_mode():
            p = torch.softmax(model(**enc).logits.float(), -1).cpu().numpy()
        if out is None:
            out = np.empty((len(texts), p.shape[1]))
        out[idx] = p
    return out


def orient(val_proba, y_val):
    """Choose the AI column on validation only. Returns (index, decision dict)."""
    from sklearn.metrics import roc_auc_score
    if val_proba.shape[1] != 2:
        raise ValueError(f"RADAR expected 2 outputs, got {val_proba.shape[1]}")
    aucs = [float(roc_auc_score(y_val, val_proba[:, i])) for i in (0, 1)]
    idx = int(np.argmax(aucs))
    return idx, {"val_auroc_per_output": aucs, "used_ai_index": idx, "n_val": int(len(y_val)),
                 "status": "label order UNVERIFIED; AI column chosen on validation only"}

In [ ]:
%%writefile src/baselines/registry.py
"""
Registry of the 8 baselines: paper, code source, detector type, what was verified in this
project, and disclosed deviations. Types match the comparison tables:
  zero-shot detector   - no training on our data (threshold calibrated on validation)
  pretrained detector  - released detector checkpoint, not trained on our data
  trained classifier   - fit on our TRAIN split (here: small feature-based heads)
All detectors return scores where LARGER = MORE LIKELY AI.
"""

BASELINES = {
    "logrank": {
        "display": "LogRank",
        "type": "zero-shot detector",
        "paper": "Ippolito et al., ACL 2020 (rank-based detection); Log-Rank as in DetectGPT/DetectLLM baselines",
        "code": "mbzuai-nlp/DetectLLM baselines/rank.py (rank 1-indexed, log, mean)",
        "score": "-mean log-rank",
        "verified": ["1-indexed ranks, log, mean over tokens (DetectLLM rank.py, fetched)"],
        "deviations": ["scoring LM = token_stats.model (disclosed)"],
        "needs": "token_stats",
    },
    "gltr": {
        "display": "GLTR",
        "type": "trained classifier",
        "paper": "Gehrmann et al., ACL 2019 (demo)",
        "code": "this repo (rank buckets) + scikit-learn logistic regression",
        "score": "P(AI) from logistic regression on 4 rank-bucket fractions",
        "verified": ["bucket edges 10/100/1000 as in the GLTR visualisation"],
        "deviations": ["LR fit on a capped whole-paper TRAIN sample (common.train_rows, disclosed)"],
        "needs": "token_stats",
    },
    "detectllm_lrr": {
        "display": "DetectLLM-LRR",
        "type": "zero-shot detector",
        "paper": "Su et al., 2023 (DetectLLM)",
        "code": "mbzuai-nlp/DetectLLM baselines/all_baselines.py: -ll / logrank",
        "score": "mean NLL / mean log-rank",
        "verified": ["ratio and sign from the official code (fetched)"],
        "deviations": ["epsilon 1e-6 added to the denominator (disclosed)"],
        "needs": "token_stats",
    },
    "mfd": {
        "display": "MFD",
        "type": "trained classifier",
        "paper": "Xiang & Wu, 2023, Research Square preprint (not peer reviewed; no official repo found)",
        "code": "this repo",
        "score": "P(AI) from LR (MFD) or 4-16-16-16-1 MLP (MFD-M) on 4 LM features",
        "verified": ["feature definitions as stated in the thesis protocol; preprint not re-read in this session"],
        "deviations": ["preprint internally inconsistent about the head; LR used as primary (disclosed)",
                       "fit on capped TRAIN sample"],
        "needs": "token_stats",
    },
    "fast_detectgpt": {
        "display": "Fast-DetectGPT",
        "type": "zero-shot detector",
        "paper": "Bao et al., ICLR 2024",
        "code": "baoguangsheng/fast-detect-gpt scripts/fast_detect_gpt.py::get_sampling_discrepancy_analytic",
        "score": "(sum ll - sum mean_ref) / sqrt(sum var_ref), analytic",
        "verified": ["analytic formula and model ids (fetched from the official repo)"],
        "deviations": ["reference = scoring = token_stats.model (surrogate, disclosed)"],
        "needs": "token_stats",
    },
    "binoculars": {
        "display": "Binoculars",
        "type": "zero-shot detector",
        "paper": "Hans et al., ICML 2024",
        "code": "ahans30/Binoculars binoculars/detector.py + metrics.py",
        "score": "-(log-ppl_performer / x-ppl(observer, performer))",
        "verified": ["ppl uses PERFORMER logits; x_ppl = CE(observer probs, performer log-probs); "
                     "default pair falcon-7b / falcon-7b-instruct (fetched)"],
        "deviations": ["smaller same-tokenizer pair by default on 16 GB GPUs (flagged); "
                       "published thresholds not reused"],
        "needs": "own_models",
    },
    "radar": {
        "display": "RADAR",
        "type": "pretrained detector",
        "paper": "Hu et al., NeurIPS 2023",
        "code": "HF checkpoint TrustSafeAI/RADAR-Vicuna-7B",
        "score": "P(AI) from the released classifier",
        "verified": [],
        "deviations": ["label order UNVERIFIED; orientation decided on VALIDATION only (disclosed)"],
        "needs": "own_models",
    },
    "dnagpt": {
        "display": "DNA-GPT",
        "type": "zero-shot detector",
        "paper": "Yang et al., ICLR 2024",
        "code": "Xianjun-Yang/DNA-GPT (demo only); paper BScore implemented here",
        "score": "BScore (n-gram overlap, n=4..25, weight n log n, length-normalised)",
        "verified": ["official demo uses gamma=0.5, temperature 0.7 (fetched); demo formula differs from paper"],
        "deviations": ["LOCAL regenerator LM instead of the source LLM", "K regenerations compute-capped"],
        "needs": "own_models",
    },
}


def get(name):
    if name not in BASELINES:
        raise KeyError(f"unknown baseline '{name}'. Known: {sorted(BASELINES)}")
    return BASELINES[name]

In [ ]:
%%writefile src/baselines/statistical/detectllm_lrr.py
"""
DetectLLM-LRR (Su et al., 2023, "DetectLLM: Leveraging Log Rank Information for Zero-Shot
Detection of Machine-Generated Text"). Official code (mbzuai-nlp/DetectLLM,
baselines/all_baselines.py): LRR = -mean_ll / mean_logrank, i.e. mean NLL / mean log-rank.
Larger = more likely machine-generated. Zero-shot.

Deviation (disclosed): epsilon added to the denominator, because mean log-rank is 0 when
every token is the model's top-1 prediction.
"""


EPS = 1e-6


def score(stats, eps=EPS):
    nll = -stats["mean_ll"].to_numpy(dtype=float)
    return nll / (stats["mean_logrank"].to_numpy(dtype=float) + eps)

In [ ]:
%%writefile src/baselines/statistical/gltr.py
"""
GLTR (Gehrmann et al., ACL 2019 demo). Features: fractions of tokens whose 1-indexed rank
under the scoring LM is in top-10 / 11-100 / 101-1000 / >1000. A logistic regression on
these 4 features is fit on a whole-paper sample of the TRAIN split only (row cap disclosed
in baselines.yaml common.train_rows). Output = P(AI). Trained (feature-based) detector.
"""

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["gltr_top10", "gltr_10_100", "gltr_100_1000", "gltr_over_1000"]


def fit(stats_train, y_train, seed=42):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=seed))
    return clf.fit(stats_train[FEATURES].to_numpy(dtype=float), y_train)


def score(clf, stats):
    return clf.predict_proba(stats[FEATURES].to_numpy(dtype=float))[:, 1]

In [ ]:
%%writefile src/baselines/statistical/logrank.py
"""
LogRank (Log-Rank baseline; Ippolito et al., ACL 2020 rank-based detection; as implemented in
the DetectGPT/DetectLLM baseline code): score = - mean log-rank of the observed tokens
under a scoring LM (1-indexed ranks). Larger = more likely machine-generated. Zero-shot.
"""


def score(stats):
    return -stats["mean_logrank"].to_numpy(dtype=float)

In [ ]:
%%writefile src/baselines/statistical/mfd.py
"""
MFD - multi-feature detector (Xiang & Wu, 2023, Research Square preprint; NOT peer reviewed;
no official code found). Features from one scoring-LM pass:
  mean log-likelihood, mean log-rank, mean entropy, LLM-Deviation = mean (log rank)^2.
Head (fit on a whole-paper TRAIN sample only):
  logreg  "MFD"   - the variant named MFD in the preprint's ablation (primary)
  mlp     "MFD-M" - the 4-16-16-16-1 network described in the preprint's text
The preprint is internally inconsistent about which head is "MFD"; both are implemented
and the choice is disclosed. Features are standardised before either head.
"""

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["mean_ll", "mean_logrank", "mean_entropy", "mean_logrank_sq"]


def fit(stats_train, y_train, head="logreg", hidden=(16, 16, 16), seed=42):
    if head == "logreg":
        model = LogisticRegression(max_iter=2000, random_state=seed)
    elif head == "mlp":
        model = MLPClassifier(hidden_layer_sizes=tuple(hidden), max_iter=2000, early_stopping=True,
                              random_state=seed)
    else:
        raise ValueError(f"unknown MFD head '{head}'")
    return make_pipeline(StandardScaler(), model).fit(stats_train[FEATURES].to_numpy(dtype=float), y_train)


def score(clf, stats):
    return clf.predict_proba(stats[FEATURES].to_numpy(dtype=float))[:, 1]

In [ ]:
%%writefile src/baselines/statistical/token_stats.py
"""
Shared token-statistics pass with one causal LM (one forward pass per review).

Per review (over the scored positions t = 2..T; the first token has no prediction):
  n_tokens
  mean_ll          mean log p(x_t | x_<t)
  mean_logrank     mean log(rank_t), rank 1-indexed (DetectLLM baselines/rank.py)
  mean_logrank_sq  mean log(rank_t)^2          (MFD "LLM-Deviation")
  mean_entropy     mean of -sum_v p log p      (DetectLLM baselines/entropy.py)
  gltr_*           fractions of tokens with rank <= 10, 11-100, 101-1000, > 1000
  fdg_*            sums for the analytic Fast-DetectGPT criterion with the same model as
                   reference and scorer: sum log p(x_t), sum E_p[log p], sum Var_p[log p]

Results are cached per (model, max_tokens) in outputs/cache/baselines/ and only missing
row_ids are computed, so the pass is restart-safe and shared by all detectors.
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd

STAT_COLS = ["n_tokens", "mean_ll", "mean_logrank", "mean_logrank_sq", "mean_entropy",
             "gltr_top10", "gltr_10_100", "gltr_100_1000", "gltr_over_1000",
             "fdg_sum_ll", "fdg_sum_mean_ref", "fdg_sum_var_ref"]


def slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


def dtype_kwarg(torch_dtype):
    """
    from_pretrained kwargs that work across Colab images:
    - `dtype=` on transformers >= 4.56, `torch_dtype=` before;
    - low_cpu_mem_usage=True whenever accelerate is installed, so a large fp32 checkpoint
      (e.g. gpt-neo-2.7B, ~10 GB) is cast while loading instead of being held twice in RAM.
    """
    import importlib.util

    import transformers
    from packaging.version import Version
    key = "dtype" if Version(transformers.__version__) >= Version("4.56.0") else "torch_dtype"
    kw = {key: torch_dtype}
    if importlib.util.find_spec("accelerate") is not None:
        kw["low_cpu_mem_usage"] = True
    return kw


def load_causal_lm(name, dtype="float16", device=None):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    device = device or torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    td = getattr(torch, dtype) if device.type == "cuda" else torch.float32
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(name, **dtype_kwarg(td)).to(device).eval()
    return model, tok, device


def release():
    """Return freed GPU memory to the allocator. Callers must `del` their own references first."""
    import gc
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def batch_stats(model, tok, texts, max_tokens, device, buckets=(10, 100, 1000)):
    import torch
    import torch.nn.functional as F
    enc = tok(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=max_tokens).to(device)
    with torch.inference_mode():
        logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits[:, :-1].float()
    labels = enc["input_ids"][:, 1:]
    mask = enc["attention_mask"][:, 1:].float()
    vocab = logits.shape[-1]
    labels = labels.clamp(max=vocab - 1)
    logp = F.log_softmax(logits, -1)
    p = logp.exp()
    tok_lp = logp.gather(-1, labels[..., None]).squeeze(-1)
    tok_logit = logits.gather(-1, labels[..., None])
    rank = (logits > tok_logit).sum(-1).float() + 1.0           # 1-indexed
    logrank = rank.log()
    ent = -(p * logp).sum(-1)
    mean_ref = (p * logp).sum(-1)                                # = -entropy
    var_ref = (p * logp.square()).sum(-1) - mean_ref.square()
    n = mask.sum(-1)
    safe = n.clamp(min=1)

    def mean(x):
        return (x * mask).sum(-1) / safe

    b1, b2, b3 = buckets
    out = {
        "n_tokens": n, "mean_ll": mean(tok_lp), "mean_logrank": mean(logrank),
        "mean_logrank_sq": mean(logrank.square()), "mean_entropy": mean(ent),
        "gltr_top10": mean((rank <= b1).float()), "gltr_10_100": mean(((rank > b1) & (rank <= b2)).float()),
        "gltr_100_1000": mean(((rank > b2) & (rank <= b3)).float()), "gltr_over_1000": mean((rank > b3).float()),
        "fdg_sum_ll": (tok_lp * mask).sum(-1), "fdg_sum_mean_ref": (mean_ref * mask).sum(-1),
        "fdg_sum_var_ref": (var_ref * mask).sum(-1),
    }
    res = {k: v.cpu().numpy() for k, v in out.items()}
    for k in STAT_COLS:
        if k != "n_tokens":
            res[k] = np.where(res["n_tokens"] > 0, res[k], np.nan)
    return res


def _append(cached, new):
    parts = [f for f in [cached, *new] if len(f)]          # skip empty frames (pandas dtype rules)
    return pd.concat(parts, ignore_index=True) if parts else cached


def compute_token_stats(frame, model_name, cache_root, max_tokens=512, batch_size=4, dtype="float16",
                        buckets=(10, 100, 1000), loaded=None, flush_every=50):
    """
    frame: rows with row_id, text. Returns frame [row_id] + STAT_COLS aligned to `frame`.
    loaded: optional (model, tok, device) to reuse an already loaded LM.
    """
    cache_root = Path(cache_root)
    cache_root.mkdir(parents=True, exist_ok=True)
    cache_p = cache_root / f"token_stats_{slug(model_name)}_{max_tokens}.parquet"
    cached = pd.read_parquet(cache_p) if cache_p.exists() else pd.DataFrame(columns=["row_id"] + STAT_COLS)
    todo = frame.loc[~frame["row_id"].isin(cached["row_id"]), ["row_id", "text"]].drop_duplicates("row_id")
    if len(todo):
        model, tok, device = loaded or load_causal_lm(model_name, dtype)
        order = todo["text"].str.len().sort_values().index        # similar lengths per batch
        todo = todo.loc[order]
        new, batches = [], 0
        for i in range(0, len(todo), batch_size):
            chunk = todo.iloc[i:i + batch_size]
            st = batch_stats(model, tok, chunk["text"].tolist(), max_tokens, device, buckets)
            new.append(pd.DataFrame({"row_id": chunk["row_id"].values, **st}))
            batches += 1
            if batches % flush_every == 0:
                cached = _append(cached, new)
                cached.to_parquet(cache_p, index=False)
                new = []
                print(f"[token_stats] {model_name}: {len(cached)} cached")
        cached = _append(cached, new)
        cached.to_parquet(cache_p, index=False)
        if loaded is None:
            del model
            release()
    stats = frame[["row_id"]].merge(cached.drop_duplicates("row_id"), on="row_id", how="left")
    if stats[STAT_COLS].isna().all(axis=1).any():
        raise RuntimeError("token stats missing for some rows after computation")
    return stats

In [ ]:
%%writefile src/baselines/zero_shot/binoculars.py
"""
Binoculars (Hans et al., ICML 2024). Verified against the official repo (ahans30/Binoculars,
binoculars/detector.py + metrics.py):
  ppl   = perplexity(encodings, performer_logits)          mean token NLL under the PERFORMER
  x_ppl = entropy(observer_logits, performer_logits, ...)  mean over tokens of
          -sum_v softmax(observer)_v * log_softmax(performer)_v
  binoculars score = ppl / x_ppl ;  LOWER = more likely AI.
This module returns -score so that larger = more likely AI (as for all detectors here).
Official pair: observer tiiuae/falcon-7b, performer tiiuae/falcon-7b-instruct. Published
thresholds are NOT reused: the threshold is calibrated on validation. Zero-shot.
"""

import numpy as np


PROBES = ("The paper proposes a novel method and evaluates it on three benchmarks.",
          "Weaknesses: the ablation study is missing, and Table 2 lacks error bars.",
          "Questions for the authors: how does the approach scale to 10^6 tokens?")


def check_tokenizers(tok_o, tok_p, texts=PROBES):
    """
    The two models must tokenise text identically (the official code also requires this).
    Instruct variants may add chat tokens, so the check compares the actual token ids
    produced for real text instead of the full vocabularies.
    """
    for t in texts:
        a = tok_o(t, add_special_tokens=False)["input_ids"]
        b = tok_p(t, add_special_tokens=False)["input_ids"]
        if a != b:
            raise ValueError("Binoculars observer and performer tokenise text differently; "
                             "choose a base/instruct pair from the same model family")


def load_pair(observer, performer, dtype="float16"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    from src.baselines.statistical.token_stats import dtype_kwarg
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    td = getattr(torch, dtype) if device.type == "cuda" else torch.float32
    tok_o, tok_p = AutoTokenizer.from_pretrained(observer), AutoTokenizer.from_pretrained(performer)
    check_tokenizers(tok_o, tok_p)
    if tok_o.pad_token is None:
        tok_o.pad_token = tok_o.eos_token
    tok_o.padding_side = "right"
    obs = AutoModelForCausalLM.from_pretrained(observer, **dtype_kwarg(td)).to(device).eval()
    perf = AutoModelForCausalLM.from_pretrained(performer, **dtype_kwarg(td)).to(device).eval()
    return obs, perf, tok_o, device


def batch_scores(obs, perf, tok, texts, device, max_tokens=512):
    """Returns (binoculars_score, ppl, x_ppl) arrays for one batch."""
    import torch
    import torch.nn.functional as F
    enc = tok(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=max_tokens).to(device)
    with torch.inference_mode():
        lo = obs(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits[:, :-1].float()
        lp = perf(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits[:, :-1].float()
    v = min(lo.shape[-1], lp.shape[-1])          # padded embedding sizes can differ by a few rows
    lo, lp = lo[..., :v], lp[..., :v]
    labels = enc["input_ids"][:, 1:].clamp(max=v - 1)
    mask = enc["attention_mask"][:, 1:].float()
    n = mask.sum(-1).clamp(min=1)
    nll = F.cross_entropy(lp.transpose(1, 2), labels, reduction="none")
    ppl = (nll * mask).sum(-1) / n
    x = -(F.softmax(lo, -1) * F.log_softmax(lp, -1)).sum(-1)
    x_ppl = (x * mask).sum(-1) / n
    return (ppl / x_ppl).cpu().numpy(), ppl.cpu().numpy(), x_ppl.cpu().numpy()


def score(texts, obs, perf, tok, device, max_tokens=512, batch_size=2):
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = np.empty(len(texts))
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        b, _, _ = batch_scores(obs, perf, tok, [texts[j] for j in idx], device, max_tokens)
        out[idx] = b
    return -out

In [ ]:
%%writefile src/baselines/zero_shot/dnagpt.py
"""
DNA-GPT (Yang et al., ICLR 2024): divergent n-gram analysis.
1. Truncate each review at gamma (characters): prefix X, original continuation Y0.
2. Regenerate K continuations S_k of X with a LOCAL LM (deviation: the paper regenerates
   with the suspected source LLM; disclosed).
3. BScore = (1/K) sum_k sum_{n=n0..N} f(n) |G_n(S_k) & G_n(Y0)| / (|S_k| * |G_n(Y0)|),
   f(n) = n log n, n0 = 4, N = 25, G_n = set of word n-grams, |S_k| = words in S_k.
   Larger = more overlap = more likely machine-generated. Zero-shot.
Notes (disclosed): the official demo script (DNA-GPT-dist.py) differs from the paper
formula (n from 1, index-based weights, different normalisation, 30 regenerations); the
paper formula above is used. Optional WScore (white-box): mean_k of the per-token mean
log-likelihood difference log p(Y0|X) - log p(S_k|X) under the regenerator.
Regenerations are cached per row_id (restart-safe).
"""

import json
import math
import re
from pathlib import Path

import numpy as np

WORD = re.compile(r"\w+", re.UNICODE)


def words(text):
    return WORD.findall(text.lower())


def ngrams(tokens, n):
    return {tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)}


def bscore(original, regenerations, n_min=4, n_max=25):
    y0 = words(original)
    total, used = 0.0, 0
    for s in regenerations:
        sk = words(s)
        if not sk:
            continue
        acc = 0.0
        for n in range(n_min, n_max + 1):
            g0 = ngrams(y0, n)
            if not g0:
                break
            acc += n * math.log(n) * len(ngrams(sk, n) & g0) / (len(sk) * len(g0))
        total += acc
        used += 1
    return total / used if used else 0.0


def split_text(text, gamma=0.5):
    cut = int(gamma * len(text))
    return text[:cut], text[cut:]


def regenerate(model, tok, prefixes, device, k=5, max_prompt_tokens=384, max_new_tokens=200,
               temperature=0.7, seed=0):
    import torch
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    enc = tok(list(prefixes), return_tensors="pt", padding=True, truncation=True,
              max_length=max_prompt_tokens).to(device)
    torch.manual_seed(seed)
    with torch.inference_mode():
        out = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], do_sample=True, temperature=temperature, top_p=1.0, top_k=0,
                             max_new_tokens=max_new_tokens, num_return_sequences=k,
                             pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    texts = tok.batch_decode(gen, skip_special_tokens=True)
    tok.padding_side = "right"
    return [texts[i * k:(i + 1) * k] for i in range(len(prefixes))]


def mean_token_ll(model, tok, prefix, continuation, device, max_tokens=768):
    import torch
    import torch.nn.functional as F
    p_ids = tok(prefix, add_special_tokens=False)["input_ids"]
    c_ids = tok(continuation, add_special_tokens=False)["input_ids"]
    ids = (p_ids + c_ids)[-max_tokens:]
    n_c = min(len(c_ids), len(ids) - 1)
    if n_c <= 0:
        return float("nan")
    x = torch.tensor([ids], device=device)
    with torch.inference_mode():
        lp = F.log_softmax(model(x).logits[0, :-1].float(), -1)
    tgt = x[0, 1:]
    ll = lp.gather(-1, tgt[:, None]).squeeze(-1)[-n_c:]
    return float(ll.mean())


def run(frame, model, tok, device, cfg, cache_path, seed=0):
    """frame: row_id, text. Returns (bscore array, wscore array or None)."""
    cache_path = Path(cache_path)
    cache = {}
    if cache_path.exists():
        for line in cache_path.read_text(encoding="utf-8").splitlines():
            r = json.loads(line)
            cache[r["row_id"]] = r["regens"]
    todo = [(rid, t) for rid, t in zip(frame["row_id"], frame["text"]) if rid not in cache]
    bs = int(cfg.get("batch_size", 2))
    with cache_path.open("a", encoding="utf-8") as f:
        for i in range(0, len(todo), bs):
            chunk = todo[i:i + bs]
            prefixes = [split_text(t, cfg["gamma"])[0] for _, t in chunk]
            regens = regenerate(model, tok, prefixes, device, cfg["k"], cfg["max_prompt_tokens"],
                                cfg["max_new_tokens"], cfg["temperature"], seed + i)
            for (rid, _), r in zip(chunk, regens):
                cache[rid] = r
                f.write(json.dumps({"row_id": rid, "regens": r}) + "\n")
            if (i // bs) % 25 == 0:
                print(f"[dnagpt] regenerated {min(i + bs, len(todo))}/{len(todo)}")
    b, w = [], []
    for rid, t in zip(frame["row_id"], frame["text"]):
        prefix, y0 = split_text(t, cfg["gamma"])
        b.append(bscore(y0, cache[rid], cfg["ngram_min"], cfg["ngram_max"]))
        if cfg.get("wscore"):
            ll0 = mean_token_ll(model, tok, prefix, y0, device)
            w.append(float(np.nanmean([ll0 - mean_token_ll(model, tok, prefix, s, device) for s in cache[rid]])))
    return np.asarray(b), (np.asarray(w) if cfg.get("wscore") else None)

In [ ]:
%%writefile src/baselines/zero_shot/fast_detectgpt.py
"""
Fast-DetectGPT (Bao et al., ICLR 2024), ANALYTIC conditional probability curvature.
Verified against the official repo (baoguangsheng/fast-detect-gpt,
scripts/fast_detect_gpt.py::get_sampling_discrepancy_analytic):
  lp(v)    = log p_score(v | x_<t)
  mean_t   = sum_v p_ref(v | x_<t) lp(v)
  var_t    = sum_v p_ref(v | x_<t) lp(v)^2 - mean_t^2
  score    = (sum_t lp(x_t) - sum_t mean_t) / sqrt(sum_t var_t)
Larger = more likely machine-generated. Zero-shot. The old sampling approximation is not used.
Default: reference = scoring = the shared token-stats LM (white-box setting); the surrogate
model is disclosed. A different reference model can be used with score_two_models().
"""

import numpy as np


def score_from_stats(stats):
    num = stats["fdg_sum_ll"].to_numpy(dtype=float) - stats["fdg_sum_mean_ref"].to_numpy(dtype=float)
    return num / np.sqrt(np.clip(stats["fdg_sum_var_ref"].to_numpy(dtype=float), 1e-12, None))


def score_two_models(texts, ref, score_m, tok, device, max_tokens=512, batch_size=2):
    """Reference and scoring models must share the tokenizer `tok`."""
    import torch
    import torch.nn.functional as F
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(list(texts[i:i + batch_size]), return_tensors="pt", padding=True, truncation=True,
                  max_length=max_tokens).to(device)
        with torch.inference_mode():
            lr = ref(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits[:, :-1].float()
            ls = score_m(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits[:, :-1].float()
        v = min(lr.shape[-1], ls.shape[-1])
        lr, ls = lr[..., :v], ls[..., :v]
        labels = enc["input_ids"][:, 1:].clamp(max=v - 1)
        mask = enc["attention_mask"][:, 1:].float()
        lp = F.log_softmax(ls, -1)
        pref = F.softmax(lr, -1)
        ll = lp.gather(-1, labels[..., None]).squeeze(-1)
        mean = (pref * lp).sum(-1)
        var = (pref * lp.square()).sum(-1) - mean.square()
        s = ((ll - mean) * mask).sum(-1) / (var * mask).sum(-1).clamp(min=1e-12).sqrt()
        out.append(s.cpu().numpy())
    return np.concatenate(out)

In [ ]:
%%writefile src/data/common.py
"""
Shared helpers for the data layer: config, paths, seeding, labels, text,
exclusion logging and JSON output.

Every loader returns rows in the UNIFIED_COLUMNS schema so that splitting,
auditing, training and evaluation never depend on dataset-specific columns.
"""

import hashlib
import json
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[2]

UNIFIED_COLUMNS = [
    "row_id",        # unique within a dataset
    "dataset",       # intellabs | semdetect
    "partition",     # official source: calibration/test/extended or train/test
    "text",
    "label",         # int
    "label_name",
    "paper_key",     # conference_year_paperid (composite, globally unique)
    "paper_id_raw",  # naive key, information only
    "conference",
    "year",
    "conf_year",
    "generator",     # AI generator for IntelLabs AI rows, "human" for human rows
    "review_id",
    "source_file",
]


# ---------------------------------------------------------------- config/paths
def load_config(path=None):
    path = Path(path) if path else ROOT / "configs" / "config.yaml"
    if not path.exists():
        raise FileNotFoundError(f"Configuration not found: {path}")
    with path.open(encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def get_work_dir(cfg):
    """PRAD_WORK_DIR overrides paths.work_dir; relative paths resolve from the repo root."""
    wd = Path(os.environ.get("PRAD_WORK_DIR", cfg.get("paths", {}).get("work_dir", ".")))
    return wd if wd.is_absolute() else (ROOT / wd).resolve()


def resolve_path(cfg, key, default):
    p = Path(cfg.get("paths", {}).get(key, default))
    return p if p.is_absolute() else get_work_dir(cfg) / p


def dataset_dir_name(cfg, dataset):
    """Config key -> folder name (semdetect -> sem_detect)."""
    dirs = cfg.get("dataset_dirs", {"intellabs": "intellabs", "semdetect": "sem_detect"})
    if dataset not in dirs:
        raise KeyError(f"Unknown dataset '{dataset}'. Known: {sorted(dirs)}")
    return dirs[dataset]


def manifest_dir(cfg, dataset):
    return resolve_path(cfg, "manifest_dir", "outputs/manifests") / dataset_dir_name(cfg, dataset)


def audit_dir(cfg, dataset):
    return resolve_path(cfg, "audit_dir", "outputs/audit") / dataset_dir_name(cfg, dataset)


def cache_dir(cfg, dataset):
    return resolve_path(cfg, "cache_dir", "outputs/cache") / dataset_dir_name(cfg, dataset)


def config_hash(obj):
    blob = json.dumps(jsonable(obj), sort_keys=True).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:16]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


# ---------------------------------------------------------------- labels/text
def normalize_label(value):
    """Normalize CSV representations of 0/1 without silently remapping anything else."""
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return None
    value = str(value).strip()
    if value in {"0", "0.0"}:
        return 0
    if value in {"1", "1.0"}:
        return 1
    return value


def clean_text(value):
    """Minimal preprocessing: strip only. Missing values become ''."""
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return ""
    return str(value).strip()


_WS = re.compile(r"\s+")


def text_hash(text):
    """Hash for exact-duplicate detection (whitespace-collapsed, case kept)."""
    return hashlib.md5(_WS.sub(" ", text).strip().encode("utf-8")).hexdigest()


def is_text_dtype(series):
    return pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)


# ---------------------------------------------------------------- logging/IO
class ExclusionLog:
    """Every dropped row is recorded with a reason. Saved next to the manifests."""

    COLUMNS = ["dataset", "partition", "row_id", "reason", "detail"]

    def __init__(self):
        self._parts = []

    def add(self, dataset, partition, row_ids, reason, detail=""):
        row_ids = list(row_ids)
        if not row_ids:
            return
        self._parts.append(pd.DataFrame({
            "dataset": dataset, "partition": partition, "row_id": row_ids,
            "reason": reason, "detail": detail,
        }))

    def extend(self, other):
        self._parts.extend(other._parts)

    def frame(self):
        if not self._parts:
            return pd.DataFrame(columns=self.COLUMNS)
        return pd.concat(self._parts, ignore_index=True)[self.COLUMNS]

    def summary(self):
        f = self.frame()
        if f.empty:
            return {}
        return {f"{p}:{r}": int(n) for (p, r), n in f.groupby(["partition", "reason"]).size().items()}


def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if not np.isfinite(obj) else float(obj)
    if isinstance(obj, float):
        return None if not np.isfinite(obj) else obj
    if isinstance(obj, np.ndarray):
        return jsonable(obj.tolist())
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    if obj is not None and not isinstance(obj, (str, int, bool)) and pd.isna(obj):
        return None
    return obj


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(jsonable(obj), f, indent=2, ensure_ascii=False)
    return path


def load_json(path):
    with Path(path).open(encoding="utf-8") as f:
        return json.load(f)


def ensure_unified(df):
    missing = [c for c in UNIFIED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Frame is missing unified columns: {missing}")
    return df[UNIFIED_COLUMNS + [c for c in df.columns if c not in UNIFIED_COLUMNS]]

In [ ]:
%%writefile src/data/intellabs_loader.py
"""
IntelLabs/AI-Peer-Review-Detection-Benchmark loader (binary: 0 = human, 1 = AI).

Facts this loader enforces (from outputs/audit/inspection_report.json):
- CSV files at <subset>/<generator>/<Conf><Year>.<subset>.<model>.csv.
  Subsets are never merged. Folders are AI generators only: labels come from
  the `label` column, never from the folder.
- label 0 <-> custom_id starting with "human"; checked on every row of every file.
- Each review-id has exactly one human and one AI row per file.
- Human rows are replicated in every generator file of a subset: deduplicated by
  (conf_year, review-id), first copy kept (files sorted by conf_year, generator),
  every dropped copy logged, copies whose text differs counted.
- PaperId is unique only within a conference-year: paper_key = Conf_Year_PaperId.
- Review text columns differ per conference-year: text = allow-listed fields
  (config order) joined with a blank line.
- Fail-late schema guard: all schema/label problems from all files are collected
  and raised once.

The HF `datasets` auto-loader fails on this repo (schema mismatches), so files are
fetched one by one with huggingface_hub and read with pandas.
"""

import re
from collections import Counter

import numpy as np
import pandas as pd

from .common import (
    ExclusionLog, UNIFIED_COLUMNS, cache_dir, clean_text, config_hash,
    load_json, normalize_label, save_json, text_hash,
)

DATASET = "intellabs"
SUBSETS = ("calibration", "test", "extended")
FNAME_RE = re.compile(
    r"^(?P<conf>[A-Za-z]+)(?P<year>\d{4})\.(?P<subset>[A-Za-z]+)\.(?P<model>.+)\.csv$"
)
CONF_YEAR_IN_ID = re.compile(r"(iclr|neurips)[_\-.]?(\d{4})", re.IGNORECASE)
KNOWN_CONFS = ("iclr", "neurips")


class SchemaError(RuntimeError):
    """Raised once with the full list of problems across all files."""


# ---------------------------------------------------------------- hub access
def hf_lister(repo_id):
    from huggingface_hub import list_repo_files
    return list_repo_files(repo_id, repo_type="dataset")


def hf_fetcher(repo_id, path):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=repo_id, filename=path, repo_type="dataset")


def parse_path(path):
    """Return file info dict, or None if the path is not a benchmark CSV."""
    parts = path.split("/")
    if len(parts) != 3 or not path.lower().endswith(".csv"):
        return None
    subset, generator, name = parts
    m = FNAME_RE.match(name)
    if not m or m["subset"].lower() != subset.lower():
        return None
    return {
        "path": path, "subset": subset, "generator": generator,
        "conference": m["conf"], "year": int(m["year"]),
        "conf_year": f'{m["conf"]}{m["year"]}', "model": m["model"],
    }


def list_subset_files(repo_id, subset, lister=None):
    infos = [parse_path(p) for p in (lister or hf_lister)(repo_id)]
    infos = [i for i in infos if i and i["subset"] == subset]
    return sorted(infos, key=lambda i: (i["conf_year"], i["generator"], i["path"]))


# ---------------------------------------------------------------- schema
def classify_columns(df, icfg):
    allow = list(icfg["text_fields_allow"])
    never = set(icfg["text_fields_never"])
    prefixes = tuple(icfg.get("text_field_never_prefixes", []))
    min_chars = float(icfg.get("unknown_narrative_min_mean_chars", 150))

    text_fields = [c for c in allow if c in df.columns]
    prefixed, unknown_narrative, unknown_other = [], [], []
    for c in df.columns:
        if c in allow or c in never:
            continue
        # Flattened copies of an allow-listed field (e.g. strengths.strength_1) only exist in some
        # generators' files; they are excluded from the text and reported.
        if (prefixes and c.startswith(prefixes)) or ("." in c and c.split(".", 1)[0] in allow):
            prefixed.append(c)
            continue
        mean_len = df[c].astype(str).str.len().mean() if len(df) else 0.0
        (unknown_narrative if mean_len > min_chars else unknown_other).append(
            {"column": c, "mean_chars": round(float(mean_len), 1)}
        )
    return {
        "text_fields": text_fields, "prefixed_excluded": prefixed,
        "unknown_narrative": unknown_narrative, "unknown_other": unknown_other,
    }


def build_text(df, fields):
    """Join non-empty allow-listed fields with a blank line; strip only."""
    if not fields:
        return pd.Series([""] * len(df), index=df.index, dtype=object)
    cols = [df[f].map(clean_text) for f in fields]
    out = []
    for vals in zip(*cols):
        out.append("\n\n".join(v for v in vals if v))
    return pd.Series(out, index=df.index, dtype=object)


# ---------------------------------------------------------------- checks
def _conference_value_problems(values, conf, year):
    """Count `conference` values naming a different conference or year."""
    bad, unparsed = 0, 0
    for v in values:
        s = re.sub(r"[^a-z0-9]", "", str(v).lower())
        named = [k for k in KNOWN_CONFS if k in s]
        years = re.findall(r"(20\d{2})", s)
        if not named and not years:
            unparsed += 1
        elif (named and conf.lower() not in named) or (years and str(year) not in years):
            bad += 1
    return bad, unparsed


def _custom_id_conf_problems(custom_ids, conf, year):
    bad, unparsed = 0, 0
    for cid in custom_ids:
        m = CONF_YEAR_IN_ID.search(str(cid))
        if not m:
            unparsed += 1
        elif m[1].lower() != conf.lower() or m[2] != str(year):
            bad += 1
    return bad, unparsed


def load_file(info, local_path, icfg):
    """Read one CSV -> (unified frame, stats, problems). Never raises on content."""
    df = pd.read_csv(local_path, dtype=str, keep_default_na=False, na_values=[])
    path, problems = info["path"], []
    required = ["label", "custom_id", "PaperId", "review-id"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        return None, {"file": path, "rows": len(df)}, [f"{path}: missing required columns {missing}"]

    schema = classify_columns(df, icfg)
    if not schema["text_fields"]:
        problems.append(f"{path}: no allow-listed text field among columns {list(df.columns)}")
    for u in schema["unknown_narrative"]:
        problems.append(
            f"{path}: unrecognized narrative-looking column '{u['column']}' "
            f"(mean {u['mean_chars']} chars) - add it to text_fields_allow or text_fields_never"
        )

    labels = df["label"].map(normalize_label)
    bad_labels = labels[~labels.isin([0, 1])]
    if len(bad_labels):
        problems.append(f"{path}: {len(bad_labels)} rows with label outside {{0,1}}: "
                        f"{sorted(map(str, bad_labels.unique()))[:5]}")
    is_human_id = df["custom_id"].str.strip().str.lower().str.startswith("human")
    mismatch = int(((labels == 0) & ~is_human_id).sum() + ((labels == 1) & is_human_id).sum())
    if mismatch:
        problems.append(f"{path}: {mismatch} rows where label disagrees with custom_id prefix")

    pairs = labels.groupby(df["review-id"]).agg(lambda s: tuple(sorted(s.tolist())))
    bad_pairs = int((pairs != (0, 1)).sum())
    if bad_pairs:
        problems.append(f"{path}: {bad_pairs} review-ids without exactly one human + one AI row")

    pid_numeric = pd.to_numeric(df["PaperId"], errors="coerce")
    if pid_numeric.isna().any():
        problems.append(f"{path}: {int(pid_numeric.isna().sum())} non-integer PaperId values")

    conf, year = info["conference"], info["year"]
    stats_conf = {}
    if "conference" in df.columns:
        bad, unparsed = _conference_value_problems(df["conference"], conf, year)
        stats_conf["conference_column_unparsed"] = unparsed
        if bad:
            problems.append(f"{path}: {bad} rows whose `conference` disagrees with filename {conf}{year}")
    bad, unparsed = _custom_id_conf_problems(df["custom_id"], conf, year)
    stats_conf["custom_id_without_conf_year"] = unparsed
    if bad:
        problems.append(f"{path}: {bad} custom_ids naming a different conference-year than {conf}{year}")

    text = build_text(df, schema["text_fields"])
    lab = labels.where(labels.isin([0, 1]))
    out = pd.DataFrame({
        "row_id": f"{info['subset']}:{info['conf_year']}:" + df["custom_id"].str.strip(),
        "dataset": DATASET,
        "partition": info["subset"],
        "text": text,
        "label": lab,
        "label_name": lab.map({0: "human", 1: "ai"}),
        "paper_key": f"{conf}_{year}_" + df["PaperId"].str.strip(),
        "paper_id_raw": df["PaperId"].str.strip(),
        "conference": conf,
        "year": year,
        "conf_year": info["conf_year"],
        "generator": np.where(lab == 0, "human", info["generator"]),
        "review_id": df["review-id"].str.strip(),
        "source_file": path,
        "generator_model": np.where(lab == 0, "human", info["model"]),
        "text_hash": text.map(text_hash),
    })

    never_lengths = {}
    for c in icfg["text_fields_never"]:
        if c in df.columns and c not in required:
            lengths = df[c].astype(str).str.len()
            never_lengths[c] = {str(k): round(float(v), 1) for k, v in lengths.groupby(lab).mean().items()}

    stats = {
        "file": path, "rows": len(df), "generator": info["generator"], "model": info["model"],
        "conf_year": info["conf_year"], "columns": list(df.columns),
        "label_counts": {str(k): int(v) for k, v in lab.value_counts().items()},
        "text_fields": schema["text_fields"], "prefixed_excluded": schema["prefixed_excluded"],
        "unknown_other_columns": schema["unknown_other"],
        "empty_text_rows": int((text == "").sum()),
        "mean_text_chars_by_label": {
            str(k): round(float(v), 1) for k, v in text.str.len().groupby(lab).mean().items()
        },
        "never_field_mean_chars_by_label": never_lengths,
        **stats_conf,
    }
    return out, stats, problems


# ---------------------------------------------------------------- subset load
def _dedupe_humans(df, log, subset):
    human = df["label"] == 0
    key = df["conf_year"] + "|" + df["review_id"]
    dup = human & key.duplicated(keep="first")
    first_hash = df.loc[human & ~dup].set_index(key[human & ~dup])["text_hash"]
    first_paper = df.loc[human & ~dup].set_index(key[human & ~dup])["paper_key"]
    dropped = df.loc[dup]
    differs = dropped["text_hash"].values != first_hash.loc[key[dup]].values
    paper_differs = dropped["paper_key"].values != first_paper.loc[key[dup]].values
    log.add(DATASET, subset, dropped.loc[~differs, "row_id"], "human_duplicate_copy", "text_identical")
    log.add(DATASET, subset, dropped.loc[differs, "row_id"], "human_duplicate_copy", "text_differs_from_first_copy")
    stats = {
        "human_rows_before_dedupe": int(human.sum()),
        "human_duplicate_copies_dropped": int(dup.sum()),
        "human_copies_with_different_text": int(differs.sum()),
        "human_copies_with_different_paper": int(paper_differs.sum()),
    }
    return df.loc[~dup].copy(), stats


def load_subset(cfg, subset, lister=None, fetcher=None, use_cache=True):
    """
    Load one official subset -> (unified frame, stats, ExclusionLog).
    Restart-safe: result cached as parquet, keyed by the loader-relevant config.
    """
    if subset not in SUBSETS:
        raise ValueError(f"Unknown IntelLabs subset '{subset}'")
    icfg = cfg["data"]["intellabs"]
    repo_id = icfg["repo_id"]
    key = config_hash({k: icfg.get(k) for k in (
        "repo_id", "text_fields_allow", "text_fields_never",
        "text_field_never_prefixes", "unknown_narrative_min_mean_chars")})

    cdir = cache_dir(cfg, DATASET)
    data_p, excl_p, stats_p = (cdir / f"{subset}.parquet", cdir / f"{subset}.exclusions.parquet",
                               cdir / f"{subset}.stats.json")
    if use_cache and data_p.exists() and stats_p.exists():
        stats = load_json(stats_p)
        if stats.get("config_hash") == key:
            print(f"[intellabs] {subset}: using cache {data_p}")
            log = ExclusionLog()
            if excl_p.exists():
                log._parts.append(pd.read_parquet(excl_p))
            return pd.read_parquet(data_p), stats, log

    files = list_subset_files(repo_id, subset, lister)
    if not files:
        raise FileNotFoundError(f"No CSV files found for subset '{subset}' in {repo_id}")
    print(f"[intellabs] {subset}: {len(files)} files")

    frames, file_stats, problems = [], [], []
    for info in files:
        local = (fetcher or hf_fetcher)(repo_id, info["path"])
        out, st, pr = load_file(info, local, icfg)
        file_stats.append(st)
        problems.extend(pr)
        if out is not None:
            frames.append(out)
    if problems:
        raise SchemaError(
            f"IntelLabs/{subset}: {len(problems)} problem(s) across {len(files)} files:\n  - "
            + "\n  - ".join(problems)
        )

    df = pd.concat(frames, ignore_index=True)
    df["label"] = df["label"].astype(int)
    log = ExclusionLog()
    df, dedupe_stats = _dedupe_humans(df, log, subset)

    empty = df["text"] == ""
    log.add(DATASET, subset, df.loc[empty, "row_id"], "empty_text")
    df = df.loc[~empty].reset_index(drop=True)

    dup_ids = df["row_id"].duplicated(keep=False)
    if dup_ids.any():
        raise SchemaError(f"IntelLabs/{subset}: {int(dup_ids.sum())} duplicate row_ids after dedupe, "
                          f"e.g. {df.loc[dup_ids, 'row_id'].head(5).tolist()}")

    counts = df["label"].value_counts()
    n_h, n_ai = int(counts.get(0, 0)), int(counts.get(1, 0))
    stats = {
        "config_hash": key, "subset": subset, "files": len(files), "rows": len(df),
        "label_counts": {"0": n_h, "1": n_ai},
        "human_to_ai_ratio": f"1:{n_ai / n_h:.2f}" if n_h else "n/a",
        "generators": sorted(df.loc[df["label"] == 1, "generator"].unique().tolist()),
        "conf_years": sorted(df["conf_year"].unique().tolist()),
        **dedupe_stats,
        "empty_text_dropped": int(empty.sum()),
        "schemas": dict(Counter("|".join(s.get("text_fields", [])) for s in file_stats)),
        "files_with_prefixed_fields": [s["file"] for s in file_stats if s.get("prefixed_excluded")],
        "file_stats": file_stats,
    }
    print(f"[intellabs] {subset}: {len(df)} rows after dedupe, human:AI = {stats['human_to_ai_ratio']}, "
          f"{dedupe_stats['human_duplicate_copies_dropped']} human copies dropped "
          f"({dedupe_stats['human_copies_with_different_text']} with different text)")

    cdir.mkdir(parents=True, exist_ok=True)
    df[UNIFIED_COLUMNS + ["generator_model", "text_hash"]].to_parquet(data_p, index=False)
    log.frame().to_parquet(excl_p, index=False)
    save_json(stats, stats_p)
    return df, stats, log


def per_generator_view(df, generator):
    """Human rows of the conference-years `generator` covers + that generator's AI rows."""
    ai = df[(df["label"] == 1) & (df["generator"] == generator)]
    human = df[(df["label"] == 0) & df["conf_year"].isin(ai["conf_year"].unique())]
    return pd.concat([human, ai])

In [ ]:
%%writefile src/data/leakage_audit.py
"""
Leakage audit. Runs on the FULL partitions (before caps) and blocks training on FAIL.

Checks (status PASS / WARN / FAIL / INFO):
  label_values           labels outside the dataset's label set            -> FAIL
  duplicate_row_ids      within or across partitions                       -> FAIL
  paper_overlap          composite paper_key across train/val/test         -> FAIL
                         naive key (paper_id_raw)                           -> INFO
                         submission_id (Sem-Detect cross-check)             -> FAIL
                         extended vs train/val (must be 0 after exclusion)  -> FAIL
  review_pair_integrity  IntelLabs: one human per review, never split       -> FAIL/WARN
  exact_duplicate_text   within partition -> WARN; eval rows whose text is in
                         train: fraction > configured threshold -> FAIL, >0 -> WARN
  near_duplicate_text    TF-IDF cosine on a sample                          -> INFO
  distributions          class / conference-year / generator                -> INFO
  length_only_auroc      text length as the only feature (overall and per
                         IntelLabs generator); >= threshold                 -> WARN
"""

from datetime import datetime, timezone

import pandas as pd

from .common import save_json, text_hash

LABEL_SETS = {"intellabs": {0, 1}, "semdetect": {0, 1, 2}}
TRAIN_SIDE = ("train",)
EVAL_SIDE = ("val", "test", "extended")


def _check(name, status, **details):
    return {"check": name, "status": status, **details}


def _hashes(df):
    return df["text_hash"] if "text_hash" in df.columns else df["text"].map(text_hash)


def check_labels(parts, dataset):
    allowed = LABEL_SETS[dataset]
    bad = {n: sorted(set(df["label"].dropna().astype(int)) - allowed) for n, df in parts.items()}
    bad = {n: v for n, v in bad.items() if v}
    return _check("label_values", "FAIL" if bad else "PASS", allowed=sorted(allowed), unexpected=bad)


def check_row_ids(parts):
    within = {n: int(df["row_id"].duplicated().sum()) for n, df in parts.items()}
    names = list(parts)
    across = {}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            n = len(set(parts[a]["row_id"]) & set(parts[b]["row_id"]))
            if n:
                across[f"{a}|{b}"] = n
    fail = any(within.values()) or bool(across)
    return _check("duplicate_row_ids", "FAIL" if fail else "PASS", within=within, across=across)


def check_paper_overlap(parts, dataset):
    core = [n for n in ("train", "val", "test") if n in parts]
    out = {"composite": {}, "naive_paper_id (info)": {}, "submission_id": {}, "extended": {}}
    fail = False
    for i, a in enumerate(core):
        for b in core[i + 1:]:
            pa, pb = parts[a], parts[b]
            n = len(set(pa["paper_key"]) & set(pb["paper_key"]))
            out["composite"][f"{a}|{b}"] = n
            out["naive_paper_id (info)"][f"{a}|{b}"] = len(set(pa["paper_id_raw"]) & set(pb["paper_id_raw"]))
            fail |= n > 0
            if "submission_id" in pa.columns and "submission_id" in pb.columns:
                m = len((set(pa["submission_id"]) - {""}) & (set(pb["submission_id"]) - {""}))
                out["submission_id"][f"{a}|{b}"] = m
                fail |= m > 0
    if "extended" in parts:
        for a in core:
            n = len(set(parts[a]["paper_key"]) & set(parts["extended"]["paper_key"]))
            out["extended"][f"{a}|extended"] = n
            if a in ("train", "val"):
                fail |= n > 0
    return _check("paper_overlap", "FAIL" if fail else "PASS", overlaps=out)


def check_review_pairs(parts, dataset):
    if dataset != "intellabs":
        return _check("review_pair_integrity", "INFO",
                      note="not applicable: Sem-Detect has no human<->AI pairing key (og_review_id is metadata)")
    all_rows = pd.concat([df.assign(_part=n) for n, df in parts.items()], ignore_index=True)
    key = all_rows["conf_year"] + "|" + all_rows["review_id"]
    # Partition names are unique per source subset; a review may only live in one of
    # the partitions derived from the same subset (train/val come from calibration).
    derived = all_rows["partition"] + "|" + key
    split_reviews = int((all_rows.groupby(derived)["_part"].nunique() > 1).sum())
    humans = all_rows["label"].eq(0).groupby(derived).sum()
    ais = all_rows["label"].eq(1).groupby(derived).sum()
    multi_human = int((humans > 1).sum())
    orphan_ai = int(((humans == 0) & (ais > 0)).sum())
    status = "FAIL" if (split_reviews or multi_human) else ("WARN" if orphan_ai else "PASS")
    return _check("review_pair_integrity", status,
                  reviews_split_across_partitions=split_reviews,
                  reviews_with_multiple_human_rows=multi_human,
                  ai_reviews_without_human_row=orphan_ai,
                  max_ai_rows_per_review=int(ais.max()) if len(ais) else 0)


def check_exact_duplicates(parts, fail_fraction):
    hashes = {n: _hashes(df) for n, df in parts.items()}
    within = {n: int(h.duplicated(keep=False).sum()) for n, h in hashes.items()}
    cross, status = {}, ("WARN" if any(within.values()) else "PASS")
    train_h = set().union(*(set(hashes[n]) for n in TRAIN_SIDE if n in hashes))
    pairs = [(t, e) for t in TRAIN_SIDE for e in EVAL_SIDE if t in hashes and e in hashes]
    if "val" in hashes and "test" in hashes:
        pairs.append(("val", "test"))
    for a, b in pairs:
        ref = train_h if a in TRAIN_SIDE else set(hashes[a])
        n = int(hashes[b].isin(ref).sum())
        frac = n / max(len(hashes[b]), 1)
        cross[f"{a}|{b}"] = {"rows": n, "fraction_of_" + b: round(frac, 6)}
        if frac > fail_fraction:
            status = "FAIL"
        elif n and status != "FAIL":
            status = "WARN"
    return _check("exact_duplicate_text", status, within_partition_rows=within, cross_partition=cross,
                  fail_fraction=fail_fraction)


def check_near_duplicates(parts, sample_size, threshold, seed):
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
    except ImportError:
        return _check("near_duplicate_text", "INFO", skipped="scikit-learn not installed")
    if "train" not in parts:
        return _check("near_duplicate_text", "INFO", skipped="no train partition")
    sample = {n: df.sample(min(sample_size, len(df)), random_state=seed)["text"].tolist()
              for n, df in parts.items() if len(df)}
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    vec.fit([t for texts in sample.values() for t in texts])
    xt = vec.transform(sample["train"])
    out = {}
    for n in EVAL_SIDE:
        if n not in sample:
            continue
        sims = (vec.transform(sample[n]) @ xt.T).max(axis=1).toarray().ravel()
        out[f"train|{n}"] = {"sampled": len(sims), f"cosine>={threshold}": int((sims >= threshold).sum()),
                             "max_cosine": round(float(sims.max()), 4) if len(sims) else None}
    return _check("near_duplicate_text", "INFO", sample_size=sample_size, results=out)


def check_distributions(parts):
    out = {}
    for n, df in parts.items():
        out[n] = {
            "rows": len(df), "papers": int(df["paper_key"].nunique()),
            "label": df["label_name"].value_counts().to_dict(),
            "conf_year": df["conf_year"].value_counts().sort_index().to_dict(),
            "generator": df["generator"].value_counts().to_dict(),
        }
    return _check("distributions", "INFO", partitions=out)


def _length_auc(df, pos, neg):
    from sklearn.metrics import roc_auc_score
    sub = df[df["label"].isin([pos, neg])]
    if sub["label"].nunique() < 2:
        return None
    auc = roc_auc_score((sub["label"] == pos).astype(int), sub["text"].str.len())
    return {"auroc": round(float(auc), 4), "max(auroc,1-auroc)": round(float(max(auc, 1 - auc)), 4),
            "n_pos": int((sub["label"] == pos).sum()), "n_neg": int((sub["label"] == neg).sum())}


def check_length_auroc(parts, dataset, warn_at):
    tasks = {"ai_vs_human": (1, 0)}
    if dataset == "semdetect":
        tasks["rewrite_vs_human"] = (2, 0)
    out, worst = {}, 0.0
    for n, df in parts.items():
        res = {t: _length_auc(df, p, q) for t, (p, q) in tasks.items()}
        if dataset == "intellabs":
            per_gen = {}
            for g in sorted(df.loc[df["label"] == 1, "generator"].unique()):
                ai = df[(df["label"] == 1) & (df["generator"] == g)]
                human = df[(df["label"] == 0) & df["conf_year"].isin(ai["conf_year"].unique())]
                per_gen[g] = _length_auc(pd.concat([human, ai]), 1, 0)
            res["per_generator"] = per_gen
        out[n] = res
    for res in out.values():
        for v in list(res.values()) + list(res.get("per_generator", {}).values()):
            if isinstance(v, dict) and "max(auroc,1-auroc)" in v:
                worst = max(worst, v["max(auroc,1-auroc)"])
    return _check("length_only_auroc", "WARN" if worst >= warn_at else "PASS",
                  warn_at=warn_at, worst=worst, results=out,
                  note="shortcut risk: a length-only detector; WARN does not block training")


def run_audit(parts, dataset, audit_cfg, seed=42, extra_info=None):
    """parts: partition name -> unified frame (full, uncapped). Returns the report dict."""
    checks = [
        check_labels(parts, dataset),
        check_row_ids(parts),
        check_paper_overlap(parts, dataset),
        check_review_pairs(parts, dataset),
        check_exact_duplicates(parts, float(audit_cfg.get("exact_dup_cross_fail_fraction", 0.001))),
        check_near_duplicates(parts, int(audit_cfg.get("near_dup_sample_size", 2000)),
                              float(audit_cfg.get("near_dup_cosine", 0.9)), seed),
        check_distributions(parts),
        check_length_auroc(parts, dataset, float(audit_cfg.get("length_auroc_warn", 0.75))),
    ]
    statuses = [c["status"] for c in checks]
    overall = "FAIL" if "FAIL" in statuses else ("WARN" if "WARN" in statuses else "PASS")
    return {
        "dataset": dataset, "status": overall,
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "summary": {c["check"]: c["status"] for c in checks},
        "checks": checks, "info": extra_info or {},
    }


def save_audit(report, out_dir):
    save_json(report, out_dir / "leakage_audit.json")
    save_json({k: report[k] for k in ("dataset", "status", "created_utc", "summary")},
              out_dir / "audit_status.json")


def print_summary(report):
    print(f"\n=== Leakage audit [{report['dataset']}]: {report['status']} ===")
    for c in report["checks"]:
        print(f"  {c['status']:<5} {c['check']}")


def assert_audit_passed(out_dir):
    """Called by training scripts: refuses to train without a non-FAIL audit."""
    from .common import load_json
    p = out_dir / "audit_status.json"
    if not p.exists():
        raise RuntimeError(f"No leakage audit at {p}. Run scripts/02_prepare_splits.py first.")
    status = load_json(p)["status"]
    if status == "FAIL":
        raise RuntimeError(f"Leakage audit FAILED ({p}); training is blocked.")
    return status

In [ ]:
%%writefile src/data/sem_detect_loader.py
"""
Sem-Detect/ML_Conferences-Peer-Reviews loader (3-class: human=0, ai=1, rewrite=2).

- Config `reviews`, original Parquet files read with huggingface_hub + pandas.
- The official `split` column (train/test) is respected. Validation is cut later
  from train papers only (src/data/splits.py).
- paper_id is NOT globally unique: paper_key = conference_year_paper_id;
  submission_id is kept to cross-check overlap in the audit.
- og_review_id is metadata only (not a pairing key), never used for grouping.
- Only invalid rows are dropped (missing class/split/paper key, empty text), each
  logged. The test split is NOT deduplicated; duplicate texts are only flagged.
- Nothing about the dataset is hard-coded: row, class and split counts are read from
  the real files and saved in the load stats.
"""

import pandas as pd

from .common import (
    ExclusionLog, UNIFIED_COLUMNS, cache_dir, clean_text, config_hash,
    load_json, save_json, text_hash,
)

DATASET = "semdetect"
SPLITS = ("train", "test")


def hf_frame_loader(repo_id, config_name):
    from huggingface_hub import hf_hub_download, list_repo_files
    files = sorted(
        f for f in list_repo_files(repo_id, repo_type="dataset")
        if f.startswith(f"{config_name}/") and f.endswith(".parquet")
    )
    if not files:
        raise FileNotFoundError(f"No Parquet files under {config_name}/ in {repo_id}")
    frames = [pd.read_parquet(hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset"))
              for f in files]
    return pd.concat(frames, ignore_index=True), files


def _norm(series):
    return series.map(clean_text).str.lower()


def observed_counts(raw):
    """Counts read from the real files (reported, never compared with assumed numbers)."""
    return {"rows": len(raw), "columns": list(raw.columns),
            "class_counts": _norm(raw["class"]).value_counts().to_dict(),
            "split_counts": _norm(raw["split"]).value_counts().to_dict()}


def build_frame(raw, scfg):
    """Raw Sem-Detect frame -> (unified frame, stats, ExclusionLog)."""
    missing = [c for c in scfg["required_columns"] if c not in raw.columns]
    if missing:
        raise ValueError(f"Sem-Detect: missing required columns {missing}; got {list(raw.columns)}")
    label_map = {str(k).lower(): int(v) for k, v in scfg["label_map"].items()}
    raw = raw.reset_index(drop=True).copy()
    optional_missing = [c for c in scfg.get("optional_columns", []) if c not in raw.columns]
    for c in optional_missing:
        raw[c] = ""          # absent in the real files: kept empty and reported, never invented

    cls = _norm(raw["class"])
    split = _norm(raw["split"])
    unknown_cls = sorted(set(cls[cls != ""]) - set(label_map))
    unknown_split = sorted(set(split[split != ""]) - set(SPLITS))
    if unknown_cls or unknown_split:
        raise ValueError(f"Sem-Detect: unknown class values {unknown_cls} / split values {unknown_split}")

    review_id = raw["review_id"].map(clean_text)
    ids_unique = bool(review_id.ne("").all() and not review_id.duplicated().any())
    row_id = ("sd:" + review_id) if ids_unique else ("sd:row" + raw.index.astype(str))

    conference = raw["conference"].map(clean_text)
    year = raw["year"].map(clean_text).str.replace(r"\.0$", "", regex=True)
    paper_id = raw["paper_id"].map(clean_text)
    text = raw[scfg.get("text_field", "clean_review")].map(clean_text)
    label = cls.map(label_map)

    df = pd.DataFrame({
        "row_id": row_id,
        "dataset": DATASET,
        "partition": split,
        "text": text,
        "label": label,
        "label_name": cls,
        "paper_key": conference + "_" + year + "_" + paper_id,
        "paper_id_raw": paper_id,
        "conference": conference,
        "year": pd.to_numeric(year, errors="coerce"),
        "conf_year": conference + year,
        "generator": "unspecified",
        "review_id": review_id,
        "source_file": "reviews",
        "submission_id": raw["submission_id"].map(clean_text),
        "og_review_id": raw["og_review_id"].map(clean_text),   # metadata only
        "text_hash": text.map(text_hash),
    })
    df.loc[df["label"] == 0, "generator"] = "human"

    log = ExclusionLog()
    invalid = {
        "missing_class": label.isna(),
        "missing_split": split.eq(""),
        "missing_paper_key": conference.eq("") | year.eq("") | paper_id.eq(""),
        "empty_text": text.eq(""),
    }
    drop = pd.Series(False, index=df.index)
    for reason, mask in invalid.items():
        new = mask & ~drop                      # each dropped row logged once (first reason)
        for part, ids in df.loc[new].groupby("partition")["row_id"]:
            log.add(DATASET, part or "unknown", ids, reason)
        drop |= mask
    df = df.loc[~drop].reset_index(drop=True)
    df["label"] = df["label"].astype(int)

    dup_within = {s: int(g["text_hash"].duplicated(keep=False).sum()) for s, g in df.groupby("partition")}
    tr, te = df[df["partition"] == "train"], df[df["partition"] == "test"]
    stats = {
        "rows_raw": len(raw), "rows": len(df), "optional_columns_missing": optional_missing,
        "row_id_scheme": "review_id" if ids_unique else "row_index (review_id not unique)",
        "dropped": {k: int(v.sum()) for k, v in invalid.items()},
        "class_by_split": {s: g["label_name"].value_counts().to_dict() for s, g in df.groupby("partition")},
        "duplicate_text_rows_within_split (flagged, kept)": dup_within,
        "test_rows_with_text_in_train (flagged, kept)": int(te["text_hash"].isin(tr["text_hash"]).sum()),
        "naive_paper_id_overlap_train_test": len(set(tr["paper_id_raw"]) & set(te["paper_id_raw"])),
        "paper_key_overlap_train_test": len(set(tr["paper_key"]) & set(te["paper_key"])),
        "submission_id_overlap_train_test": len(
            set(tr["submission_id"]) - {""} & set(te["submission_id"]) - {""}),
        "og_review_id_distinct": int(df["og_review_id"].nunique()),
    }
    return df, stats, log


def load_semdetect(cfg, frame_loader=None, use_cache=True):
    """Load the whole config -> (unified frame with partition train/test, stats, ExclusionLog)."""
    scfg = cfg["data"]["semdetect"]
    key = config_hash({k: scfg.get(k) for k in ("repo_id", "config_name", "label_map", "text_field")})
    cdir = cache_dir(cfg, DATASET)
    data_p, excl_p, stats_p = cdir / "all.parquet", cdir / "all.exclusions.parquet", cdir / "all.stats.json"
    if use_cache and data_p.exists() and stats_p.exists():
        stats = load_json(stats_p)
        if stats.get("config_hash") == key:
            print(f"[semdetect] using cache {data_p}")
            log = ExclusionLog()
            if excl_p.exists():
                log._parts.append(pd.read_parquet(excl_p))
            return pd.read_parquet(data_p), stats, log

    loader = frame_loader or hf_frame_loader
    raw, files = loader(scfg["repo_id"], scfg.get("config_name", "reviews"))
    observed = observed_counts(raw)
    df, stats, log = build_frame(raw, scfg)
    stats = {"config_hash": key, "files": files, "observed_raw": observed, **stats}
    print(f"[semdetect] observed in the real files: rows={observed['rows']} "
          f"classes={observed['class_counts']} splits={observed['split_counts']}")
    if stats["optional_columns_missing"]:
        print(f"[semdetect] optional columns not present: {stats['optional_columns_missing']}")
    print(f"[semdetect] {len(df)} rows kept of {len(raw)}; class_by_split={stats['class_by_split']}")

    cdir.mkdir(parents=True, exist_ok=True)
    keep = UNIFIED_COLUMNS + ["submission_id", "og_review_id", "text_hash"]
    df[keep].to_parquet(data_p, index=False)
    log.frame().to_parquet(excl_p, index=False)
    save_json(stats, stats_p)
    return df, stats, log

In [ ]:
%%writefile src/data/splits.py
"""
Paper-level splitting and caps. Individual reviews are never split: every
function selects or assigns WHOLE papers (group_col, default paper_key).

- split_by_paper: train/val split of a train pool, stratified by conference-year.
- cap_by_papers: random whole-paper subset whose (counted) rows fit a budget,
  allocated proportionally per conference-year. Used for train/val/test caps and
  for the baseline subset, which is drawn from the (capped) model test set so it
  is nested inside it.
- check_invariants: re-checkable conditions on saved manifests (--verify-only).
"""

import numpy as np
import pandas as pd


def _papers_by_stratum(df, group_col, strata_col):
    strata = df[strata_col] if strata_col and strata_col in df.columns else pd.Series("all", index=df.index)
    pairs = pd.DataFrame({"g": df[group_col], "s": strata}).drop_duplicates()
    if pairs["g"].duplicated().any():
        raise ValueError(f"papers span several {strata_col} values: "
                         f"{pairs.loc[pairs['g'].duplicated(), 'g'].head(5).tolist()}")
    first = pairs
    return {s: sorted(g["g"].tolist()) for s, g in first.groupby("s", sort=True)}


def split_by_paper(df, val_fraction, seed, group_col="paper_key", strata_col="conf_year"):
    """Return (train, val). About val_fraction of the papers of each stratum go to val."""
    if not 0 < val_fraction < 1:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")
    rng = np.random.default_rng(seed)
    val_papers = set()
    for _, papers in _papers_by_stratum(df, group_col, strata_col).items():
        papers = list(papers)
        rng.shuffle(papers)
        n_val = int(round(val_fraction * len(papers)))
        if len(papers) > 1:
            n_val = min(max(n_val, 1), len(papers) - 1)
        val_papers.update(papers[:n_val])
    in_val = df[group_col].isin(val_papers)
    return df.loc[~in_val].copy(), df.loc[in_val].copy()


def cap_by_papers(df, max_rows, seed, group_col="paper_key", strata_col="conf_year", count_mask=None):
    """
    Whole-paper subset with at most max_rows counted rows (all rows if max_rows is None
    or the frame already fits). count_mask (bool Series) selects which rows count
    toward the budget; uncounted rows of chosen papers are kept too.
    """
    counted = count_mask if count_mask is not None else pd.Series(True, index=df.index)
    total = int(counted.sum())
    if max_rows is None or total <= max_rows:
        return df.copy()
    rng = np.random.default_rng(seed)
    sizes = counted.groupby(df[group_col]).sum()
    strata = _papers_by_stratum(df, group_col, strata_col)
    stratum_rows = {s: int(sizes.loc[p].sum()) for s, p in strata.items()}

    chosen, used, leftovers = set(), 0, []
    for s, papers in strata.items():
        budget = int(np.floor(max_rows * stratum_rows[s] / total))
        papers = list(papers)
        rng.shuffle(papers)
        got = 0
        for p in papers:
            n = int(sizes[p])
            if got + n <= budget:
                chosen.add(p)
                got += n
            else:
                leftovers.append(p)
        used += got
    rng.shuffle(leftovers)                       # fill the rounding remainder
    for p in leftovers:
        n = int(sizes[p])
        if used + n <= max_rows:
            chosen.add(p)
            used += n
    return df.loc[df[group_col].isin(chosen)].copy()


def paper_overlaps(parts, group_col="paper_key"):
    """{"a|b": n_shared_papers} for every pair of partitions."""
    names = list(parts)
    out = {}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            out[f"{a}|{b}"] = len(set(parts[a][group_col]) & set(parts[b][group_col]))
    return out


def check_invariants(parts, caps, disjoint=("train", "val", "test"), baseline_count_mask=None):
    """
    Conditions every saved manifest set must satisfy. Returns a list of failures (empty = PASS).
    parts: name -> frame; caps: name -> max_rows or None.
    """
    fails = []
    for name, df in parts.items():
        if df["row_id"].duplicated().any():
            fails.append(f"{name}: duplicate row_id")
        if df["label"].isna().any():
            fails.append(f"{name}: missing labels")
        if (df["text"].fillna("") == "").any():
            fails.append(f"{name}: empty text")
    present = [d for d in disjoint if d in parts]
    for pair, n in paper_overlaps({d: parts[d] for d in present}).items():
        if n:
            fails.append(f"paper overlap {pair}: {n}")
    for name in ("train", "val"):
        if name in parts and "extended" in parts:
            n = len(set(parts[name]["paper_key"]) & set(parts["extended"]["paper_key"]))
            if n:
                fails.append(f"paper overlap {name}|extended: {n} (should have been excluded)")
    if "test_baseline" in parts and "test" in parts:
        if not set(parts["test_baseline"]["row_id"]) <= set(parts["test"]["row_id"]):
            fails.append("test_baseline is not nested inside test")
    for name, cap in caps.items():
        if cap is None or name not in parts:
            continue
        df = parts[name]
        n = int(baseline_count_mask(df).sum()) if (name == "test_baseline" and baseline_count_mask) else len(df)
        if n > cap:
            fails.append(f"{name}: {n} counted rows exceed cap {cap}")
    return fails

In [ ]:
%%writefile src/evaluation/aggregate.py
"""
Breakdowns and tables.

- by_group: binary metrics per value of a column (conf_year, ...).
- by_generator (IntelLabs): for generator g, AI rows of g plus the human rows of the
  conference-years g covers (human rows are shared across generators).
- seed_summary: mean +/- std over seeds (sample std, ddof=1).
- comparison table: one row per method with its type
  (trained classifier / pretrained detector / zero-shot detector) -> CSV + JSON.
"""

from pathlib import Path

import numpy as np
import pandas as pd

from src.data.common import load_json, save_json

from .metrics import binary_metrics, headline

METHOD_TYPES = ("trained classifier", "pretrained detector", "zero-shot detector")


def by_group(pred, col, threshold=0.5, min_neg_events=10):
    out = {}
    for g, sub in pred.groupby(col):
        out[str(g)] = headline(binary_metrics(sub["label"], sub["score"], threshold, min_neg_events))
    return out


def by_length_bin(pred, n_words, threshold=0.5, n_bins=3, min_neg_events=10):
    """
    Metrics per review-length bin (equal-sized quantile bins of word count on this split), to
    check that performance does not hinge on length (cf. CoCoDet, Table 11).
    n_words: Series aligned with pred.
    """
    words = np.asarray(n_words, dtype=float)
    edges = np.unique(np.quantile(words, np.linspace(0, 1, n_bins + 1)))
    if len(edges) < 2:
        return {}
    bins = np.clip(np.searchsorted(edges, words, side="right") - 1, 0, len(edges) - 2)
    out = {}
    for b in range(len(edges) - 1):
        sub = pred[bins == b]
        if len(sub):
            key = f"{int(edges[b])}-{int(edges[b + 1])} words"
            out[key] = headline(binary_metrics(sub["label"], sub["score"], threshold, min_neg_events))
    return out


def by_generator(pred, threshold=0.5, min_neg_events=10):
    out = {}
    ai = pred[pred["label"] == 1]
    for g in sorted(ai["generator"].unique()):
        a = ai[ai["generator"] == g]
        h = pred[(pred["label"] == 0) & pred["conf_year"].isin(a["conf_year"].unique())]
        out[g] = headline(binary_metrics(pd.concat([h, a])["label"], pd.concat([h, a])["score"],
                                         threshold, min_neg_events))
    return out


def seed_summary(metric_dicts, keys=("accuracy", "macro_f1", "f1", "roc_auc", "pr_auc")):
    out = {"n_seeds": len(metric_dicts)}
    for k in keys:
        vals = [m.get(k) for m in metric_dicts if m.get(k) is not None]
        if vals:
            out[k] = {"mean": float(np.mean(vals)),
                      "std": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
                      "values": vals}
    return out


def comparison_row(method, method_type, split, threshold_kind, metrics, **extra):
    if method_type not in METHOD_TYPES:
        raise ValueError(f"method_type must be one of {METHOD_TYPES}")
    return {"method": method, "type": method_type, "split": split,
            "threshold_kind": threshold_kind, **headline(metrics), **extra}


def save_comparison_table(rows, out_dir, name="comparison_table"):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(rows)
    if not df.empty:
        order = {t: i for i, t in enumerate(METHOD_TYPES)}
        df = df.sort_values(["split", "threshold_kind", "type", "method"],
                            key=lambda s: s.map(order) if s.name == "type" else s).reset_index(drop=True)
    df.to_csv(out_dir / f"{name}.csv", index=False)
    save_json(df.to_dict(orient="records"), out_dir / f"{name}.json")
    return df


def merge_comparison_rows(path_json, new_rows, key=("method", "split", "threshold_kind")):
    """Upsert rows into an existing comparison_table.json (restart-safe incremental runs)."""
    path_json = Path(path_json)
    old = load_json(path_json) if path_json.exists() else []
    new_keys = {tuple(r[k] for k in key) for r in new_rows}
    return [r for r in old if tuple(r.get(k) for k in key) not in new_keys] + list(new_rows)

In [ ]:
%%writefile src/evaluation/error_analysis.py
"""
Error analysis on a prediction frame (binary: label/pred/score; multiclass: label/pred/proba_*).

- fp_fn_tables: false positives / false negatives with ids, label, prediction and
  confidence, most confident errors first.
- error_breakdown: error rate per class and per conference-year (and generator).
- Text is attached (truncated) when a text column/lookup is given, so the tables
  can be read without opening the manifests.
"""

from pathlib import Path

import numpy as np
import pandas as pd


def _confidence(pred):
    pcols = [c for c in pred.columns if c.startswith("proba_")]
    if pcols and "score" not in pred.columns:
        return pred[pcols].max(axis=1)
    return np.where(pred["pred"] == 1, pred["score"], 1 - pred["score"]) \
        if pred["score"].between(0, 1).all() else pred["score"].abs()


def fp_fn_tables(pred, texts=None, top_k=50, text_chars=400):
    p = pred.copy()
    p["confidence"] = _confidence(p)
    if texts is not None:
        p["text"] = p["row_id"].map(texts).fillna("").str.slice(0, text_chars)
    cols = [c for c in ("row_id", "paper_key", "conf_year", "generator", "label", "label_name",
                        "pred", "score", "confidence", "text") if c in p.columns]
    wrong = p[p["pred"] != p["label"]].sort_values("confidence", ascending=False)
    if set(p["label"].unique()) <= {0, 1}:
        fp = wrong[(wrong["label"] == 0) & (wrong["pred"] == 1)][cols].head(top_k)
        fn = wrong[(wrong["label"] == 1) & (wrong["pred"] == 0)][cols].head(top_k)
        return {"false_positives": fp, "false_negatives": fn}
    return {"misclassified": wrong[cols].head(top_k)}


def error_breakdown(pred, cols=("label_name", "conf_year", "generator")):
    out = {}
    err = (pred["pred"] != pred["label"]).astype(int)
    for c in cols:
        if c in pred.columns:
            g = err.groupby(pred[c])
            out[c] = {str(k): {"n": int(n), "errors": int(e), "error_rate": round(e / n, 6) if n else None}
                      for k, n, e in zip(g.size().index, g.size().values, g.sum().values)}
    return out


def save_error_analysis(pred, out_dir, prefix, texts=None, top_k=50):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for name, table in fp_fn_tables(pred, texts, top_k).items():
        p = out_dir / f"{prefix}_{name}.csv"
        table.to_csv(p, index=False)
        paths[name] = str(p)
    return paths, error_breakdown(pred)


def text_lookup(manifest):
    return pd.Series(manifest["text"].values, index=manifest["row_id"].values)

In [ ]:
%%writefile src/evaluation/metrics.py
"""
Classification metrics. Positive class = AI (label 1).

binary_metrics:     Accuracy, Precision, Recall, F1 (AI), Macro-F1, Weighted-F1,
                    ROC-AUC, PR-AUC, confusion matrix, per-class metrics, AI TPR/FPR,
                    TPR at 1/5/10% FPR (reported only when n_neg supports it).
multiclass_metrics: Accuracy, Macro/Weighted-F1, per-class metrics, confusion
                    matrix, one-vs-rest macro ROC-AUC (Sem-Detect native 3-class).
calibrate_threshold: chosen on VALIDATION scores only.

Edge cases: single-class inputs give None for ranking metrics (with a note);
divisions by zero give 0.0 (sklearn zero_division=0).
"""

import numpy as np
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_recall_fscore_support, roc_auc_score, roc_curve,
)

FPR_TARGETS = (0.01, 0.05, 0.10)
BINARY_NAMES = ("human", "ai")


def _r(x):
    return None if x is None else round(float(x), 6)


def tpr_at_fpr(y_true, score, targets=FPR_TARGETS, min_neg_events=10):
    """
    Highest TPR with FPR <= target. `supported` is False when n_neg * target < min_neg_events
    (fewer than min_neg_events negatives would define the tail); the value is then None.
    """
    y_true = np.asarray(y_true).astype(int)
    n_neg = int((y_true == 0).sum())
    out = {}
    have_both = len(np.unique(y_true)) == 2
    fpr, tpr, thr = roc_curve(y_true, score) if have_both else (None, None, None)
    for t in targets:
        key = f"{int(round(t * 100))}%"
        supported = have_both and n_neg * t >= min_neg_events
        if not supported:
            out[key] = {"tpr": None, "threshold": None, "supported": False, "n_neg": n_neg}
            continue
        ok = fpr <= t
        i = int(np.argmax(np.where(ok, tpr, -1)))
        out[key] = {"tpr": _r(tpr[i]), "threshold": _r(thr[i]), "supported": True, "n_neg": n_neg}
    return out


def binary_metrics(y_true, score, threshold=0.5, min_neg_events=10):
    """y_true in {0,1}; score = P(AI) or any 'larger = more AI' score; pred = score >= threshold."""
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score, dtype=float)
    if len(y) != len(s):
        raise ValueError("y_true and score lengths differ")
    if not np.isfinite(s).all():
        raise ValueError(f"{int((~np.isfinite(s)).sum())} non-finite scores")
    pred = (s >= threshold).astype(int)
    n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())
    p, r, f, sup = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    both = n_pos > 0 and n_neg > 0
    return {
        "n": len(y), "n_pos": n_pos, "n_neg": n_neg, "threshold": _r(threshold),
        "accuracy": _r(accuracy_score(y, pred)) if len(y) else None,
        "precision": _r(p[1]), "recall": _r(r[1]), "f1": _r(f[1]),
        "macro_f1": _r(f.mean()),
        "weighted_f1": _r(f1_score(y, pred, labels=[0, 1], average="weighted", zero_division=0)),
        "roc_auc": _r(roc_auc_score(y, s)) if both else None,
        "pr_auc": _r(average_precision_score(y, s)) if both else None,
        "confusion_matrix": {"labels": list(BINARY_NAMES), "matrix": [[int(tn), int(fp)], [int(fn), int(tp)]]},
        "per_class": {BINARY_NAMES[k]: {"precision": _r(p[k]), "recall": _r(r[k]), "f1": _r(f[k]),
                                        "support": int(sup[k])} for k in (0, 1)},
        "ai_tpr": _r(tp / n_pos) if n_pos else None,
        "ai_fpr": _r(fp / n_neg) if n_neg else None,
        "tpr_at_fpr": tpr_at_fpr(y, s, min_neg_events=min_neg_events),
        "notes": [] if both else ["single class present: ROC-AUC/PR-AUC/TPR@FPR undefined"],
    }


def multiclass_metrics(y_true, proba, class_names):
    y = np.asarray(y_true).astype(int)
    P = np.asarray(proba, dtype=float)
    k = len(class_names)
    if P.shape != (len(y), k):
        raise ValueError(f"proba shape {P.shape} != ({len(y)}, {k})")
    pred = P.argmax(1)
    labels = list(range(k))
    p, r, f, sup = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    present = np.unique(y)
    auc, notes = None, []
    if len(present) == k:
        auc = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=labels)
    else:
        notes.append(f"classes present {present.tolist()}: macro OvR ROC-AUC undefined")
    return {
        "n": len(y), "class_counts": {class_names[i]: int(sup[i]) for i in labels},
        "accuracy": _r(accuracy_score(y, pred)) if len(y) else None,
        "macro_f1": _r(f.mean()),
        "weighted_f1": _r(f1_score(y, pred, labels=labels, average="weighted", zero_division=0)),
        "macro_ovr_roc_auc": _r(auc),
        "confusion_matrix": {"labels": list(class_names),
                             "matrix": confusion_matrix(y, pred, labels=labels).tolist()},
        "per_class": {class_names[i]: {"precision": _r(p[i]), "recall": _r(r[i]), "f1": _r(f[i]),
                                       "support": int(sup[i])} for i in labels},
        "notes": notes,
    }


def calibrate_threshold(y_val, score_val, objective="macro_f1"):
    """
    Threshold chosen on validation only. objective: 'macro_f1' (default) or 'youden'.
    Candidates are midpoints between sorted unique scores (plus the extremes).
    """
    y = np.asarray(y_val).astype(int)
    s = np.asarray(score_val, dtype=float)
    if len(np.unique(y)) < 2:
        return 0.5
    u = np.unique(s)
    cands = np.concatenate([[u[0] - 1e-12], (u[:-1] + u[1:]) / 2, [u[-1] + 1e-12]])
    if len(cands) > 2000:                                   # quantile grid keeps it fast
        cands = np.unique(np.quantile(s, np.linspace(0, 1, 2001)))
    best_t, best_v = 0.5, -np.inf
    for t in cands:
        pred = (s >= t).astype(int)
        if objective == "youden":
            tp = ((pred == 1) & (y == 1)).sum() / max((y == 1).sum(), 1)
            fp = ((pred == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)
            v = tp - fp
        else:
            v = f1_score(y, pred, average="macro", zero_division=0)
        if v > best_v:
            best_t, best_v = float(t), v
    return best_t


def headline(m):
    """Flat row of the main numbers for comparison tables."""
    row = {k: m.get(k) for k in ("n", "n_pos", "n_neg", "threshold", "accuracy", "precision", "recall",
                                 "f1", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "ai_tpr", "ai_fpr")}
    for k, v in (m.get("tpr_at_fpr") or {}).items():
        row[f"tpr@{k}fpr"] = v["tpr"]
    return row

In [ ]:
%%writefile src/evaluation/predict.py
"""
Prediction frames: one row per example with id, label, probabilities and metadata.
Saved as CSV under outputs/results/<dataset>/.

Columns: row_id, label, pred, score (= P(AI) or a 'larger = more AI' detector score),
         proba_<class> (multiclass), paper_key, conf_year, generator, label_name.

Sem-Detect binary view of the 3-class model (for comparison with binary baselines):
rows with label in {human, ai} only, score = P(ai) / (P(human) + P(ai)).
Thresholding that score at 0.5 equals argmax over the two classes.
"""

from pathlib import Path

import numpy as np
import pandas as pd

META_COLS = ["row_id", "label", "label_name", "paper_key", "conf_year", "generator"]


def softmax(logits):
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def prediction_frame(meta, proba=None, score=None, class_names=("human", "ai"), threshold=0.5):
    """
    meta: frame with META_COLS (extra columns ignored).
    proba: (n, K) probabilities (model). score: (n,) detector score (baselines).
    """
    out = meta[[c for c in META_COLS if c in meta.columns]].reset_index(drop=True).copy()
    if proba is not None:
        P = np.asarray(proba, dtype=float)
        for i, name in enumerate(class_names):
            out[f"proba_{name}"] = P[:, i]
        out["pred"] = P.argmax(1)
        if len(class_names) == 2:
            out["score"] = P[:, 1]
            out["pred"] = (out["score"] >= threshold).astype(int)
    elif score is not None:
        out["score"] = np.asarray(score, dtype=float)
        out["pred"] = (out["score"] >= threshold).astype(int)
    else:
        raise ValueError("give proba or score")
    return out


def binary_view(pred3, human="human", ai="ai", threshold=0.5):
    """3-class prediction frame -> human/ai-only binary frame (rewrite rows dropped)."""
    keep = pred3["label"].isin([0, 1])
    b = pred3.loc[keep].copy()
    ph, pa = b[f"proba_{human}"], b[f"proba_{ai}"]
    b["score"] = pa / (ph + pa).clip(lower=1e-12)
    b["pred"] = (b["score"] >= threshold).astype(int)
    return b


def save_predictions(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def load_predictions(path):
    return pd.read_csv(path, dtype={"row_id": str, "paper_key": str})


def run_inference(model, loader, device, amp_dtype=None):
    """Logits for every batch of a DataLoader yielding dicts (input_ids, attention_mask, ...)."""
    import torch
    model.eval()
    chunks = []
    with torch.inference_mode():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items() if k in ("input_ids", "attention_mask")}
            if amp_dtype is not None and device.type == "cuda":
                with torch.autocast("cuda", dtype=amp_dtype):
                    logits = model(**batch)
            else:
                logits = model(**batch)
            chunks.append(logits.float().cpu().numpy())
    return np.concatenate(chunks) if chunks else np.zeros((0, 0))

In [ ]:
%%writefile src/evaluation/stats.py
"""
Statistics for comparing detectors.

- cluster_bootstrap_ci: percentile CI where PAPERS (not reviews) are resampled with
  replacement, because reviews of one paper are correlated.
- paired_cluster_bootstrap: difference of a metric between two methods scored on the
  SAME examples; same paper resample for both methods; two-sided bootstrap p-value.
- mcnemar_exact: paired test on correct/incorrect decisions (exact binomial).
- holm: Holm-Bonferroni adjustment for a family of comparisons.

Assumptions and limits (state them in the thesis):
- papers are treated as independent clusters; dependence across papers (same
  reviewer, same venue, same generator) is not modelled;
- percentile bootstrap CIs can be too narrow with few clusters (< ~30 papers);
- McNemar ignores paper clustering (reviews treated as independent) and is
  therefore reported next to, not instead of, the paired cluster bootstrap;
- bootstrap p-values are bounded below by 1/(n_boot+1).
"""

import numpy as np
from sklearn.metrics import f1_score, roc_auc_score


def auc_fn(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan


def macro_f1_fn(threshold=0.5):
    def f(y, s):
        return f1_score(y, (np.asarray(s) >= threshold).astype(int), average="macro",
                        labels=[0, 1], zero_division=0)
    return f


def _cluster_index(clusters):
    clusters = np.asarray(clusters)
    _, inv = np.unique(clusters, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    bounds = np.flatnonzero(np.diff(inv[order])) + 1
    return np.split(order, bounds)


def _resample(groups, rng):
    pick = rng.integers(0, len(groups), len(groups))
    return np.concatenate([groups[i] for i in pick])


def cluster_bootstrap_ci(y, s, clusters, metric_fn=auc_fn, n_boot=1000, alpha=0.05, seed=0):
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    groups = _cluster_index(clusters)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = _resample(groups, rng)
        boots.append(metric_fn(y[idx], s[idx]))
    boots = np.asarray(boots, dtype=float)
    ok = boots[np.isfinite(boots)]
    est = metric_fn(y, s)
    return {
        "estimate": float(est) if np.isfinite(est) else None,
        "ci_low": float(np.quantile(ok, alpha / 2)) if len(ok) else None,
        "ci_high": float(np.quantile(ok, 1 - alpha / 2)) if len(ok) else None,
        "alpha": alpha, "n_boot": n_boot, "valid_boot": int(len(ok)), "n_clusters": len(groups),
    }


def paired_cluster_bootstrap(y, s_a, s_b, clusters, metric_fn=auc_fn, n_boot=1000, alpha=0.05, seed=0):
    """Metric(a) - Metric(b) on shared examples. Returns estimate, CI and two-sided p-value."""
    y = np.asarray(y)
    s_a, s_b = np.asarray(s_a, dtype=float), np.asarray(s_b, dtype=float)
    if not (len(y) == len(s_a) == len(s_b)):
        raise ValueError("paired test needs the same examples for both methods")
    groups = _cluster_index(clusters)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = _resample(groups, rng)
        diffs.append(metric_fn(y[idx], s_a[idx]) - metric_fn(y[idx], s_b[idx]))
    d = np.asarray(diffs, dtype=float)
    d = d[np.isfinite(d)]
    est = metric_fn(y, s_a) - metric_fn(y, s_b)
    if len(d) == 0:
        return {"diff": None, "ci_low": None, "ci_high": None, "p_value": None, "n_boot": n_boot}
    p = 2 * min((np.sum(d <= 0) + 1) / (len(d) + 1), (np.sum(d >= 0) + 1) / (len(d) + 1))
    return {
        "diff": float(est), "ci_low": float(np.quantile(d, alpha / 2)),
        "ci_high": float(np.quantile(d, 1 - alpha / 2)), "p_value": float(min(p, 1.0)),
        "n_boot": n_boot, "valid_boot": int(len(d)), "n_clusters": len(groups),
    }


def mcnemar_exact(y, pred_a, pred_b):
    from scipy.stats import binomtest
    y, a, b = map(np.asarray, (y, pred_a, pred_b))
    ca, cb = a == y, b == y
    n01 = int((ca & ~cb).sum())     # a right, b wrong
    n10 = int((~ca & cb).sum())     # a wrong, b right
    n = n01 + n10
    p = 1.0 if n == 0 else float(binomtest(n01, n, 0.5).pvalue)
    return {"a_right_b_wrong": n01, "a_wrong_b_right": n10, "p_value": p}


def holm(pvalues):
    """dict name -> p  ->  dict name -> Holm-adjusted p (monotone, capped at 1)."""
    items = [(k, v) for k, v in pvalues.items() if v is not None]
    items.sort(key=lambda kv: kv[1])
    m, running, out = len(items), 0.0, {}
    for i, (k, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        out[k] = running
    for k, v in pvalues.items():
        out.setdefault(k, None)
    return out

In [ ]:
%%writefile src/models/dataset.py
"""
Tokenized review dataset with truncation reporting.

- Text is used as stored in the manifests (strip-only preprocessing upstream).
- truncation="head": first max_length tokens (incl. special tokens).
  truncation="head_tail": first head_tokens + last remaining tokens of long reviews.
- truncation_stats(): fraction of truncated examples and token-length summary,
  saved with every training/evaluation run.
"""

import numpy as np
import torch
from torch.utils.data import Dataset


def special_template(tokenizer):
    """
    (prefix_ids, suffix_ids) the tokenizer adds around a single sequence, read from the
    tokenizer itself (e.g. [CLS] ... [SEP] for ModernBERT). Fast tokenizers do not
    implement build_inputs_with_special_tokens, so this is detected instead of assumed.
    """
    probe = tokenizer("review text", add_special_tokens=False)["input_ids"]
    full = tokenizer("review text", add_special_tokens=True)["input_ids"]
    for i in range(len(full) - len(probe) + 1):
        if full[i:i + len(probe)] == probe:
            return full[:i], full[i + len(probe):]
    raise ValueError("could not locate the special-token template of this tokenizer")


class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=1024, truncation="head", head_tokens=256):
        if truncation not in ("head", "head_tail"):
            raise ValueError(f"truncation must be head or head_tail, got {truncation}")
        self.labels = None if labels is None else np.asarray(labels).astype(int)
        prefix, suffix = special_template(tokenizer)
        n_special = len(prefix) + len(suffix)
        budget = max_length - n_special
        if budget <= 0:
            raise ValueError("max_length too small for special tokens")
        enc = tokenizer(list(texts), add_special_tokens=False, truncation=False)["input_ids"]
        self.raw_lengths = np.array([len(e) + n_special for e in enc])
        self.input_ids = []
        for ids in enc:
            if len(ids) > budget:
                if truncation == "head_tail":
                    h = min(head_tokens, budget)
                    ids = ids[:h] + (ids[len(ids) - (budget - h):] if budget > h else [])
                else:
                    ids = ids[:budget]
            self.input_ids.append(prefix + ids + suffix)
        self.max_length = max_length
        self.pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        item = {"input_ids": self.input_ids[i]}
        if self.labels is not None:
            item["labels"] = int(self.labels[i])
        return item

    def collate(self, batch):
        width = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), width), self.pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch), width), dtype=torch.long)
        for j, b in enumerate(batch):
            n = len(b["input_ids"])
            ids[j, :n] = torch.tensor(b["input_ids"], dtype=torch.long)
            mask[j, :n] = 1
        out = {"input_ids": ids, "attention_mask": mask}
        if "labels" in batch[0]:
            out["labels"] = torch.tensor([b["labels"] for b in batch], dtype=torch.long)
        return out

    def truncation_stats(self):
        L = self.raw_lengths
        if len(L) == 0:
            return {"n": 0}
        return {
            "n": int(len(L)), "max_length": self.max_length,
            "truncated_fraction": round(float((L > self.max_length).mean()), 6),
            "tokens_mean": round(float(L.mean()), 1), "tokens_median": float(np.median(L)),
            "tokens_p95": float(np.quantile(L, 0.95)), "tokens_max": int(L.max()),
        }

In [ ]:
%%writefile src/models/losses.py
"""
PeerBERT loss:  L = L_cls + alpha * L_cons

L_cls  classification loss, one of
       ce               plain cross-entropy (unweighted)
       weighted_ce      cross-entropy with w_c = N / (K * n_c), counts from the TRAIN split only
       label_smoothing  weighted cross-entropy with label smoothing (default 0.05)
       focal            weighted focal loss, gamma = 2 (Lin et al., 2017)
       csm              cost-sensitive margin loss adapted from CoCoDet (Chen et al., ACL 2026):
                        on cosine logits z_j, the true class gets z_y - m_base and, when the
                        true and the competing class are both critical (human, ai), the
                        competing logit gets + m_cost; cross-entropy on s * z. Needs the
                        cosine classification head (model.head: cosine). Unweighted, as in the paper.
L_cons R-Drop consistency (Liang et al., NeurIPS 2021): symmetric KL between the
       predictive distributions of two stochastic (dropout) passes of the same batch.
       With R-Drop on, L_cls is the mean of the two passes' losses, as in R-Drop.
"""

import numpy as np
import torch
import torch.nn.functional as F

LOSS_TYPES = ("ce", "weighted_ce", "label_smoothing", "focal", "csm")
CRITICAL_CLASSES = (0, 1)      # human, ai: confusing these two is the costly error


def class_weights(train_labels, num_classes):
    """w_c = N / (K * n_c) from training labels. Raises if a class is absent from train."""
    y = np.asarray(train_labels).astype(int)
    counts = np.bincount(y, minlength=num_classes)
    if (counts == 0).any():
        raise ValueError(f"class(es) {np.flatnonzero(counts == 0).tolist()} absent from the training split")
    return torch.tensor(len(y) / (num_classes * counts), dtype=torch.float32)


def focal_loss(logits, target, gamma=2.0, weight=None):
    logp = F.log_softmax(logits.float(), dim=-1)
    logp_t = logp.gather(1, target[:, None]).squeeze(1)
    loss = -((1 - logp_t.exp()) ** gamma) * logp_t
    if weight is None:
        return loss.mean()
    w = weight.to(logits.device)[target]
    return (w * loss).sum() / w.sum()           # same normalisation as weighted F.cross_entropy


def csm_logits(cos, target, scale=30.0, m_base=0.25, m_cost=0.25, critical=CRITICAL_CLASSES):
    """Margin-adjusted, scaled logits from cosine similarities (CoCoDet Eq. 1-2)."""
    z = cos.float().clone()
    rows = torch.arange(len(target), device=z.device)
    z[rows, target] = z[rows, target] - m_base
    for c in critical:
        for d in critical:
            if c != d and d < z.shape[1]:
                z[target == c, d] = z[target == c, d] + m_cost
    return scale * z


def classification_loss(logits, target, kind="weighted_ce", weight=None, smoothing=0.05, gamma=2.0,
                        csm=None):
    logits = logits.float()
    if kind == "csm":
        c = csm or {}
        scale = float(c.get("scale", 30.0))
        cos = logits / scale                       # the cosine head outputs scale * cos
        return F.cross_entropy(csm_logits(cos, target, scale, float(c.get("m_base", 0.25)),
                                          float(c.get("m_cost", 0.25))), target)
    w = weight.to(logits.device) if weight is not None else None
    if kind == "ce":
        return F.cross_entropy(logits, target)
    if kind == "weighted_ce":
        return F.cross_entropy(logits, target, weight=w)
    if kind == "label_smoothing":
        return F.cross_entropy(logits, target, weight=w, label_smoothing=smoothing)
    if kind == "focal":
        return focal_loss(logits, target, gamma, w)
    raise ValueError(f"unknown loss '{kind}', expected one of {LOSS_TYPES}")


def rdrop_kl(logits1, logits2):
    """Symmetric KL: 0.5 * (KL(p1||p2) + KL(p2||p1)), averaged over the batch."""
    lp1, lp2 = F.log_softmax(logits1.float(), -1), F.log_softmax(logits2.float(), -1)
    kl12 = F.kl_div(lp2, lp1, log_target=True, reduction="batchmean")   # KL(p1 || p2)
    kl21 = F.kl_div(lp1, lp2, log_target=True, reduction="batchmean")   # KL(p2 || p1)
    return 0.5 * (kl12 + kl21)


class PeerBERTLoss:
    def __init__(self, kind="weighted_ce", weight=None, rdrop_alpha=0.0, smoothing=0.05, gamma=2.0, csm=None):
        if kind not in LOSS_TYPES:
            raise ValueError(f"unknown loss '{kind}'")
        if kind in ("ce", "csm"):
            weight = None
        self.kind, self.weight, self.alpha = kind, weight, float(rdrop_alpha)
        self.smoothing, self.gamma, self.csm = smoothing, gamma, csm

    @property
    def uses_rdrop(self):
        return self.alpha > 0

    def __call__(self, logits1, target, logits2=None):
        """Returns (total, parts dict). logits2 is required when R-Drop is on."""
        cls1 = classification_loss(logits1, target, self.kind, self.weight, self.smoothing, self.gamma, self.csm)
        if not self.uses_rdrop:
            return cls1, {"cls": float(cls1.detach()), "cons": 0.0}
        if logits2 is None:
            raise ValueError("R-Drop needs two stochastic passes")
        cls2 = classification_loss(logits2, target, self.kind, self.weight, self.smoothing, self.gamma, self.csm)
        cls = 0.5 * (cls1 + cls2)
        cons = rdrop_kl(logits1, logits2)
        return cls + self.alpha * cons, {"cls": float(cls.detach()), "cons": float(cons.detach())}

In [ ]:
%%writefile src/models/modernbert_classifier.py
"""
PeerBERT = ModernBERT-base + sequence-classification head (2 classes for IntelLabs,
3 classes for Sem-Detect), loaded with AutoModelForSequenceClassification.

Version-robust choices (transformers changes quickly; features are detected, not assumed):
- reference_compile=False is set when the config has that field (avoids torch.compile
  inside ModernBERT on Kaggle/Colab GPUs).
- Dropout: ModernBERT ships with dropout 0.0. Every *_dropout field present in the
  config is reported, and set to `dropout` when dropout_on is true. R-Drop requires
  non-zero dropout; the trainer refuses R-Drop otherwise.
- attn_implementation is left to the library default (sdpa when available). XAI later
  uses an eager-attention copy.
"""

import json
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn

HEAD_FILE = "peerbert_head.json"

DROPOUT_FIELDS = ("embedding_dropout", "attention_dropout", "mlp_dropout", "classifier_dropout",
                  "hidden_dropout_prob", "attention_probs_dropout_prob")


def dropout_values(config):
    return {f: getattr(config, f) for f in DROPOUT_FIELDS if getattr(config, f, None) is not None}


class CosineHead(nn.Module):
    """logits = scale * cos(features, class weight vectors) (large-margin cosine classifier)."""

    def __init__(self, in_features, num_labels, scale=30.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_labels, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.scale = float(scale)

    def forward(self, x):
        return self.scale * F.linear(F.normalize(x, dim=-1), F.normalize(self.weight, dim=-1))


def use_cosine_head(hf, scale):
    old = hf.classifier
    head = CosineHead(old.in_features, old.out_features, scale).to(old.weight.device)
    with torch.no_grad():
        head.weight.copy_(old.weight)
    hf.classifier = head
    return hf


class PeerBERT(nn.Module):
    """Thin wrapper: forward(input_ids, attention_mask) -> logits."""

    def __init__(self, hf_model):
        super().__init__()
        self.hf = hf_model

    @property
    def config(self):
        return self.hf.config

    def forward(self, input_ids, attention_mask=None):
        return self.hf(input_ids=input_ids, attention_mask=attention_mask).logits

    def save(self, out_dir, tokenizer=None):
        self.hf.save_pretrained(out_dir)
        head = self.hf.classifier
        meta = ({"head": "cosine", "scale": head.scale} if isinstance(head, CosineHead) else {"head": "linear"})
        (Path(out_dir) / HEAD_FILE).write_text(json.dumps(meta))
        if tokenizer is not None:
            tokenizer.save_pretrained(out_dir)


def build_model(name_or_path, num_labels, class_names, dropout_on=True, dropout=0.1, head="linear",
                cosine_scale=30.0):
    """Returns (PeerBERT, tokenizer, info dict with verified dropout before/after)."""
    from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

    config = AutoConfig.from_pretrained(name_or_path)
    before = dropout_values(config)
    if hasattr(config, "reference_compile"):
        config.reference_compile = False
    if dropout_on:
        for f in before:
            setattr(config, f, float(dropout))
    config.num_labels = num_labels
    config.id2label = {i: n for i, n in enumerate(class_names)}
    config.label2id = {n: i for i, n in enumerate(class_names)}
    # The classification head is always new; a checkpoint that already has a head of another
    # size (e.g. 2 vs 3 labels) is re-initialised instead of raising.
    hf = AutoModelForSequenceClassification.from_pretrained(name_or_path, config=config,
                                                            ignore_mismatched_sizes=True)
    if head == "cosine":
        use_cosine_head(hf, cosine_scale)
    elif head != "linear":
        raise ValueError(f"unknown head '{head}' (linear | cosine)")
    tok = AutoTokenizer.from_pretrained(name_or_path)
    info = {"model": name_or_path, "head": head, "model_type": config.model_type, "dropout_before": before,
            "dropout_after": dropout_values(hf.config),
            "reference_compile": getattr(hf.config, "reference_compile", "n/a"),
            "num_parameters": sum(p.numel() for p in hf.parameters())}
    return PeerBERT(hf), tok, info


def load_trained(model_dir, eager_attention=False):
    """Load a saved checkpoint for inference (eager_attention=True for attention XAI)."""
    from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

    config = AutoConfig.from_pretrained(model_dir)
    if hasattr(config, "reference_compile"):
        config.reference_compile = False
    kwargs = {"attn_implementation": "eager"} if eager_attention else {}
    hf = AutoModelForSequenceClassification.from_pretrained(model_dir, config=config, **kwargs)
    meta_p = Path(model_dir) / HEAD_FILE
    meta = json.loads(meta_p.read_text()) if meta_p.exists() else {"head": "linear"}
    if meta["head"] == "cosine":
        use_cosine_head(hf, meta["scale"])     # classifier.weight was loaded into the Linear first
    return PeerBERT(hf), AutoTokenizer.from_pretrained(model_dir)


def has_active_dropout(model):
    return any(isinstance(m, nn.Dropout) and m.p > 0 for m in model.modules()) or any(
        v and v > 0 for v in dropout_values(model.config).values())


def passes_differ(model, batch):
    """Two train-mode passes on the same batch must differ when dropout is active."""
    was = model.training
    model.train()
    with torch.no_grad():
        a = model(batch["input_ids"], batch["attention_mask"])
        b = model(batch["input_ids"], batch["attention_mask"])
    model.train(was)
    return not torch.allclose(a.float(), b.float())

In [ ]:
%%writefile src/models/trainer.py
"""
Custom single-GPU PyTorch training loop for PeerBERT (no HF Trainer, no DataParallel).

- CUDA_VISIBLE_DEVICES must be set by the calling script BEFORE torch is imported
  (scripts 03/04/07 do this from --gpu / train.yaml gpu_id).
- AMP: bf16 when supported, else fp16 with GradScaler; "off" = fp32.
- Gradient accumulation, gradient clipping, linear warmup/decay.
- Non-finite loss guard: the batch is skipped (counted); too many in a row aborts.
- Validation every epoch; early stopping and best-checkpoint selection on val Macro-F1.
- Restart-safe: last_state.pt (resume at epoch boundaries) and done.json (skip finished runs).
- Saves: best/ (HF checkpoint + tokenizer), val_predictions.csv (best model),
  history.json, run_info.json (config, dropout check, truncation, AMP, timings).
"""

import copy
import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

from src.data.common import ROOT, jsonable, save_json, set_seed
from src.evaluation.metrics import binary_metrics, calibrate_threshold, multiclass_metrics
from src.evaluation.predict import prediction_frame, run_inference, softmax

from .dataset import ReviewDataset
from .losses import PeerBERTLoss, class_weights
from .modernbert_classifier import build_model, has_active_dropout, load_trained, passes_differ

CLASS_NAMES = {"intellabs": ("human", "ai"), "semdetect": ("human", "ai", "rewrite")}


# ---------------------------------------------------------------- config
def deep_merge(base, over):
    out = copy.deepcopy(base)
    for k, v in (over or {}).items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def load_train_config(path=None, overrides=None):
    path = Path(path) if path else ROOT / "configs" / "train.yaml"
    with path.open(encoding="utf-8") as f:
        return deep_merge(yaml.safe_load(f) or {}, overrides)


def amp_dtype(setting, device):
    if setting == "off" or device.type != "cuda":
        return None
    if setting in ("auto", "bf16") and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---------------------------------------------------------------- helpers
def _loader(ds, batch_size, shuffle, seed, workers):
    g = torch.Generator()
    g.manual_seed(seed)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=g,
                                       collate_fn=ds.collate, num_workers=workers,
                                       pin_memory=torch.cuda.is_available())


def _sorted_loader(ds, batch_size, workers):
    """Length-sorted order for fast inference; returns (loader, order)."""
    order = np.argsort([len(x) for x in ds.input_ids], kind="stable")
    sub = torch.utils.data.Subset(ds, order.tolist())
    return torch.utils.data.DataLoader(sub, batch_size=batch_size, shuffle=False, collate_fn=ds.collate,
                                       num_workers=workers), order


def predict_logits(model, ds, batch_size, device, dtype, workers=0):
    loader, order = _sorted_loader(ds, batch_size, workers)
    logits = run_inference(model, loader, device, dtype)
    out = np.empty_like(logits)
    out[order] = logits
    return out


def eval_metrics(y, proba, class_names):
    if len(class_names) == 2:
        return binary_metrics(y, proba[:, 1])
    return multiclass_metrics(y, proba, class_names)


def _optimizer(model, lr, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if p.ndim < 2 or "norm" in n.lower() or n.endswith("bias") else decay).append(p)
    return torch.optim.AdamW([{"params": decay, "weight_decay": wd},
                              {"params": no_decay, "weight_decay": 0.0}], lr=lr)


def _scheduler(opt, total_steps, warmup_ratio):
    warm = max(1, int(total_steps * warmup_ratio))

    def f(step):
        if step < warm:
            return (step + 1) / warm
        return max(0.0, (total_steps - step) / max(1, total_steps - warm))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


def _rng_state():
    return {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def _set_rng_state(s):
    random.setstate(s["py"])
    np.random.set_state(s["np"])
    torch.set_rng_state(s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(s["cuda"])


# ---------------------------------------------------------------- training
def train_one(train_df, val_df, tcfg, dataset, out_dir, seed, resume=True):
    """Train one seed. Returns run_info dict (also saved as run_info.json)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    if (out_dir / "done.json").exists():
        print(f"[train] {out_dir} already finished; skipping (delete done.json to retrain)")
        return json.loads((out_dir / "done.json").read_text())

    mcfg, tr, lcfg = tcfg["model"], tcfg["training"], tcfg["loss"]
    class_names = CLASS_NAMES[dataset]
    k = len(class_names)
    set_seed(seed)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    dtype = amp_dtype(tr.get("amp", "auto"), device)
    t0 = time.time()

    head = mcfg.get("head", "linear")
    if lcfg["type"] == "csm" and head != "cosine":
        raise ValueError("loss 'csm' needs the cosine head: set model.head: cosine")
    model, tok, model_info = build_model(mcfg["name"], k, class_names, mcfg.get("dropout_on", True),
                                         mcfg.get("dropout", 0.1), head, mcfg.get("cosine_scale", 30.0))
    if tr.get("gradient_checkpointing"):
        model.hf.gradient_checkpointing_enable()
        model_info["gradient_checkpointing"] = True
    model.to(device)
    ds_kw = dict(max_length=mcfg["max_length"], truncation=mcfg.get("truncation", "head"),
                 head_tokens=mcfg.get("head_tokens", 256))
    train_ds = ReviewDataset(train_df["text"].tolist(), train_df["label"].values, tok, **ds_kw)
    val_ds = ReviewDataset(val_df["text"].tolist(), val_df["label"].values, tok, **ds_kw)

    weights = class_weights(train_df["label"].values, k)
    loss_fn = PeerBERTLoss(lcfg["type"], weights, lcfg.get("rdrop_alpha", 0.0),
                           lcfg.get("label_smoothing", 0.05), lcfg.get("focal_gamma", 2.0),
                           {"scale": mcfg.get("cosine_scale", 30.0), "m_base": lcfg.get("csm_base_margin", 0.25),
                            "m_cost": lcfg.get("csm_cost_margin", 0.25)})
    if loss_fn.uses_rdrop and not has_active_dropout(model.hf):
        raise RuntimeError("R-Drop is on but the model has no active dropout (set model.dropout_on: true)")

    bs, accum = int(tr["batch_size"]), int(tr["grad_accum"])
    loader = _loader(train_ds, bs, True, seed, int(tr.get("num_workers", 0)))
    steps_per_epoch = math.ceil(len(loader) / accum)
    total_steps = steps_per_epoch * int(tr["epochs"])
    opt = _optimizer(model, float(tr["lr"]), float(tr["weight_decay"]))
    sched = _scheduler(opt, total_steps, float(tr["warmup_ratio"]))
    scaler = make_scaler(dtype == torch.float16)

    state = {"epoch_done": 0, "best": -1.0, "bad_epochs": 0, "history": [], "skipped_nonfinite": 0,
             "rdrop_passes_differ": None}
    state_p = out_dir / "last_state.pt"
    if resume and state_p.exists():
        ck = torch.load(state_p, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        state = ck["state"]
        _set_rng_state(ck["rng"])
        print(f"[train] resumed from {state_p} after epoch {state['epoch_done']}")

    print(f"[train] {dataset} seed={seed} device={device} amp={dtype} train={len(train_ds)} val={len(val_ds)} "
          f"loss={lcfg['type']} rdrop_alpha={loss_fn.alpha} dropout={model_info['dropout_after']}")
    bad_in_row = 0
    for epoch in range(state["epoch_done"] + 1, int(tr["epochs"]) + 1):
        model.train()
        loader = _loader(train_ds, bs, True, seed + epoch, int(tr.get("num_workers", 0)))
        run_loss, n_ok, te = 0.0, 0, time.time()
        opt.zero_grad(set_to_none=True)
        for i, batch in enumerate(loader):
            batch = {kk: v.to(device, non_blocking=True) for kk, v in batch.items()}
            if loss_fn.uses_rdrop and state["rdrop_passes_differ"] is None:
                state["rdrop_passes_differ"] = passes_differ(model, batch)
                if not state["rdrop_passes_differ"]:
                    raise RuntimeError("R-Drop: two train-mode passes are identical (dropout inactive)")
            try:
                with torch.autocast(device_type=device.type, dtype=dtype or torch.float32,
                                    enabled=dtype is not None):
                    l1 = model(batch["input_ids"], batch["attention_mask"])
                    l2 = model(batch["input_ids"], batch["attention_mask"]) if loss_fn.uses_rdrop else None
                loss, parts = loss_fn(l1, batch["labels"], l2)
            except torch.cuda.OutOfMemoryError as e:
                raise RuntimeError(
                    "CUDA out of memory. In configs/train.yaml lower training.batch_size (and raise "
                    "grad_accum to keep the effective batch), set training.gradient_checkpointing: true, "
                    "or reduce model.max_length. The run resumes from the last finished epoch.") from e
            if not torch.isfinite(loss):
                state["skipped_nonfinite"] += 1
                bad_in_row += 1
                opt.zero_grad(set_to_none=True)
                if bad_in_row >= int(tr.get("max_bad_steps", 20)):
                    raise RuntimeError(f"{bad_in_row} consecutive non-finite losses; aborting")
                continue
            bad_in_row = 0
            scaler.scale(loss / accum).backward()
            run_loss += float(loss.detach())
            n_ok += 1
            if (i + 1) % accum == 0 or (i + 1) == len(loader):
                scaler.unscale_(opt)
                gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), float(tr["max_grad_norm"]))
                if torch.isfinite(gnorm):
                    scaler.step(opt)
                else:
                    state["skipped_nonfinite"] += 1
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()

        val_logits = predict_logits(model, val_ds, int(tr["eval_batch_size"]), device, dtype)
        val_proba = softmax(val_logits)
        vm = eval_metrics(val_df["label"].values, val_proba, class_names)
        score = vm["macro_f1"]
        improved = score > state["best"]
        state["history"].append({"epoch": epoch, "train_loss": run_loss / max(n_ok, 1), "val_macro_f1": score,
                                 "val_accuracy": vm["accuracy"], "lr": sched.get_last_lr()[0],
                                 "seconds": round(time.time() - te, 1), "improved": improved})
        print(f"[train] epoch {epoch}: loss={run_loss / max(n_ok, 1):.4f} val_macro_f1={score:.4f}"
              f"{' *' if improved else ''}")
        if improved:
            state["best"], state["bad_epochs"] = score, 0
            model.save(out_dir / "best", tok)
            vp = prediction_frame(val_df, proba=val_proba, class_names=class_names)
            vp.to_csv(out_dir / "val_predictions.csv", index=False)
        else:
            state["bad_epochs"] += 1
        state["epoch_done"] = epoch
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "state": state, "rng": _rng_state()}, state_p)
        save_json(state["history"], out_dir / "history.json")
        if state["bad_epochs"] >= int(tr["early_stopping_patience"]):
            print(f"[train] early stopping after epoch {epoch}")
            break

    vp = pd.read_csv(out_dir / "val_predictions.csv")
    val_threshold = None
    if k == 2:
        val_threshold = calibrate_threshold(vp["label"], vp["score"], tcfg.get("threshold", {}).get("objective", "macro_f1"))
    info = {
        "dataset": dataset, "seed": seed, "best_val_macro_f1": state["best"], "epochs_run": state["epoch_done"],
        "val_threshold": val_threshold, "skipped_nonfinite_steps": state["skipped_nonfinite"],
        "rdrop_passes_differ": state["rdrop_passes_differ"], "amp": str(dtype), "device": str(device),
        "model_info": model_info, "class_weights": weights.tolist(), "train_config": tcfg,
        "truncation": {"train": train_ds.truncation_stats(), "val": val_ds.truncation_stats()},
        "minutes": round((time.time() - t0) / 60, 2), "best_dir": str(out_dir / "best"),
    }
    save_json(info, out_dir / "run_info.json")
    save_json(info, out_dir / "done.json")
    if state_p.exists():
        state_p.unlink()            # finished: the large optimizer state is no longer needed
    del model, opt, sched, scaler
    free_gpu()
    return jsonable(info)


def predict_proba(model_dir, texts, tcfg, eager_attention=False):
    """Probabilities from a saved checkpoint + truncation stats of these texts."""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    dtype = amp_dtype(tcfg["training"].get("amp", "auto"), device)
    model, tok = load_trained(model_dir, eager_attention)
    model.to(device)
    m = tcfg["model"]
    ds = ReviewDataset(list(texts), None, tok, m["max_length"], m.get("truncation", "head"), m.get("head_tokens", 256))
    proba = softmax(predict_logits(model, ds, int(tcfg["training"]["eval_batch_size"]), device, dtype))
    stats = ds.truncation_stats()
    del model
    free_gpu()
    return proba, stats


# ---------------------------------------------------------------- script entry
def model_root(cfg, dataset):
    from src.data.common import dataset_dir_name, resolve_path
    return resolve_path(cfg, "model_dir", "outputs/models") / dataset_dir_name(cfg, dataset)


def train_from_manifests(dataset, cfg, tcfg, seeds, tag="final", max_train_rows=None, max_val_rows=None):
    """
    Audit gate -> manifests -> one run per seed -> seed summary.
    Models: outputs/models/<ds>/<tag>/seed<k>/. Smoke-test limits keep whole papers.
    """
    from src.data.common import audit_dir, manifest_dir
    from src.data.leakage_audit import assert_audit_passed
    from src.data.splits import cap_by_papers
    from src.evaluation.aggregate import seed_summary

    status = assert_audit_passed(audit_dir(cfg, dataset))
    mdir = manifest_dir(cfg, dataset)
    train = pd.read_parquet(mdir / "train.parquet")
    val = pd.read_parquet(mdir / "val.parquet")
    if max_train_rows:
        train = cap_by_papers(train, max_train_rows, seeds[0])
    if max_val_rows:
        val = cap_by_papers(val, max_val_rows, seeds[0])
    root = model_root(cfg, dataset) / tag
    runs = []
    for s in seeds:
        runs.append(train_one(train, val, tcfg, dataset, root / f"seed{s}", s))
    summary = {"dataset": dataset, "tag": tag, "audit_status": status, "seeds": list(seeds),
               "train_rows": len(train), "val_rows": len(val),
               "smoke_limits": {"max_train_rows": max_train_rows, "max_val_rows": max_val_rows},
               "best_val_macro_f1": seed_summary([{"macro_f1": r["best_val_macro_f1"]} for r in runs],
                                                 keys=("macro_f1",)),
               "runs": [{k: r[k] for k in ("seed", "best_val_macro_f1", "epochs_run", "val_threshold",
                                           "best_dir", "minutes")} for r in runs]}
    save_json(summary, root / "seed_summary.json")
    print(f"[train] {dataset}/{tag}: val Macro-F1 over seeds = {summary['best_val_macro_f1'].get('macro_f1')}")
    return summary

In [ ]:
%%writefile src/xai/attention.py
"""
Attention-based explanations from an EAGER-attention copy of a trained PeerBERT
(sdpa/flash kernels do not return attention weights).

For one example (no padding) three token-importance vectors are returned:
  last_layer  attention FROM the pooled position TO each token in the last layer, mean over heads
  mean_layers the same, averaged over all layers
  rollout     attention rollout (Abnar & Zuidema, ACL 2020): per layer 0.5*A + 0.5*I
              (residual), rows renormalised, multiplied across layers
"Pooled position": ModernBERT's classifier pools with `classifier_pooling` = "cls" (row 0)
or "mean" (mean over all rows); the config value is read, not assumed.
Attention weights are not faithful explanations by themselves (Jain & Wallace, 2019);
they are reported next to Integrated Gradients and checked with faithfulness tests.
"""

import numpy as np
import torch


def pooling_rows(config, n):
    mode = getattr(config, "classifier_pooling", "cls")
    return (slice(0, 1), mode) if mode == "cls" else (slice(0, n), mode)


@torch.no_grad()
def attention_scores(model, input_ids, device):
    """input_ids: list[int] (one example). Returns dict of np arrays of length len(input_ids)."""
    ids = torch.tensor([input_ids], device=device)
    mask = torch.ones_like(ids)
    out = model.hf(input_ids=ids, attention_mask=mask, output_attentions=True)
    if not out.attentions or out.attentions[0] is None:
        raise RuntimeError("model returned no attentions; load it with eager_attention=True")
    att = torch.stack([a[0].float().mean(0) for a in out.attentions])     # (layers, n, n)
    n = att.shape[-1]
    rows, mode = pooling_rows(model.hf.config, n)
    last = att[-1, rows].mean(0)
    mean_layers = att[:, rows].mean(1).mean(0)
    eye = torch.eye(n, device=att.device)
    roll = eye
    for a in att:
        a = 0.5 * a + 0.5 * eye
        a = a / a.sum(-1, keepdim=True)
        roll = a @ roll
    rollout = roll[rows].mean(0)
    to_np = lambda x: x.detach().cpu().numpy().astype(float)  # noqa: E731
    return {"last_layer": to_np(last), "mean_layers": to_np(mean_layers), "rollout": to_np(rollout),
            "pooling": mode, "n_layers": int(att.shape[0])}


def spearman(a, b):
    """Rank correlation between two importance vectors (ties averaged)."""
    from scipy.stats import spearmanr
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 3 or np.allclose(a, a[0]) or np.allclose(b, b[0]):
        return None
    return float(spearmanr(a, b).statistic)

In [ ]:
%%writefile src/xai/faithfulness.py
"""
Faithfulness of token-importance explanations (ERASER metrics; DeYoung et al., ACL 2020).

For an importance vector over the NON-special tokens and a fraction k:
  comprehensiveness(k) = p(x) - p(x without its top-k% tokens)   (higher = more faithful)
  sufficiency(k)       = p(x) - p(only the top-k% tokens)         (lower  = more faithful)
p = probability of the explained class; special tokens are always kept; tokens are
deleted (not masked). A seeded RANDOM ranking is evaluated the same way as a reference:
an explanation is only informative if it beats random.
"""

import numpy as np
import torch


@torch.no_grad()
def prob_of(model, ids_list, target, device):
    out = []
    for ids in ids_list:
        x = torch.tensor([ids], device=device)
        out.append(float(torch.softmax(model(x, torch.ones_like(x)).float(), -1)[0, target]))
    return np.array(out)


def _subsets(input_ids, special_mask, ranking, k):
    content = [i for i, s in enumerate(special_mask) if not s]
    n_top = max(1, int(round(k * len(content)))) if content else 0
    top = set(ranking[:n_top])
    without = [t for i, t in enumerate(input_ids) if special_mask[i] or i not in top]
    only = [t for i, t in enumerate(input_ids) if special_mask[i] or i in top]
    return without, only


def faithfulness(model, input_ids, special_mask, importances, target, device, ks=(0.1, 0.2), seed=0):
    """
    importances: dict name -> importance array over all positions (larger = more important for
    the target class). Returns {name: {k: {comprehensiveness, sufficiency}}} incl. "random".
    """
    content = np.array([i for i, s in enumerate(special_mask) if not s])
    p_full = prob_of(model, [input_ids], target, device)[0]
    rng = np.random.default_rng(seed)
    rankings = {name: content[np.argsort(-np.asarray(imp)[content], kind="stable")].tolist()
                for name, imp in importances.items()}
    rankings["random"] = rng.permutation(content).tolist()
    out = {"p_full": float(p_full)}
    for name, ranking in rankings.items():
        res = {}
        for k in ks:
            without, only = _subsets(input_ids, special_mask, ranking, k)
            p_wo, p_only = prob_of(model, [without, only], target, device)
            res[str(k)] = {"comprehensiveness": float(p_full - p_wo), "sufficiency": float(p_full - p_only)}
        out[name] = res
    return out

In [ ]:
%%writefile src/xai/integrated_gradients.py
"""
Integrated Gradients (Sundararajan et al., ICML 2017) with Captum's LayerIntegratedGradients
on the token-embedding layer of a trained PeerBERT.

- Target: the class being explained (default: the model's predicted class).
- Output explained: softmax probability of the target class.
- Baseline: the same sequence with every non-special token replaced by the pad token
  (special tokens such as [CLS]/[SEP] are kept), so only the review's words are attributed.
- Token attribution = sum over the embedding dimension (signed: > 0 pushes towards the
  target class). The completeness gap |sum(attr) - (F(x) - F(baseline))| is returned as
  `delta` so convergence can be reported (raise n_steps if it is large).
"""

import numpy as np
import torch


def _forward(model):
    def f(input_ids, attention_mask, target):
        probs = torch.softmax(model(input_ids, attention_mask).float(), -1)
        return probs[:, target]
    return f


def ig_attributions(model, input_ids, special_mask, target, device, pad_id, n_steps=50, internal_batch_size=8):
    """
    input_ids: list[int]; special_mask: list[bool] (True = special token kept in the baseline).
    Returns (attributions np.array[len], delta float, p_target float, p_baseline float).
    """
    from captum.attr import LayerIntegratedGradients

    model.eval()
    ids = torch.tensor([input_ids], device=device)
    keep = torch.tensor([special_mask], device=device)
    base = torch.where(keep, ids, torch.full_like(ids, pad_id))
    mask = torch.ones_like(ids)
    fwd = _forward(model)
    lig = LayerIntegratedGradients(lambda i, m: fwd(i, m, target), model.hf.get_input_embeddings())
    attr, delta = lig.attribute(inputs=ids, baselines=base, additional_forward_args=(mask,), n_steps=n_steps,
                                internal_batch_size=internal_batch_size, return_convergence_delta=True)
    with torch.no_grad():
        p = float(fwd(ids, mask, target)[0])
        p0 = float(fwd(base, mask, target)[0])
    tok_attr = attr.sum(-1)[0].detach().float().cpu().numpy().astype(float)
    return tok_attr, float(delta.abs().item()), p, p0


def special_token_mask(tokenizer, input_ids):
    """
    True at the template positions the tokenizer adds (e.g. [CLS] first, [SEP] last), read from
    the tokenizer. Content tokens are never marked, even [UNK], so they are always attributed.
    """
    from src.models.dataset import special_template
    prefix, suffix = special_template(tokenizer)
    n = len(input_ids)
    return [i < len(prefix) or i >= n - len(suffix) for i in range(n)]


def display_tokens(tokenizer, input_ids):
    """Human-readable token strings (BPE markers removed; leading space kept as a word start)."""
    return [tokenizer.decode([i], clean_up_tokenization_spaces=False) for i in input_ids]


def normalise(x):
    x = np.asarray(x, float)
    m = np.abs(x).max()
    return x / m if m > 0 else x

In [ ]:
%%writefile src/xai/visualize.py
"""
HTML views of explanations (self-contained, no external assets).
Signed Integrated Gradients: red = pushes towards the explained class, blue = against it.
Attention: one-sided, grey scale.
"""

import html


def _span(tok, w, signed=True):
    a = min(1.0, abs(float(w)))
    if signed:
        color = f"rgba(220,38,38,{a:.2f})" if w > 0 else f"rgba(37,99,235,{a:.2f})"
    else:
        color = f"rgba(107,114,128,{a:.2f})"
    return f'<span title="{w:+.3f}" style="background:{color}">{html.escape(tok)}</span>'


def example_html(ex):
    """ex: dict from scripts/10_explain.py (tokens, ig_norm, attention_norm, labels, probs)."""
    toks = ex["tokens"]
    ig = "".join(_span(t, w) for t, w in zip(toks, ex["ig_norm"]))
    at = "".join(_span(t, w, signed=False) for t, w in zip(toks, ex["attention_norm"]))
    head = (f"<p><b>{html.escape(ex['row_id'])}</b> | true: {ex['label_name']} | predicted: {ex['pred_name']} "
            f"(p={ex['p_target']:.3f}) | explained class: {ex['target_name']} | IG delta: {ex['ig_delta']:.4f}</p>")
    return (f"<section style='margin-bottom:2em'>{head}<h4>Integrated Gradients</h4>"
            f"<div style='line-height:1.9;white-space:pre-wrap'>{ig}</div>"
            f"<h4>Attention ({html.escape(ex['attention_kind'])})</h4>"
            f"<div style='line-height:1.9;white-space:pre-wrap'>{at}</div></section>")


def page(title, examples):
    body = "\n".join(example_html(e) for e in examples)
    return (f"<!doctype html><html><head><meta charset='utf-8'><title>{html.escape(title)}</title></head>"
            f"<body style='font-family:system-ui,sans-serif;max-width:960px;margin:2em auto;padding:0 16px'>"
            f"<h2>{html.escape(title)}</h2><p>Red: pushes towards the explained class; blue: against it. "
            f"Hover a token for its normalised score.</p>{body}</body></html>")

In [ ]:
%%writefile scripts/00_inspect_datasets.py

"""
00_inspect_datasets.py
Read-only dataset inspection for AI-generated peer-review detection.

Default: IntelLabs only (calibration, test, extended).
Optional: --include-semdetect

Run from project root:
    python scripts/00_inspect_datasets.py
    python scripts/00_inspect_datasets.py --include-semdetect
    python scripts/00_inspect_datasets.py --skip-intellabs --include-semdetect

Only writes outputs/audit/inspection_report.json.
No training, dataset modification, or model generation.
"""

import argparse
import json
import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parent.parent
EXPECTED_SUBSETS = ("calibration", "test", "extended")
EXPECTED_LABELS = {0, 1}

FNAME_RE = re.compile(
    r"^(?P<conf>[A-Za-z]+)(?P<year>\d{4})\."
    r"(?P<subset>[A-Za-z]+)\.(?P<gen>.+)\.csv$"
)


def load_cfg():
    config_path = ROOT / "configs" / "config.yaml"
    if not config_path.exists():
        raise FileNotFoundError(f"Configuration not found: {config_path}")

    with config_path.open(encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def get_work_dir(cfg):
    return Path(
        os.environ.get("PRAD_WORK_DIR", cfg.get("paths", {}).get("work_dir", str(ROOT)))
    )


def jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if not np.isfinite(obj) else float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


def normalize_label(value):
    """Normalize common CSV representations without silently remapping labels."""
    if pd.isna(value):
        return None
    value = str(value).strip()
    if value in {"0", "0.0"}:
        return 0
    if value in {"1", "1.0"}:
        return 1
    return value


def analyze_intel_file(df, filename):
    """Summarize one IntelLabs CSV and flag unexpected data."""
    report = {
        "file": filename,
        "rows": int(len(df)),
        "columns": list(df.columns),
        "missing_required_columns": [
            col for col in ("label", "review") if col not in df.columns
        ],
    }

    if "label" in df.columns:
        labels = df["label"].map(normalize_label)
        report["label_counts"] = {
            str(k): int(v)
            for k, v in labels.value_counts(dropna=False).items()
        }
        observed = set(labels.dropna().tolist())
        report["observed_labels"] = sorted(observed, key=str)
        report["labels_match_expected_binary_values"] = observed.issubset(
            EXPECTED_LABELS
        )
        report["missing_labels"] = int(labels.isna().sum())

    if "review" in df.columns:
        text = df["review"].fillna("").astype(str).str.strip()
        lengths = text.str.len()
        report["empty_review_count"] = int(text.eq("").sum())
        report["review_length_chars"] = {
            "mean": float(lengths.mean()) if len(lengths) else 0.0,
            "median": float(lengths.median()) if len(lengths) else 0.0,
            "max": int(lengths.max()) if len(lengths) else 0,
        }

        if "label" in df.columns:
            temp = pd.DataFrame({"label": df["label"].map(normalize_label),
                                 "length": lengths})
            report["mean_review_length_by_label"] = {
                str(k): float(v)
                for k, v in temp.groupby("label", dropna=False)["length"].mean().items()
            }

    for col in ("PaperId", "review-id", "custom_id"):
        if col in df.columns:
            ids = df[col]
            non_null = ids.dropna()
            report[f"{col}_stats"] = {
                "non_null": int(ids.notna().sum()),
                "unique_non_null": int(non_null.nunique()),
                "duplicate_non_null_rows": int(non_null.duplicated().sum()),
                "dtype": str(ids.dtype),
            }

    if "review-id" in df.columns and "label" in df.columns:
        valid = df[["review-id", "label"]].dropna(subset=["review-id"]).copy()
        valid["label"] = valid["label"].map(normalize_label)
        label_sets = valid.groupby("review-id")["label"].agg(
            lambda values: tuple(sorted(set(values), key=str))
        )
        report["review_id_label_patterns"] = {
            str(k): int(v)
            for k, v in label_sets.value_counts().head(10).items()
        }
        report["review_ids_associated_with_multiple_labels"] = int(
            (label_sets.map(len) > 1).sum()
        )

    return report


def inspect_intellabs(cfg):
    from huggingface_hub import hf_hub_download, list_repo_files

    ic = cfg.get("data", {}).get("intellabs", {})
    repo_id = ic.get("repo_id", "IntelLabs/AI-Peer-Review-Detection-Benchmark")

    print(f"\n=== Inspecting IntelLabs: {repo_id} ===")

    files = [
        f for f in list_repo_files(repo_id, repo_type="dataset")
        if f.lower().endswith(".csv")
    ]

    # Index files by official subset, generator and conference/year.
    indexed = defaultdict(list)
    layout = defaultdict(lambda: defaultdict(int))
    for path in files:
        parts = path.split("/")
        if len(parts) < 3:
            continue

        subset, generator = parts[0], parts[1]
        match = FNAME_RE.match(Path(path).name)
        if not match:
            continue

        info = match.groupdict()
        if info["subset"].lower() != subset.lower():
            continue

        conf_year = f'{info["conf"]}{info["year"]}'
        indexed[(subset, conf_year)].append({
            "path": path,
            "generator": generator,
            "conf_year": conf_year,
        })
        layout[subset][generator] += 1

    report = {
        "repo_id": repo_id,
        "expected_label_mapping": {
            "0": "human-written",
            "1": "AI-generated",
        },
        "label_mapping_source": "Dataset card supplied for this experiment",
        "subset_file_counts": {
            subset: dict(generators)
            for subset, generators in layout.items()
        },
        "subsets_found": sorted(layout.keys()),
        "subsets_missing": [
            s for s in EXPECTED_SUBSETS if s not in layout
        ],
        "files": [],
        "subset_summaries": {},
        "overlap_audits": {},
        "errors": [],
    }

    # Use configured conference-years when supplied; otherwise inspect all.
    inspection_cfg = cfg.get("inspection", {})
    configured_years = inspection_cfg.get("sample_conf_years")
    wanted_years = set(map(str, configured_years)) if configured_years else None

    selected = []
    for (subset, conf_year), entries in sorted(indexed.items()):
        if subset not in EXPECTED_SUBSETS:
            continue
        if wanted_years and conf_year not in wanted_years:
            continue
        selected.extend(entries)

    cache = {}

    def fetch(path):
        if path not in cache:
            local_path = hf_hub_download(
                repo_id=repo_id,
                filename=path,
                repo_type="dataset",
            )
            cache[path] = pd.read_csv(local_path, low_memory=False)
        return cache[path]

    # Inspect each selected file, including all generator folders.
    subset_frames = defaultdict(list)
    human_review_ids = defaultdict(set)
    human_review_texts = defaultdict(set)
    paper_ids = defaultdict(set)

    for entry in selected:
        path = entry["path"]
        subset = path.split("/")[0]
        generator = entry["generator"]
        conf_year = entry["conf_year"]

        try:
            df = fetch(path)
            item = analyze_intel_file(df, path)
            item.update({
                "subset": subset,
                "generator_folder": generator,
                "conference_year": conf_year,
            })
            report["files"].append(item)
            subset_frames[subset].append(df)

            print(
                f"[{subset}] {path}: rows={len(df)}, "
                f"labels={item.get('label_counts', 'missing')}"
            )

            # Track papers and human reviews to detect repeated samples.
            if "PaperId" in df.columns:
                paper_ids[(subset, conf_year)].update(
                    df["PaperId"].dropna().astype(str)
                )

            if "label" in df.columns:
                labels = df["label"].map(normalize_label)
                humans = df.loc[labels.eq(0)]

                if "review-id" in humans.columns:
                    human_review_ids[(subset, conf_year)].update(
                        humans["review-id"].dropna().astype(str)
                    )

                # Text fingerprints help find repeated human reviews even
                # when review IDs are missing or inconsistent.
                if "review" in humans.columns:
                    texts = (
                        humans["review"].dropna().astype(str).str.strip()
                    )
                    human_review_texts[(subset, conf_year)].update(
                        texts[texts.ne("")].tolist()
                    )

        except Exception as exc:
            error = {"file": path, "error": repr(exc)}
            report["errors"].append(error)
            print(f"ERROR reading {path}: {exc!r}")

    # Summarize rows and labels by subset. These totals include repeated
    # records from generator-specific files, so they are raw file totals.
    for subset, frames in subset_frames.items():
        total_rows = sum(len(df) for df in frames)
        label_counts = defaultdict(int)
        missing_review_column = 0
        missing_review_rows = 0

        for df in frames:
            if "label" in df.columns:
                for label, count in df["label"].map(normalize_label).value_counts(
                    dropna=False
                ).items():
                    label_counts[str(label)] += int(count)

            if "review" not in df.columns:
                missing_review_column += 1
            else:
                missing_review_rows += int(
                    df["review"].isna().sum()
                    + df["review"].fillna("").astype(str).str.strip().eq("").sum()
                )

        report["subset_summaries"][subset] = {
            "files_inspected": len(frames),
            "raw_rows_across_files": int(total_rows),
            "label_counts_across_files": dict(label_counts),
            "files_missing_review_column": missing_review_column,
            "missing_or_empty_review_cells": missing_review_rows,
            "note": "Raw totals may count repeated human reviews in multiple generator files.",
        }

    # Compare available subsets for each conference/year.
    overlap = {}
    all_years = sorted({cy for _, cy in indexed})
    for cy in all_years:
        if wanted_years and cy not in wanted_years:
            continue

        subset_results = {}
        for subset in EXPECTED_SUBSETS:
            ids = human_review_ids.get((subset, cy), set())
            texts = human_review_texts.get((subset, cy), set())
            papers = paper_ids.get((subset, cy), set())
            subset_results[subset] = {
                "unique_human_review_ids": len(ids),
                "unique_human_review_texts": len(texts),
                "unique_paper_ids": len(papers),
            }

        for i, subset_a in enumerate(EXPECTED_SUBSETS):
            for subset_b in EXPECTED_SUBSETS[i + 1:]:
                ids_a = human_review_ids.get((subset_a, cy), set())
                ids_b = human_review_ids.get((subset_b, cy), set())
                texts_a = human_review_texts.get((subset_a, cy), set())
                texts_b = human_review_texts.get((subset_b, cy), set())
                papers_a = paper_ids.get((subset_a, cy), set())
                papers_b = paper_ids.get((subset_b, cy), set())

                subset_results[f"{subset_a}_vs_{subset_b}"] = {
                    "shared_human_review_ids": len(ids_a & ids_b),
                    "shared_human_review_texts": len(texts_a & texts_b),
                    "shared_paper_ids": len(papers_a & papers_b),
                }

        overlap[cy] = subset_results

    report["overlap_audits"] = overlap

    report["inspection_scope"] = {
        "conference_year_filter": sorted(wanted_years) if wanted_years else "all",
        "subsets": list(EXPECTED_SUBSETS),
        "files_selected": len(selected),
        "files_successfully_inspected": len(report["files"]),
        "failed_files": len(report["errors"]),
    }

    print("\n=== IntelLabs subset summaries ===")
    print(json.dumps(jsonable(report["subset_summaries"]), indent=2))
    print("\n=== Inspection scope ===")
    print(json.dumps(report["inspection_scope"], indent=2))

    return report


# ----------------------------------------------------------------------------
# Optional Sem-Detect inspection (not needed for the current binary stage)
# ----------------------------------------------------------------------------
def analyze_semdetect(df):
    required = {"class", "split", "paper_id", "clean_review"}
    missing = sorted(required - set(df.columns))

    report = {
        "rows": int(len(df)),
        "columns": list(df.columns),
        "missing_required_columns": missing,
    }

    if "class" not in df.columns or "split" not in df.columns:
        return report

    report["class_counts"] = {
        str(k): int(v)
        for k, v in df["class"].value_counts(dropna=False).items()
    }
    report["split_counts"] = {
        str(k): int(v)
        for k, v in df["split"].value_counts(dropna=False).items()
    }
    report["class_by_split"] = {
        str(split): {
            str(cls): int(count)
            for cls, count in counts.items()
        }
        for split, counts in pd.crosstab(df["split"], df["class"]).iterrows()
    }

    if "paper_id" in df.columns:
        tr = df.loc[df["split"] == "train"]
        te = df.loc[df["split"] == "test"]
        report["paper_id_overlap_train_test"] = len(
            set(tr["paper_id"].dropna()) & set(te["paper_id"].dropna())
        )

    if "clean_review" in df.columns:
        text = df["clean_review"].fillna("").astype(str).str.strip()
        report["empty_review_count"] = int(text.eq("").sum())

    return report


def inspect_semdetect(cfg):
    from huggingface_hub import hf_hub_download, list_repo_files

    sc = cfg.get("data", {}).get("semdetect", {})
    repo_id = sc.get("repo_id", "Sem-Detect/ML_Conferences-Peer-Reviews")
    config_name = sc.get("config_name", "reviews")

    files = [
        f for f in list_repo_files(repo_id, repo_type="dataset")
        if f.startswith(f"{config_name}/") and f.endswith(".parquet")
    ]

    if not files:
        raise FileNotFoundError(
            f"No Parquet files found under {config_name}/ in {repo_id}"
        )

    frames = []
    for path in files:
        local_path = hf_hub_download(
            repo_id=repo_id,
            filename=path,
            repo_type="dataset",
        )
        frames.append(pd.read_parquet(local_path))

    df = pd.concat(frames, ignore_index=True)
    report = {
        "repo_id": repo_id,
        "config": config_name,
        "files": files,
        "analysis": analyze_semdetect(df),
    }
    print("\n=== Optional Sem-Detect inspection ===")
    print(json.dumps(jsonable(report["analysis"]), indent=2))
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--skip-intellabs",
        action="store_true",
        help="Skip IntelLabs inspection.",
    )
    parser.add_argument(
        "--include-semdetect",
        action="store_true",
        help="Also inspect Sem-Detect; not required for the current stage.",
    )
    args = parser.parse_args()

    if args.skip_intellabs and not args.include_semdetect:
        parser.error("Nothing to inspect. Enable --include-semdetect.")

    cfg = load_cfg()
    report = {}

    if not args.skip_intellabs:
        report["intellabs"] = inspect_intellabs(cfg)

    if args.include_semdetect:
        report["semdetect"] = inspect_semdetect(cfg)

    paths_cfg = cfg.get("paths", {})
    audit_dir_name = paths_cfg.get("audit_dir", "outputs/audit")
    audit_dir = Path(audit_dir_name)
    if not audit_dir.is_absolute():
        audit_dir = get_work_dir(cfg) / audit_dir

    audit_dir.mkdir(parents=True, exist_ok=True)
    output_path = audit_dir / "inspection_report.json"

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(jsonable(report), f, indent=2, ensure_ascii=False)

    print(f"\nInspection report saved to: {output_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/01_run_audit.py
"""
01_run_audit.py
Run the leakage audit only (no manifests). Thin wrapper around
`02_prepare_splits.py --audit-only` so the audit logic lives in one place.

  python scripts/01_run_audit.py                    # both datasets
  python scripts/01_run_audit.py --dataset semdetect
"""

import importlib.util
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent

if __name__ == "__main__":
    spec = importlib.util.spec_from_file_location("prepare_splits", HERE / "02_prepare_splits.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    sys.exit(mod.main(["--audit-only"] + sys.argv[1:]))

In [ ]:
%%writefile scripts/02_prepare_splits.py
"""
02_prepare_splits.py
Load both datasets, split by paper, run the leakage audit on the FULL partitions,
then apply whole-paper compute caps and write manifests.

Outputs (per dataset folder: intellabs | sem_detect):
  outputs/audit/<ds>/leakage_audit.json, audit_status.json
  outputs/manifests/<ds>/{train,val,test,test_baseline[,extended]}.parquet
  outputs/manifests/<ds>/exclusions.parquet, summary.json, loader_stats.json

Run from the project root:
  python scripts/02_prepare_splits.py                       # both datasets
  python scripts/02_prepare_splits.py --dataset intellabs
  python scripts/02_prepare_splits.py --audit-only          # audit, no manifests
  python scripts/02_prepare_splits.py --verify-only         # re-check saved manifests (offline)
Exit codes: 0 ok, 2 audit FAIL (no manifests written), 3 manifest verification FAIL.
"""

import argparse
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

import pandas as pd  # noqa: E402

from src.data.common import (  # noqa: E402
    ExclusionLog, audit_dir, config_hash, load_config, load_json, manifest_dir, save_json, set_seed,
)
from src.data.intellabs_loader import load_subset  # noqa: E402
from src.data.leakage_audit import print_summary, run_audit, save_audit  # noqa: E402
from src.data.sem_detect_loader import load_semdetect  # noqa: E402
from src.data.splits import cap_by_papers, check_invariants, split_by_paper  # noqa: E402

DATASETS = ("intellabs", "semdetect")


def binary_rows(df):
    """Rows usable by binary baselines (Sem-Detect: human/ai only, rewrite excluded)."""
    return df["label"].isin([0, 1])


def full_partitions_intellabs(cfg, seed, use_cache):
    icfg = cfg["data"]["intellabs"]
    log, stats = ExclusionLog(), {}
    pool, stats["calibration"], l1 = load_subset(cfg, icfg["train_source_subset"], use_cache=use_cache)
    test, stats["test"], l2 = load_subset(cfg, icfg["test_subset"], use_cache=use_cache)
    log.extend(l1)
    log.extend(l2)
    train, val = split_by_paper(pool, icfg["val_fraction_papers"], seed)
    parts = {"train": train, "val": val, "test": test}
    if icfg.get("use_extended"):
        ext, stats["extended"], l3 = load_subset(cfg, icfg["extended_subset"], use_cache=use_cache)
        log.extend(l3)
        seen = set(train["paper_key"]) | set(val["paper_key"])
        overlap = ext["paper_key"].isin(seen)
        log.add("intellabs", "extended", ext.loc[overlap, "row_id"], "paper_in_train_or_val",
                "extended rows whose paper appears in train/val are excluded from extended evaluation")
        stats["extended_rows_excluded_for_train_val_overlap"] = int(overlap.sum())
        parts["extended"] = ext.loc[~overlap].copy()
    caps = {"train": icfg.get("train_max_rows"), "val": icfg.get("val_max_rows"),
            "test": icfg.get("model_eval_max_rows"), "test_baseline": icfg.get("baseline_eval_max_rows"),
            "extended": icfg.get("extended_max_rows")}
    return parts, caps, stats, log


def full_partitions_semdetect(cfg, seed, use_cache):
    scfg = cfg["data"]["semdetect"]
    df, stats, log = load_semdetect(cfg, use_cache=use_cache)
    train_pool = df[df["partition"] == "train"]
    test = df[df["partition"] == "test"].copy()          # official test, never deduplicated
    train, val = split_by_paper(train_pool, scfg["val_fraction_papers"], seed)
    caps = {"train": scfg.get("train_max_rows"), "val": scfg.get("val_max_rows"),
            "test": scfg.get("model_eval_max_rows"), "test_baseline": scfg.get("baseline_eval_max_rows")}
    return {"train": train, "val": val, "test": test}, caps, {"all": stats}, log


def apply_caps(parts, caps, seed, dataset, log):
    out = {}
    for name in ("train", "val", "test", "extended"):
        if name not in parts:
            continue
        capped = cap_by_papers(parts[name], caps.get(name), seed)
        dropped = parts[name].loc[~parts[name]["row_id"].isin(capped["row_id"])]
        log.add(dataset, name, dropped["row_id"], "compute_cap", f"whole-paper cap {caps.get(name)}")
        out[name] = capped
    # Baseline subset nested inside the (capped) model test set; for Sem-Detect only
    # human/ai rows count toward the cap (rewrite rows of chosen papers are kept for diagnostics).
    mask = binary_rows(out["test"]) if dataset == "semdetect" else None
    out["test_baseline"] = cap_by_papers(out["test"], caps.get("test_baseline"), seed, count_mask=mask)
    return out


def describe(parts):
    return {n: {"rows": len(df), "papers": int(df["paper_key"].nunique()),
                "label_counts": {str(k): int(v) for k, v in df["label"].value_counts().sort_index().items()}}
            for n, df in parts.items()}


def slim_stats(stats):
    return {k: ({kk: vv for kk, vv in v.items() if kk != "file_stats"} if isinstance(v, dict) else v)
            for k, v in stats.items()}


def prepare(dataset, cfg, audit_only, use_cache):
    seed = int(cfg["project"]["seed"])
    set_seed(seed)
    builder = full_partitions_intellabs if dataset == "intellabs" else full_partitions_semdetect
    parts, caps, stats, log = builder(cfg, seed, use_cache)

    print(f"\n[{dataset}] full partitions: {describe(parts)}")
    report = run_audit(parts, dataset, cfg.get("audit", {}), seed=seed,
                       extra_info={"loader": slim_stats(stats), "full_partitions": describe(parts)})
    adir = audit_dir(cfg, dataset)
    save_audit(report, adir)
    print_summary(report)
    print(f"  report: {adir / 'leakage_audit.json'}")
    if report["status"] == "FAIL":
        print(f"[{dataset}] audit FAILED: manifests NOT written. Fix the data issue; "
              f"never edit the official test split to pass.")
        return 2
    if audit_only:
        return 0

    capped = apply_caps(parts, caps, seed, dataset, log)
    mdir = manifest_dir(cfg, dataset)
    mdir.mkdir(parents=True, exist_ok=True)
    for name, df in capped.items():
        df.reset_index(drop=True).to_parquet(mdir / f"{name}.parquet", index=False)
    log.frame().to_parquet(mdir / "exclusions.parquet", index=False)
    save_json(stats, mdir / "loader_stats.json")
    summary = {
        "dataset": dataset, "seed": seed, "caps": caps,
        "config_hash": config_hash(cfg["data"][dataset]), "audit_status": report["status"],
        "full_partitions": describe(parts), "manifests": describe(capped),
        "exclusions": log.summary(),
        "notes": [
            "Audit ran on full partitions before caps; caps keep whole papers.",
            "test_baseline is nested inside test; all methods are compared on it.",
        ] + (["Sem-Detect test_baseline cap counts human/ai rows only; rewrite rows are "
              "kept for the descriptive 'fraction flagged as AI' diagnostic."]
             if dataset == "semdetect" else []),
    }
    save_json(summary, mdir / "summary.json")
    print(f"[{dataset}] manifests: {describe(capped)}\n  written to {mdir}")
    return verify(dataset, cfg)


def verify(dataset, cfg):
    mdir = manifest_dir(cfg, dataset)
    sp = mdir / "summary.json"
    if not sp.exists():
        print(f"[{dataset}] no manifests at {mdir}")
        return 3
    summary = load_json(sp)
    parts = {p.stem: pd.read_parquet(p) for p in sorted(mdir.glob("*.parquet")) if p.stem != "exclusions"}
    mask = binary_rows if dataset == "semdetect" else None
    fails = check_invariants(parts, summary["caps"], baseline_count_mask=mask)
    status_p = audit_dir(cfg, dataset) / "audit_status.json"
    audit_status = load_json(status_p)["status"] if status_p.exists() else "MISSING"
    if audit_status in ("FAIL", "MISSING"):
        fails.append(f"audit status is {audit_status}")
    print(f"[{dataset}] verify: {'PASS' if not fails else 'FAIL'} "
          f"(audit {audit_status}; manifests {describe(parts)})")
    for f in fails:
        print(f"  - {f}")
    return 0 if not fails else 3


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", choices=DATASETS + ("all",), default="all")
    ap.add_argument("--audit-only", action="store_true", help="run the audit; write no manifests")
    ap.add_argument("--verify-only", action="store_true", help="re-check saved manifests; no network")
    ap.add_argument("--no-cache", action="store_true", help="ignore the parquet cache and reload")
    ap.add_argument("--config", default=None)
    args = ap.parse_args(argv)
    if args.audit_only and args.verify_only:
        ap.error("--audit-only and --verify-only are exclusive")

    cfg = load_config(args.config)
    codes = []
    for ds in (DATASETS if args.dataset == "all" else (args.dataset,)):
        if args.verify_only:
            codes.append(verify(ds, cfg))
        else:
            codes.append(prepare(ds, cfg, args.audit_only, not args.no_cache))
    return max(codes)


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/03_train_intellabs.py
"""
03_train_intellabs.py
Train PeerBERT (ModernBERT-base) on the intellabs manifests. Training is blocked unless
outputs/audit/<ds>/audit_status.json exists and is not FAIL (scripts/02_prepare_splits.py).

  python scripts/03_train_intellabs.py                          # all seeds in configs/train.yaml
  python scripts/03_train_intellabs.py --seeds 42               # one seed
  python scripts/03_train_intellabs.py --smoke                  # tiny whole-paper subset, 1 epoch (pipeline check)
Restart-safe: rerun the same command to resume (last_state.pt) or skip finished seeds (done.json).
"""

import argparse
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
DATASET = "intellabs"


def parse():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--seeds", type=int, nargs="+", default=None)
    ap.add_argument("--tag", default="final", help="output sub-folder under outputs/models/<ds>/")
    ap.add_argument("--gpu", type=int, default=None, help="GPU index (overrides train.yaml gpu_id)")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--model", default=None, help="override model name/path (e.g. local copy)")
    ap.add_argument("--max-train-rows", type=int, default=None)
    ap.add_argument("--max-val-rows", type=int, default=None)
    ap.add_argument("--smoke", action="store_true", help="400 train / 200 val rows, 1 epoch, tag=smoke")
    ap.add_argument("--config", default=None)
    ap.add_argument("--train-config", default=None)
    return ap.parse_args()


def main():
    args = parse()
    import yaml
    tc_path = Path(args.train_config) if args.train_config else ROOT / "configs" / "train.yaml"
    gpu = args.gpu if args.gpu is not None else (yaml.safe_load(tc_path.read_text())["training"].get("gpu_id", 0))
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)   # must be set before torch is imported

    from src.data.common import load_config
    from src.models.trainer import load_train_config, train_from_manifests

    over = {}
    if args.epochs or args.smoke:
        over.setdefault("training", {})["epochs"] = 1 if args.smoke else args.epochs
    if args.model:
        over.setdefault("model", {})["name"] = args.model
    tcfg = load_train_config(tc_path, over)
    cfg = load_config(args.config)
    seeds = args.seeds or tcfg["training"]["seeds"]
    if args.smoke:
        seeds = seeds[:1]
    train_from_manifests(DATASET, cfg, tcfg, seeds, "smoke" if args.smoke else args.tag,
                         400 if args.smoke else args.max_train_rows, 200 if args.smoke else args.max_val_rows)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/04_train_semdetect.py
"""
04_train_semdetect.py
Train PeerBERT (ModernBERT-base) on the semdetect manifests. Training is blocked unless
outputs/audit/<ds>/audit_status.json exists and is not FAIL (scripts/02_prepare_splits.py).

  python scripts/04_train_semdetect.py                          # all seeds in configs/train.yaml
  python scripts/04_train_semdetect.py --seeds 42               # one seed
  python scripts/04_train_semdetect.py --smoke                  # tiny whole-paper subset, 1 epoch (pipeline check)
Restart-safe: rerun the same command to resume (last_state.pt) or skip finished seeds (done.json).
"""

import argparse
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
DATASET = "semdetect"


def parse():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--seeds", type=int, nargs="+", default=None)
    ap.add_argument("--tag", default="final", help="output sub-folder under outputs/models/<ds>/")
    ap.add_argument("--gpu", type=int, default=None, help="GPU index (overrides train.yaml gpu_id)")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--model", default=None, help="override model name/path (e.g. local copy)")
    ap.add_argument("--max-train-rows", type=int, default=None)
    ap.add_argument("--max-val-rows", type=int, default=None)
    ap.add_argument("--smoke", action="store_true", help="400 train / 200 val rows, 1 epoch, tag=smoke")
    ap.add_argument("--config", default=None)
    ap.add_argument("--train-config", default=None)
    return ap.parse_args()


def main():
    args = parse()
    import yaml
    tc_path = Path(args.train_config) if args.train_config else ROOT / "configs" / "train.yaml"
    gpu = args.gpu if args.gpu is not None else (yaml.safe_load(tc_path.read_text())["training"].get("gpu_id", 0))
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)   # must be set before torch is imported

    from src.data.common import load_config
    from src.models.trainer import load_train_config, train_from_manifests

    over = {}
    if args.epochs or args.smoke:
        over.setdefault("training", {})["epochs"] = 1 if args.smoke else args.epochs
    if args.model:
        over.setdefault("model", {})["name"] = args.model
    tcfg = load_train_config(tc_path, over)
    cfg = load_config(args.config)
    seeds = args.seeds or tcfg["training"]["seeds"]
    if args.smoke:
        seeds = seeds[:1]
    train_from_manifests(DATASET, cfg, tcfg, seeds, "smoke" if args.smoke else args.tag,
                         400 if args.smoke else args.max_train_rows, 200 if args.smoke else args.max_val_rows)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/05_evaluate.py
"""
05_evaluate.py
Evaluate trained PeerBERT checkpoints on test, test_baseline and (IntelLabs, optional) extended.

Per seed:
- IntelLabs (binary): metrics at threshold 0.5 and at the VALIDATION-calibrated threshold,
  per-generator and per-conference-year breakdowns, paper-level cluster bootstrap CIs.
- Sem-Detect: native 3-class metrics AND a human/ai binary view (rewrite rows excluded;
  score = P(ai) / (P(human)+P(ai))), with the binary threshold calibrated on the
  validation binary view.
- Error analysis (FP/FN tables, per-class / conference-year breakdowns) on test.
Across seeds: mean +/- std. Comparison rows (type "trained classifier") are merged into
outputs/results/<ds>/comparison_table.{csv,json}.

  python scripts/05_evaluate.py --dataset intellabs
  python scripts/05_evaluate.py --dataset semdetect --seeds 42 --n-boot 200
Predictions are cached per seed/split; rerunning skips finished predictions.
"""

import argparse
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))


def parse():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", choices=("intellabs", "semdetect"), required=True)
    ap.add_argument("--tag", default="final")
    ap.add_argument("--seeds", type=int, nargs="+", default=None)
    ap.add_argument("--splits", nargs="+", default=["test", "test_baseline", "extended"])
    ap.add_argument("--n-boot", type=int, default=1000)
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--config", default=None)
    ap.add_argument("--train-config", default=None)
    return ap.parse_args()


def evaluate_split(pred, dataset, val_thr, n_boot, seed, n_words=None):
    """pred: prediction frame for one split -> metrics dict."""
    from src.evaluation.aggregate import by_generator, by_group, by_length_bin
    from src.evaluation.metrics import binary_metrics, multiclass_metrics
    from src.evaluation.predict import binary_view
    from src.evaluation.stats import auc_fn, cluster_bootstrap_ci, macro_f1_fn

    out = {}
    if dataset == "semdetect":
        names = ("human", "ai", "rewrite")
        out["native_3class"] = multiclass_metrics(pred["label"], pred[[f"proba_{c}" for c in names]].values, names)
        b = binary_view(pred)
    else:
        b = pred
    out["binary@0.5"] = binary_metrics(b["label"], b["score"], 0.5)
    out["binary@val_threshold"] = binary_metrics(b["label"], b["score"], val_thr)
    out["binary_note"] = ("human/ai rows only; rewrite excluded; score = P(ai)/(P(human)+P(ai))"
                          if dataset == "semdetect" else "positive class = AI")
    out["by_conf_year@0.5"] = by_group(b, "conf_year")
    if n_words is not None:
        out["by_length_tercile@0.5"] = by_length_bin(b, b["row_id"].map(n_words).values)
    if dataset == "intellabs":
        out["by_generator@0.5"] = by_generator(b)
    if n_boot:
        out["cluster_bootstrap_ci"] = {
            "roc_auc": cluster_bootstrap_ci(b["label"], b["score"], b["paper_key"], auc_fn, n_boot, seed=seed),
            "macro_f1@0.5": cluster_bootstrap_ci(b["label"], b["score"], b["paper_key"], macro_f1_fn(0.5),
                                                 n_boot, seed=seed),
        }
    return out


def main():
    args = parse()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)
    import pandas as pd

    from src.data.common import dataset_dir_name, load_config, load_json, manifest_dir, resolve_path, save_json
    from src.evaluation.aggregate import (
        comparison_row, merge_comparison_rows, save_comparison_table, seed_summary,
    )
    from src.evaluation.error_analysis import save_error_analysis, text_lookup
    from src.evaluation.metrics import calibrate_threshold
    from src.evaluation.predict import binary_view, load_predictions, prediction_frame, save_predictions
    from src.models.trainer import CLASS_NAMES, load_train_config, model_root, predict_proba

    cfg = load_config(args.config)
    tcfg = load_train_config(args.train_config)
    ds = args.dataset
    names = CLASS_NAMES[ds]
    seeds = args.seeds or tcfg["training"]["seeds"]
    mdir = manifest_dir(cfg, ds)
    res_root = resolve_path(cfg, "results_dir", "outputs/results") / dataset_dir_name(cfg, ds)
    out_root = res_root / "peerbert" / args.tag
    manifests = {s: pd.read_parquet(mdir / f"{s}.parquet") for s in args.splits if (mdir / f"{s}.parquet").exists()}
    missing = [s for s in args.splits if s not in manifests]
    if missing:
        print(f"[eval] skipping missing manifests: {missing}")

    per_seed, rows = {}, []
    for seed in seeds:
        run_dir = model_root(cfg, ds) / args.tag / f"seed{seed}"
        if not (run_dir / "done.json").exists():
            print(f"[eval] {run_dir} has no finished model; skipping seed {seed}")
            continue
        info = load_json(run_dir / "run_info.json")
        val_pred = pd.read_csv(run_dir / "val_predictions.csv")
        if ds == "semdetect":
            vb = binary_view(val_pred)
            val_thr = calibrate_threshold(vb["label"], vb["score"], tcfg["threshold"]["objective"])
        else:
            val_thr = info["val_threshold"]
        seed_out = {"val_threshold": val_thr, "model_dir": info["best_dir"], "splits": {}}
        for split, man in manifests.items():
            pp = out_root / f"seed{seed}" / f"{split}_predictions.csv"
            if pp.exists():
                pred = load_predictions(pp)
                trunc = load_json(pp.with_suffix(".truncation.json"))
            else:
                proba, trunc = predict_proba(run_dir / "best", man["text"].tolist(), tcfg)
                pred = prediction_frame(man, proba=proba, class_names=names)
                save_predictions(pred, pp)
                save_json(trunc, pp.with_suffix(".truncation.json"))
            n_words = pd.Series(man["text"].str.split().str.len().values, index=man["row_id"].values)
            m = evaluate_split(pred, ds, val_thr, args.n_boot if split != "test_baseline" else 0, seed, n_words)
            m["truncation"] = trunc
            seed_out["splits"][split] = m
            print(f"[eval] seed {seed} {split}: binary@0.5 macro_f1={m['binary@0.5']['macro_f1']} "
                  f"roc_auc={m['binary@0.5']['roc_auc']}"
                  + (f" | 3-class macro_f1={m['native_3class']['macro_f1']}" if "native_3class" in m else ""))
            if split == "test":
                b = binary_view(pred) if ds == "semdetect" else pred
                paths, breakdown = save_error_analysis(b, out_root / f"seed{seed}", "test_binary", text_lookup(man))
                seed_out["error_analysis"] = {"tables": paths, "breakdown": breakdown}
                if ds == "semdetect":
                    p3, bd3 = save_error_analysis(pred, out_root / f"seed{seed}", "test_3class", text_lookup(man))
                    seed_out["error_analysis_3class"] = {"tables": p3, "breakdown": bd3}
            for kind, key in (("0.5", "binary@0.5"), ("val_calibrated", "binary@val_threshold")):
                rows.append(comparison_row(f"PeerBERT[{args.tag}] seed{seed}", "trained classifier", split, kind,
                                           m[key], seed=seed))
        per_seed[seed] = seed_out
        save_json(seed_out, out_root / f"seed{seed}" / "metrics.json")

    if not per_seed:
        print("[eval] nothing evaluated")
        return 1
    summary = {"dataset": ds, "tag": args.tag, "seeds": list(per_seed), "per_split": {}}
    for split in manifests:
        block = {}
        for key in ("binary@0.5", "binary@val_threshold"):
            block[key] = seed_summary([per_seed[s]["splits"][split][key] for s in per_seed])
        if ds == "semdetect":
            block["native_3class"] = seed_summary([per_seed[s]["splits"][split]["native_3class"] for s in per_seed],
                                                  keys=("accuracy", "macro_f1", "weighted_f1", "macro_ovr_roc_auc"))
        summary["per_split"][split] = block
    summary["note"] = "mean +/- std over seeds (ddof=1); claim improvements only with all seeds evaluated"
    save_json(summary, out_root / "metrics_summary.json")
    table = merge_comparison_rows(res_root / "comparison_table.json", rows)
    save_comparison_table(table, res_root)
    print(f"[eval] summary: {out_root / 'metrics_summary.json'}\n[eval] table: {res_root / 'comparison_table.csv'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/06_run_baselines.py
"""
06_run_baselines.py
Run the 8 baselines on the test_baseline subset (nested inside the model test set).

- Validation sample (whole papers, common.val_rows) -> threshold calibration ONLY.
- Train sample (whole papers, common.train_rows) -> fits GLTR / MFD heads ONLY.
- Zero-shot / pretrained detectors are never trained; nothing is tuned on test.
- Sem-Detect: human/ai rows only; rewrite rows -> descriptive "fraction flagged as AI".
- One shared token-statistics LM pass feeds LogRank, GLTR, DetectLLM-LRR, MFD and
  Fast-DetectGPT (analytic, same reference/scoring model).
Outputs: outputs/results/<ds>/baselines/<name>/{scores_*.parquet, metrics.json,
predictions_test_baseline.csv} and rows merged into outputs/results/<ds>/comparison_table.*

  python scripts/06_run_baselines.py --dataset intellabs
  python scripts/06_run_baselines.py --dataset semdetect --only logrank radar
Restart-safe: finished detectors are skipped (use --force to recompute).
"""

import argparse
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

TOKEN_STATS_DETECTORS = ("logrank", "gltr", "detectllm_lrr", "mfd", "fast_detectgpt")
PROBABILITY_OUTPUT = ("gltr", "mfd", "radar")


def parse():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", choices=("intellabs", "semdetect"), required=True)
    ap.add_argument("--only", nargs="+", default=None, help="subset of baselines.yaml run list")
    ap.add_argument("--force", action="store_true")
    ap.add_argument("--gpu", type=int, default=None)
    ap.add_argument("--config", default=None)
    ap.add_argument("--baselines-config", default=None)
    return ap.parse_args()


def main():
    args = parse()
    import yaml
    bc_path = Path(args.baselines_config) if args.baselines_config else ROOT / "configs" / "baselines.yaml"
    bcfg = yaml.safe_load(bc_path.read_text())
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu if args.gpu is not None else bcfg["common"].get("gpu_id", 0))

    import numpy as np
    import pandas as pd

    from src.baselines import registry
    from src.baselines.statistical import detectllm_lrr, gltr, logrank, mfd
    from src.baselines.statistical.token_stats import compute_token_stats, load_causal_lm, release
    from src.baselines.zero_shot import fast_detectgpt
    from src.data.common import (
        cache_dir, dataset_dir_name, load_config, load_json, manifest_dir, resolve_path, save_json, set_seed,
    )
    from src.data.splits import cap_by_papers
    from src.evaluation.aggregate import comparison_row, merge_comparison_rows, save_comparison_table
    from src.evaluation.metrics import binary_metrics, calibrate_threshold
    from src.evaluation.predict import prediction_frame, save_predictions

    cfg = load_config(args.config)
    ds = args.dataset
    seed = int(cfg["project"]["seed"])
    set_seed(seed)
    com = bcfg["common"]
    names = args.only or bcfg["run"]
    for n in names:
        registry.get(n)

    mdir = manifest_dir(cfg, ds)
    train = pd.read_parquet(mdir / "train.parquet")
    val = pd.read_parquet(mdir / "val.parquet")
    test_b = pd.read_parquet(mdir / "test_baseline.parquet")
    binary = lambda d: d[d["label"].isin([0, 1])].reset_index(drop=True)  # noqa: E731
    sets = {
        "train": cap_by_papers(binary(train), com["train_rows"], seed).reset_index(drop=True),
        "val": cap_by_papers(binary(val), com["val_rows"], seed).reset_index(drop=True),
        "test_baseline": binary(test_b),
    }
    rewrite = test_b[test_b["label"] == 2].reset_index(drop=True)
    if len(rewrite):
        sets["rewrite"] = rewrite
    print(f"[baselines] {ds}: " + ", ".join(f"{k}={len(v)}" for k, v in sets.items()))

    res_root = resolve_path(cfg, "results_dir", "outputs/results") / dataset_dir_name(cfg, ds)
    out_root = res_root / "baselines"
    bcache = cache_dir(cfg, ds) / "baselines"

    def done(name):
        return (out_root / name / "metrics.json").exists() and not args.force

    # ------------------------------------------------ shared token statistics
    stats = {}
    need_stats = [n for n in names if n in TOKEN_STATS_DETECTORS and not done(n)]
    if need_stats:
        ts = bcfg["token_stats"]
        loaded = load_causal_lm(ts["model"], com["dtype"])
        for split, frame in sets.items():
            if split == "train" and not ({"gltr", "mfd"} & set(need_stats)):
                continue
            stats[split] = compute_token_stats(frame, ts["model"], bcache, com["max_tokens"], com["batch_size"],
                                               com["dtype"], tuple(bcfg["gltr"]["rank_buckets"]), loaded=loaded)
        del loaded
        release()

    # ------------------------------------------------ per-detector scores
    def scores_for(name):
        """-> dict split -> score array, extra info."""
        info = {}
        eval_splits = [s for s in sets if s != "train"]
        if name == "logrank":
            return {s: logrank.score(stats[s]) for s in eval_splits}, info
        if name == "detectllm_lrr":
            return {s: detectllm_lrr.score(stats[s]) for s in eval_splits}, {"epsilon": detectllm_lrr.EPS}
        if name == "fast_detectgpt":
            fc = bcfg["fast_detectgpt"]
            if fc["reference_model"] != "same" or fc["scoring_model"] != "same":
                raise NotImplementedError("use score_two_models() for a separate reference model")
            return ({s: fast_detectgpt.score_from_stats(stats[s]) for s in eval_splits},
                    {"reference_model": bcfg["token_stats"]["model"], "scoring_model": bcfg["token_stats"]["model"]})
        if name in ("gltr", "mfd"):
            y = sets["train"]["label"].values
            if name == "gltr":
                clf = gltr.fit(stats["train"], y, seed)
                return {s: gltr.score(clf, stats[s]) for s in eval_splits}, {"train_rows": len(y)}
            mc = bcfg["mfd"]
            clf = mfd.fit(stats["train"], y, mc["head"], mc.get("mlp_hidden", (16, 16, 16)), seed)
            return ({s: mfd.score(clf, stats[s]) for s in eval_splits},
                    {"train_rows": len(y), "head": mc["head"], "label": "MFD" if mc["head"] == "logreg" else "MFD-M"})
        if name == "binoculars":
            from src.baselines.zero_shot import binoculars
            bc = bcfg["binoculars"]
            o, p = ((bc["paper_observer"], bc["paper_performer"]) if bc["use_paper_pair"]
                    else (bc["observer"], bc["performer"]))
            obs, perf, tok, dev = binoculars.load_pair(o, p, com["dtype"])
            out = {s: binoculars.score(sets[s]["text"].tolist(), obs, perf, tok, dev, com["max_tokens"],
                                       bc.get("batch_size", 2)) for s in eval_splits}
            del obs, perf
            release()
            return out, {"observer": o, "performer": p, "paper_pair": bool(bc["use_paper_pair"])}
        if name == "radar":
            from src.baselines.neural_detectors import radar
            rc = bcfg["radar"]
            model, tok, dev = radar.load(rc["model"])
            probs = {s: radar.probabilities(model, tok, sets[s]["text"].tolist(), dev, rc["max_length"],
                                            rc.get("batch_size", 16)) for s in eval_splits}
            del model
            release()
            idx, decision = radar.orient(probs["val"], sets["val"]["label"].values)
            return {s: p[:, idx] for s, p in probs.items()}, {"orientation": decision, "model": rc["model"]}
        if name == "dnagpt":
            from src.baselines.zero_shot import dnagpt
            dc = bcfg["dnagpt"]
            model, tok, dev = load_causal_lm(dc["regenerator"], com["dtype"])
            out, w = {}, {}
            for s in eval_splits:
                out[s], w[s] = dnagpt.run(sets[s], model, tok, dev, dc, bcache / f"dnagpt_{ds}_{s}.jsonl", seed)
            del model
            release()
            info = {"regenerator": dc["regenerator"], "k": dc["k"], "gamma": dc["gamma"]}
            if dc.get("wscore"):
                info["wscore_auroc_val"] = binary_metrics(sets["val"]["label"], np.nan_to_num(w["val"]))["roc_auc"]
            return out, info
        raise KeyError(name)

    rows = []
    for name in names:
        meta = registry.get(name)
        odir = out_root / name
        if done(name):
            print(f"[baselines] {name}: done (metrics.json exists); skipping")
            m = load_json(odir / "metrics.json")
        else:
            print(f"[baselines] {name} ...")
            scores, info = scores_for(name)
            odir.mkdir(parents=True, exist_ok=True)
            for s, sc in scores.items():
                if not np.isfinite(sc).all():
                    raise RuntimeError(f"{name}: non-finite scores on {s}")
                pd.DataFrame({"row_id": sets[s]["row_id"], "score": sc}).to_parquet(
                    odir / f"scores_{s}.parquet", index=False)
            thr = calibrate_threshold(sets["val"]["label"], scores["val"], com["threshold_objective"])
            m = {"baseline": name, "display": meta["display"], "type": meta["type"], "registry": meta,
                 "info": info, "val_threshold": thr, "n_val": len(sets["val"]),
                 "val@val_threshold": binary_metrics(sets["val"]["label"], scores["val"], thr),
                 "test_baseline@val_threshold": binary_metrics(sets["test_baseline"]["label"],
                                                               scores["test_baseline"], thr)}
            if name in PROBABILITY_OUTPUT:
                m["test_baseline@0.5"] = binary_metrics(sets["test_baseline"]["label"], scores["test_baseline"], 0.5)
            if "rewrite" in scores:
                m["rewrite_fraction_flagged_ai@val_threshold"] = {
                    "n_rewrite": len(scores["rewrite"]),
                    "fraction": float((scores["rewrite"] >= thr).mean()),
                    "note": "descriptive only; rewrite is not part of any classification metric"}
            pred = prediction_frame(sets["test_baseline"], score=scores["test_baseline"], threshold=thr)
            save_predictions(pred, odir / "predictions_test_baseline.csv")
            save_json(m, odir / "metrics.json")
            mt = m["test_baseline@val_threshold"]
            print(f"[baselines] {name}: test_baseline roc_auc={mt['roc_auc']} macro_f1@val_thr={mt['macro_f1']}")
        rows.append(comparison_row(m["display"], m["type"], "test_baseline", "val_calibrated",
                                   m["test_baseline@val_threshold"]))
        if "test_baseline@0.5" in m:
            rows.append(comparison_row(m["display"], m["type"], "test_baseline", "0.5", m["test_baseline@0.5"]))

    table = merge_comparison_rows(res_root / "comparison_table.json", rows)
    save_comparison_table(table, res_root)
    save_json({"dataset": ds, "baselines": names, "sample_sizes": {k: len(v) for k, v in sets.items()},
               "config": bcfg}, out_root / "run_summary.json")
    print(f"[baselines] table: {res_root / 'comparison_table.csv'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/07_run_ablation.py
"""
07_run_ablation.py
Loss / regularisation ablations for PeerBERT, selected on VALIDATION Macro-F1 only.

Variants (configs/train.yaml -> ablation.variants):
  unweighted CE | class-weighted CE | label smoothing 0.05 | focal (gamma=2, weighted)
  R-Drop alpha in {0, 0.5, 1.0} (alpha 0 = ce_weighted) | dropout off
Each variant is trained with ablation.seed into outputs/models/<ds>/ablation/<variant>/.
Test-set numbers of the selected variant come from 05_evaluate.py --tag ablation/<variant>;
they are never used for selection. Repeat the chosen configuration with 3 seeds
(03/04) before claiming an improvement.

  python scripts/07_run_ablation.py --dataset intellabs
  python scripts/07_run_ablation.py --dataset semdetect --variants ce_weighted rdrop_1.0 --max-train-rows 8000
"""

import argparse
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))


def parse():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", choices=("intellabs", "semdetect"), required=True)
    ap.add_argument("--variants", nargs="+", default=None)
    ap.add_argument("--max-train-rows", type=int, default=None, help="whole-paper compute cap (disclose it)")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--model", default=None)
    ap.add_argument("--config", default=None)
    ap.add_argument("--train-config", default=None)
    return ap.parse_args()


def main():
    args = parse()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)
    import pandas as pd

    from src.data.common import dataset_dir_name, load_config, resolve_path, save_json
    from src.models.trainer import deep_merge, load_train_config, train_from_manifests

    cfg = load_config(args.config)
    base = load_train_config(args.train_config)
    if args.epochs:
        base["training"]["epochs"] = args.epochs
    if args.model:
        base["model"]["name"] = args.model
    abl = base["ablation"]
    variants = [v for v in abl["variants"] if not args.variants or v["name"] in args.variants]
    unknown = set(args.variants or []) - {v["name"] for v in abl["variants"]}
    if unknown:
        raise SystemExit(f"unknown variants: {sorted(unknown)}")

    rows = []
    for v in variants:
        tcfg = deep_merge(base, {k: v[k] for k in ("loss", "model", "training") if k in v})
        if tcfg["loss"].get("rdrop_alpha", 0) > 0 and not tcfg["model"].get("dropout_on", True):
            raise SystemExit(f"{v['name']}: R-Drop requires dropout")
        s = train_from_manifests(args.dataset, cfg, tcfg, [int(abl["seed"])], f"ablation/{v['name']}",
                                 args.max_train_rows)
        r = s["runs"][0]
        rows.append({"variant": v["name"], "loss": tcfg["loss"]["type"], "rdrop_alpha": tcfg["loss"]["rdrop_alpha"],
                     "dropout_on": tcfg["model"].get("dropout_on", True), "seed": r["seed"],
                     "val_macro_f1": r["best_val_macro_f1"], "epochs_run": r["epochs_run"], "minutes": r["minutes"]})

    df = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)
    out = resolve_path(cfg, "results_dir", "outputs/results") / dataset_dir_name(cfg, args.dataset)
    out.mkdir(parents=True, exist_ok=True)
    df.to_csv(out / "ablation.csv", index=False)
    save_json({"dataset": args.dataset, "selection": "validation Macro-F1 (single seed)",
               "selected_variant": df.iloc[0]["variant"] if len(df) else None,
               "train_row_cap": args.max_train_rows, "rows": df.to_dict(orient="records"),
               "note": "single-seed ablation; confirm the selected configuration with 3 seeds"},
              out / "ablation.json")
    print(df.to_string(index=False))
    print(f"[ablation] {out / 'ablation.json'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/08_cross_dataset.py
"""
08_cross_dataset.py
Cross-dataset transfer, Human vs AI only, NO retraining.

  IntelLabs model (binary)  -> Sem-Detect test human/ai rows (rewrite excluded)
  Sem-Detect model (3-class)-> IntelLabs test rows, scored with the binary view
                               P(ai) / (P(human) + P(ai))
Thresholds: 0.5 and the SOURCE validation-calibrated threshold (never the target's).

Known mismatches (documented in the output):
- generators: IntelLabs AI reviews come from 5 named LLMs; Sem-Detect "ai" reviews come
  from its own generation pipeline (models not taken from its metadata here);
- label definitions: Sem-Detect has a separate "rewrite" (LLM-refined human) class that
  has no IntelLabs counterpart and is excluded;
- venues/years and review templates differ between the datasets.

  python scripts/08_cross_dataset.py --source intellabs
  python scripts/08_cross_dataset.py --source semdetect --seeds 42
"""

import argparse
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

MISMATCH = [
    "Generators differ: IntelLabs AI = claude/gemini/gpt-4o/llama/qwen; Sem-Detect AI from its own pipeline.",
    "Sem-Detect 'rewrite' (LLM-refined human) has no IntelLabs counterpart and is excluded.",
    "Conference-years and review templates differ between the datasets.",
    "Thresholds come from the SOURCE validation set; nothing is tuned on the target.",
]


def parse():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--source", choices=("intellabs", "semdetect"), required=True)
    ap.add_argument("--tag", default="final")
    ap.add_argument("--seeds", type=int, nargs="+", default=None)
    ap.add_argument("--target-split", default="test")
    ap.add_argument("--max-target-rows", type=int, default=None, help="whole-paper cap on the target rows")
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--config", default=None)
    ap.add_argument("--train-config", default=None)
    return ap.parse_args()


def main():
    args = parse()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)
    import pandas as pd

    from src.data.common import dataset_dir_name, load_config, load_json, manifest_dir, resolve_path, save_json
    from src.data.splits import cap_by_papers
    from src.evaluation.aggregate import seed_summary
    from src.evaluation.metrics import binary_metrics, calibrate_threshold
    from src.evaluation.predict import binary_view, prediction_frame, save_predictions
    from src.models.trainer import CLASS_NAMES, load_train_config, model_root, predict_proba

    cfg = load_config(args.config)
    tcfg = load_train_config(args.train_config)
    src = args.source
    tgt = "semdetect" if src == "intellabs" else "intellabs"
    seeds = args.seeds or tcfg["training"]["seeds"]
    target = pd.read_parquet(manifest_dir(cfg, tgt) / f"{args.target_split}.parquet")
    target = target[target["label"].isin([0, 1])].reset_index(drop=True)
    if args.max_target_rows:
        target = cap_by_papers(target, args.max_target_rows, int(cfg["project"]["seed"])).reset_index(drop=True)
    out = resolve_path(cfg, "results_dir", "outputs/results") / dataset_dir_name(cfg, tgt) / f"cross_from_{src}"

    per_seed = {}
    for seed in seeds:
        run_dir = model_root(cfg, src) / args.tag / f"seed{seed}"
        if not (run_dir / "done.json").exists():
            print(f"[cross] no finished source model at {run_dir}; skipping")
            continue
        vp = pd.read_csv(run_dir / "val_predictions.csv")
        if src == "semdetect":
            vb = binary_view(vp)
            src_thr = calibrate_threshold(vb["label"], vb["score"], tcfg["threshold"]["objective"])
        else:
            src_thr = load_json(run_dir / "run_info.json")["val_threshold"]
        pp = out / f"seed{seed}_predictions.csv"
        if pp.exists():
            pred = pd.read_csv(pp, dtype={"row_id": str})
        else:
            proba, _ = predict_proba(run_dir / "best", target["text"].tolist(), tcfg)
            pred = prediction_frame(target, proba=proba, class_names=CLASS_NAMES[src])
            if src == "semdetect":
                pred = binary_view(pred)
            save_predictions(pred, pp)
        per_seed[seed] = {"source_val_threshold": src_thr,
                          "binary@0.5": binary_metrics(pred["label"], pred["score"], 0.5),
                          "binary@source_val_threshold": binary_metrics(pred["label"], pred["score"], src_thr)}
        print(f"[cross] {src}->{tgt} seed {seed}: roc_auc={per_seed[seed]['binary@0.5']['roc_auc']} "
              f"macro_f1@src_thr={per_seed[seed]['binary@source_val_threshold']['macro_f1']}")
    if not per_seed:
        return 1
    summary = {"source": src, "target": tgt, "target_split": args.target_split, "n_target": len(target),
               "target_label_counts": target["label"].value_counts().to_dict(), "seeds": list(per_seed),
               "per_seed": per_seed,
               "mean_std@0.5": seed_summary([v["binary@0.5"] for v in per_seed.values()]),
               "mean_std@source_val_threshold": seed_summary([v["binary@source_val_threshold"]
                                                              for v in per_seed.values()]),
               "mismatch_notes": MISMATCH}
    save_json(summary, out / "metrics.json")
    print(f"[cross] {out / 'metrics.json'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/09_build_report.py
"""
09_build_report.py
Collect every result that EXISTS on disk into thesis tables. Nothing is estimated: a
missing artefact is reported as "NOT RUN".

Outputs: outputs/results/report/
  report.md                       what was run, audit status, main tables, notes
  <ds>_comparison.csv / .tex      all methods on test_baseline, grouped by type
  <ds>_paired_tests.csv           PeerBERT vs each baseline on the SAME test_baseline rows:
                                  paired cluster bootstrap (ROC-AUC) + McNemar (val-calibrated
                                  decisions), Holm-adjusted across baselines
  <ds>_roc.png                    ROC curves on test_baseline (if matplotlib is available)

PeerBERT reference seed for paired tests = the seed with the median VALIDATION Macro-F1
(selection on validation only).

  python scripts/09_build_report.py
  python scripts/09_build_report.py --n-boot 2000
"""

import argparse
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from src.data.common import (  # noqa: E402
    audit_dir, dataset_dir_name, load_config, load_json, manifest_dir, resolve_path, save_json,
)
from src.evaluation.metrics import calibrate_threshold  # noqa: E402
from src.evaluation.predict import binary_view, load_predictions  # noqa: E402
from src.evaluation.stats import auc_fn, holm, mcnemar_exact, paired_cluster_bootstrap  # noqa: E402

DATASETS = ("intellabs", "semdetect")
TABLE_COLS = ["type", "method", "threshold_kind", "n", "accuracy", "precision", "recall", "f1", "macro_f1",
              "roc_auc", "pr_auc", "tpr@1%fpr", "tpr@5%fpr", "tpr@10%fpr"]


def latex_table(df, caption, label):
    cols = [c for c in TABLE_COLS if c in df.columns]
    lines = ["\\begin{table}[t]", "\\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
             "\\begin{tabular}{" + "l" * 3 + "r" * (len(cols) - 3) + "}", "\\toprule",
             " & ".join(c.replace("_", "\\_").replace("%", "\\%").replace("@", "@") for c in cols) + " \\\\",
             "\\midrule"]
    prev = None
    for _, r in df[cols].iterrows():
        if prev is not None and r["type"] != prev:
            lines.append("\\midrule")
        prev = r["type"]
        cells = []
        for c in cols:
            v = r[c]
            if isinstance(v, float):
                cells.append("--" if np.isnan(v) else f"{v:.3f}")
            else:
                cells.append("--" if v is None else str(v).replace("_", "\\_").replace("%", "\\%"))
        lines.append(" & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines)


def reference_seed(model_dir):
    """Seed with the median validation Macro-F1 among finished seeds."""
    runs = []
    for d in sorted(model_dir.glob("seed*")):
        if (d / "done.json").exists():
            runs.append((load_json(d / "done.json")["best_val_macro_f1"], int(d.name[4:])))
    if not runs:
        return None
    runs.sort()
    return runs[(len(runs) - 1) // 2][1]


def peerbert_test_baseline(cfg, ds, res_root, tag):
    model_dir = resolve_path(cfg, "model_dir", "outputs/models") / dataset_dir_name(cfg, ds) / tag
    seed = reference_seed(model_dir)
    if seed is None:
        return None, None, None
    p = res_root / "peerbert" / tag / f"seed{seed}" / "test_baseline_predictions.csv"
    if not p.exists():
        return seed, None, None
    pred = load_predictions(p)
    vp = pd.read_csv(model_dir / f"seed{seed}" / "val_predictions.csv")
    if ds == "semdetect":
        pred, vp = binary_view(pred), binary_view(vp)
    thr = calibrate_threshold(vp["label"], vp["score"])
    return seed, pred, thr


def paired_tests(cfg, ds, res_root, tag, n_boot):
    seed, pb, pb_thr = peerbert_test_baseline(cfg, ds, res_root, tag)
    if pb is None:
        return None, f"PeerBERT test_baseline predictions NOT RUN (reference seed: {seed})"
    rows, pvals = [], {}
    for mdir in sorted((res_root / "baselines").glob("*/")):
        pp = mdir / "predictions_test_baseline.csv"
        if not pp.exists():
            continue
        m = load_json(mdir / "metrics.json")
        bp = load_predictions(pp)
        j = pb.merge(bp[["row_id", "score", "pred"]], on="row_id", suffixes=("_pb", "_bl"))
        if len(j) != len(bp) or len(j) != len(pb):
            rows.append({"baseline": m["display"], "note": f"row mismatch: peerbert {len(pb)}, baseline {len(bp)}, "
                                                           f"shared {len(j)}"})
            continue
        boot = paired_cluster_bootstrap(j["label"], j["score_pb"], j["score_bl"], j["paper_key"], auc_fn, n_boot)
        mc = mcnemar_exact(j["label"], (j["score_pb"] >= pb_thr).astype(int), j["pred_bl"])
        pvals[m["display"]] = boot["p_value"]
        rows.append({"baseline": m["display"], "type": m["type"], "n_shared": len(j), "n_papers": boot.get("n_clusters"),
                     "auc_diff_peerbert_minus_baseline": boot["diff"], "ci_low": boot["ci_low"],
                     "ci_high": boot["ci_high"], "p_bootstrap": boot["p_value"], "p_mcnemar": mc["p_value"],
                     "peerbert_right_baseline_wrong": mc["a_right_b_wrong"],
                     "peerbert_wrong_baseline_right": mc["a_wrong_b_right"]})
    adj = holm(pvals)
    for r in rows:
        if r.get("baseline") in adj:
            r["p_bootstrap_holm"] = adj[r["baseline"]]
    return pd.DataFrame(rows), f"PeerBERT reference seed {seed} (median validation Macro-F1); n_boot={n_boot}"


def roc_figure(ds, res_root, tag, out_png):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from sklearn.metrics import roc_curve
    except ImportError:
        return False
    curves = []
    for mdir in sorted((res_root / "baselines").glob("*/")):
        pp = mdir / "predictions_test_baseline.csv"
        if pp.exists():
            curves.append((load_json(mdir / "metrics.json")["display"], load_predictions(pp)))
    pb = sorted((res_root / "peerbert" / tag).glob("seed*/test_baseline_predictions.csv"))
    if pb:
        p = load_predictions(pb[0])
        curves.insert(0, (f"PeerBERT ({pb[0].parent.name})", binary_view(p) if ds == "semdetect" else p))
    if not curves:
        return False
    fig, ax = plt.subplots(figsize=(5, 5))
    for name, p in curves:
        if p["label"].nunique() == 2:
            fpr, tpr, _ = roc_curve(p["label"], p["score"])
            ax.plot(fpr, tpr, label=name, lw=1.4)
    ax.plot([0, 1], [0, 1], ls=":", c="grey")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"{ds}: test_baseline")
    ax.legend(fontsize=7, loc="lower right")
    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    return True


def status_of(path):
    return "present" if Path(path).exists() else "NOT RUN"


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--tag", default="final")
    ap.add_argument("--n-boot", type=int, default=1000)
    ap.add_argument("--config", default=None)
    args = ap.parse_args()
    cfg = load_config(args.config)
    out = resolve_path(cfg, "results_dir", "outputs/results") / "report"
    out.mkdir(parents=True, exist_ok=True)
    md = ["# Results report", "", "Generated by scripts/09_build_report.py from artefacts on disk. "
          "Anything marked NOT RUN was not executed.", ""]
    index = {}
    for ds in DATASETS:
        dd = dataset_dir_name(cfg, ds)
        res_root = resolve_path(cfg, "results_dir", "outputs/results") / dd
        md += [f"## {ds}", ""]
        a = audit_dir(cfg, ds) / "audit_status.json"
        md.append(f"- Leakage audit: {load_json(a)['status'] if a.exists() else 'NOT RUN'}")
        s = manifest_dir(cfg, ds) / "summary.json"
        if s.exists():
            for name, d in load_json(s).get("manifests", {}).items():
                md.append(f"- Manifest `{name}`: {d['rows']} rows, {d['papers']} papers, labels {d['label_counts']}")
        else:
            md.append("- Manifests: NOT RUN")
        arts = {
            "PeerBERT metrics": res_root / "peerbert" / args.tag / "metrics_summary.json",
            "Baselines": res_root / "baselines" / "run_summary.json",
            "Ablation": res_root / "ablation.json",
            "Cross-dataset (into this dataset)": next(iter(sorted(res_root.glob("cross_from_*/metrics.json"))),
                                                      res_root / "cross_from_x" / "metrics.json"),
            "Explainability (XAI)": next(iter(sorted((res_root / "xai" / args.tag).glob("seed*/summary.json"))),
                                         res_root / "xai" / "missing" / "summary.json"),
        }
        md += [f"- {k}: {status_of(v)}" for k, v in arts.items()] + [""]

        ms = arts["PeerBERT metrics"]
        if ms.exists():
            summ = load_json(ms)
            md += ["### PeerBERT (mean +/- std over seeds)", "", "| split | metric | mean | std | n_seeds |",
                   "|---|---|---|---|---|"]
            for split, block in summ["per_split"].items():
                for kind, vals in block.items():
                    for metric in ("macro_f1", "roc_auc", "accuracy"):
                        if metric in vals:
                            md.append(f"| {split} | {kind} {metric} | {vals[metric]['mean']:.4f} | "
                                      f"{vals[metric]['std']:.4f} | {vals['n_seeds']} |")
            md.append("")

        ct = res_root / "comparison_table.json"
        if ct.exists():
            df = pd.DataFrame(load_json(ct))
            df = df[df["split"] == "test_baseline"]
            df.to_csv(out / f"{ds}_comparison.csv", index=False)
            (out / f"{ds}_comparison.tex").write_text(latex_table(
                df, f"{ds}: all methods on the shared test\\_baseline subset", f"tab:{ds}_comparison"))
            md += ["### Comparison on test_baseline", "", f"See `{ds}_comparison.csv` / `.tex` "
                   "(zero-shot detectors, pretrained detectors and trained classifiers are separate groups).", ""]
            pt, note = paired_tests(cfg, ds, res_root, args.tag, args.n_boot)
            if pt is not None:
                pt.to_csv(out / f"{ds}_paired_tests.csv", index=False)
                md += ["### Paired tests", "", note, f"See `{ds}_paired_tests.csv`. Assumptions: papers are "
                       "independent clusters; McNemar ignores clustering; Holm across baselines.", ""]
            else:
                md += [f"Paired tests: {note}", ""]
            if roc_figure(ds, res_root, args.tag, out / f"{ds}_roc.png"):
                md.append(f"![ROC]({ds}_roc.png)\n")
        else:
            md += ["Comparison table: NOT RUN", ""]
        if arts["Ablation"].exists():
            ab = load_json(arts["Ablation"])
            md += ["### Ablation (validation Macro-F1, single seed)", "", "| variant | val Macro-F1 |", "|---|---|"]
            md += [f"| {r['variant']} | {r['val_macro_f1']:.4f} |" for r in ab["rows"]] + [""]
        cr = arts["Cross-dataset (into this dataset)"]
        if cr.exists():
            c = load_json(cr)
            m = c["mean_std@source_val_threshold"]
            md += [f"### Cross-dataset {c['source']} -> {c['target']} (no retraining)", "",
                   f"Macro-F1 @ source-val threshold: {m.get('macro_f1', {}).get('mean')} "
                   f"+/- {m.get('macro_f1', {}).get('std')}; ROC-AUC {m.get('roc_auc', {}).get('mean')}", ""]
            md += [f"- {n}" for n in c["mismatch_notes"]] + [""]
        xs = arts["Explainability (XAI)"]
        if xs.exists():
            x = load_json(xs)
            md += [f"### Explainability (seed {x['seed']}, {x['n_examples']} {x['split']} reviews)", "",
                   "| method | k | comprehensiveness (higher better) | sufficiency (lower better) |",
                   "|---|---|---|---|"]
            for meth, byk in x["faithfulness"].items():
                for k, v in byk.items():
                    c, su = v["comprehensiveness"], v["sufficiency"]
                    md.append(f"| {meth} | {k} | {c['mean']:.4f} | {su['mean']:.4f} |" if c and su
                              else f"| {meth} | {k} | -- | -- |")
            sp = x["spearman_abs_ig_vs_attention"]
            md += ["", "Spearman(|IG|, attention): " + ", ".join(
                f"{k} {v['mean']:.3f}" for k, v in sp.items() if v), f"Top tokens: `{xs.parent / 'top_tokens.csv'}`", ""]
        index[ds] = {k: str(v) for k, v in arts.items()}
    (out / "report.md").write_text("\n".join(md), encoding="utf-8")
    save_json(index, out / "artefacts.json")
    print(f"[report] {out / 'report.md'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/10_explain.py
"""
10_explain.py
Stage 9 - explainability for a trained PeerBERT on REAL manifest rows (no SHAP/LIME):
  * attention from an eager-attention copy (last layer, mean over layers, rollout)
  * Integrated Gradients (Captum) on the token embeddings, signed, with convergence delta
  * faithfulness: comprehensiveness / sufficiency at k = 10%, 20%, vs a random ranking
  * attention-vs-IG rank agreement (Spearman)
  * global view: tokens with the largest mean IG attribution per explained class

Examples come from 05_evaluate.py predictions of one seed on one split (default test_baseline):
for every (true class, correct/incorrect) group, up to --n-per-group rows are drawn with a fixed
seed. The explained class is the model's PREDICTED class. Reviews are tokenised exactly as in
training (configs/train.yaml max_length / truncation).
Default seed = the seed with the median VALIDATION Macro-F1.

Outputs: outputs/results/<ds>/xai/<tag>/seed<k>/
  explanations.jsonl (restart-safe, one row per example), summary.json, top_tokens.csv,
  examples.html

  python scripts/10_explain.py --dataset intellabs
  python scripts/10_explain.py --dataset semdetect --n-per-group 10 --n-steps 32
"""

import argparse
import json
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))


def parse():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--dataset", choices=("intellabs", "semdetect"), required=True)
    ap.add_argument("--tag", default="final")
    ap.add_argument("--seed", type=int, default=None, help="model seed (default: median validation Macro-F1)")
    ap.add_argument("--split", default="test_baseline")
    ap.add_argument("--n-per-group", type=int, default=25)
    ap.add_argument("--n-steps", type=int, default=50, help="Integrated Gradients steps")
    ap.add_argument("--internal-batch-size", type=int, default=8)
    ap.add_argument("--ks", type=float, nargs="+", default=[0.1, 0.2])
    ap.add_argument("--min-token-count", type=int, default=5, help="for the global top-token table")
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--config", default=None)
    ap.add_argument("--train-config", default=None)
    return ap.parse_args()


def reference_seed(model_dir):
    from src.data.common import load_json
    runs = sorted((load_json(d / "done.json")["best_val_macro_f1"], int(d.name[4:]))
                  for d in model_dir.glob("seed*") if (d / "done.json").exists())
    return runs[(len(runs) - 1) // 2][1] if runs else None


def select_examples(pred, n_per_group, seed):
    """Up to n rows per (true class, correct/incorrect) group, fixed random seed."""
    import pandas as pd
    pred = pred.assign(correct=(pred["pred"] == pred["label"]))
    parts = [g.sample(min(n_per_group, len(g)), random_state=seed)
             for _, g in pred.groupby(["label", "correct"], sort=True)]
    return pd.concat(parts).reset_index(drop=True) if parts else pred.iloc[:0]


def main():
    args = parse()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)
    import numpy as np
    import pandas as pd
    import torch

    from src.data.common import dataset_dir_name, load_config, manifest_dir, resolve_path, save_json
    from src.evaluation.predict import load_predictions
    from src.models.dataset import ReviewDataset
    from src.models.modernbert_classifier import load_trained
    from src.models.trainer import CLASS_NAMES, load_train_config, model_root
    from src.xai.attention import attention_scores, spearman
    from src.xai.faithfulness import faithfulness
    from src.xai.integrated_gradients import display_tokens, ig_attributions, normalise, special_token_mask
    from src.xai.visualize import page

    cfg = load_config(args.config)
    tcfg = load_train_config(args.train_config)
    ds = args.dataset
    names = CLASS_NAMES[ds]
    mroot = model_root(cfg, ds) / args.tag
    seed = args.seed if args.seed is not None else reference_seed(mroot)
    if seed is None:
        raise SystemExit(f"no finished model under {mroot}; run 03/04 first")
    res_root = resolve_path(cfg, "results_dir", "outputs/results") / dataset_dir_name(cfg, ds)
    pred_p = res_root / "peerbert" / args.tag / f"seed{seed}" / f"{args.split}_predictions.csv"
    if not pred_p.exists():
        raise SystemExit(f"{pred_p} not found; run 05_evaluate.py first")
    manifest = pd.read_parquet(manifest_dir(cfg, ds) / f"{args.split}.parquet")
    pred = load_predictions(pred_p)
    chosen = select_examples(pred, args.n_per_group, int(cfg["project"]["seed"]))
    chosen = chosen.merge(manifest[["row_id", "text"]], on="row_id", how="left")
    if chosen["text"].isna().any():
        raise SystemExit("prediction rows not found in the manifest (manifests changed after evaluation?)")

    out = res_root / "xai" / args.tag / f"seed{seed}"
    out.mkdir(parents=True, exist_ok=True)
    jl = out / "explanations.jsonl"
    done = set()
    if jl.exists():
        done = {json.loads(line)["row_id"] for line in jl.read_text(encoding="utf-8").splitlines()}

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model, tok = load_trained(mroot / f"seed{seed}" / "best", eager_attention=True)
    model.to(device).float().eval()
    m = tcfg["model"]
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
    todo = chosen[~chosen["row_id"].isin(done)]
    ds_tok = ReviewDataset(todo["text"].tolist(), None, tok, m["max_length"], m.get("truncation", "head"),
                           m.get("head_tokens", 256))
    print(f"[xai] {ds} seed {seed}: {len(chosen)} examples ({len(done)} already done), device={device}")

    with jl.open("a", encoding="utf-8") as f:
        for i, r in enumerate(todo.itertuples(index=False)):
            ids = ds_tok.input_ids[i]
            special = special_token_mask(tok, ids)
            target = int(r.pred)
            attn = attention_scores(model, ids, device)
            ig, delta, p, p0 = ig_attributions(model, ids, special, target, device, pad_id, args.n_steps,
                                               args.internal_batch_size)
            content = [j for j, s in enumerate(special) if not s]
            faith = faithfulness(model, ids, special, {"integrated_gradients": ig, "attention_last": attn["last_layer"],
                                                       "attention_rollout": attn["rollout"]},
                                 target, device, tuple(args.ks), seed=i)
            att_show = np.where(special, 0.0, attn["rollout"])
            ex = {
                "row_id": r.row_id, "label": int(r.label), "label_name": names[int(r.label)],
                "pred": target, "pred_name": names[target], "target_name": names[target],
                "correct": bool(r.pred == r.label), "p_target": p, "p_baseline": p0, "ig_delta": delta,
                "truncated": bool(ds_tok.raw_lengths[i] > m["max_length"]), "n_tokens": len(ids),
                "tokens": display_tokens(tok, ids), "token_ids": [int(t) for t in ids], "special": special,
                "ig": ig.tolist(), "ig_norm": normalise(np.where(special, 0.0, ig)).tolist(),
                "attention": {k: v.tolist() for k, v in attn.items() if k in ("last_layer", "mean_layers", "rollout")},
                "attention_norm": normalise(att_show).tolist(), "attention_kind": "rollout",
                "attention_pooling": attn["pooling"],
                "spearman_abs_ig_vs_attention": {
                    k: spearman(np.abs(ig)[content], np.asarray(attn[k])[content])
                    for k in ("last_layer", "mean_layers", "rollout")},
                "faithfulness": faith,
            }
            f.write(json.dumps(ex) + "\n")
            f.flush()
            if (i + 1) % 10 == 0:
                print(f"[xai] {i + 1}/{len(todo)}")

    exs = [json.loads(line) for line in jl.read_text(encoding="utf-8").splitlines()]
    exs = [e for e in exs if e["row_id"] in set(chosen["row_id"])]

    def mean_of(get):
        vals = [v for v in (get(e) for e in exs) if v is not None]
        return {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "n": len(vals)} if vals else None

    methods = ["integrated_gradients", "attention_last", "attention_rollout", "random"]
    faith_summary = {meth: {str(k): {metric: mean_of(lambda e, meth=meth, k=k, metric=metric:
                                                     e["faithfulness"][meth][str(k)][metric])
                                     for metric in ("comprehensiveness", "sufficiency")}
                            for k in args.ks} for meth in methods}
    rows = []
    for e in exs:
        for t, a, s in zip(e["tokens"], e["ig"], e["special"]):
            if not s and t.strip():
                rows.append({"explained_class": e["target_name"], "token": t.strip().lower(), "ig": a})
    tok_df = pd.DataFrame(rows)
    if len(tok_df):
        top = (tok_df.groupby(["explained_class", "token"])["ig"].agg(["mean", "count"]).reset_index()
               .query("count >= @args.min_token_count")
               .sort_values(["explained_class", "mean"], ascending=[True, False]))
        top = top.groupby("explained_class").head(30)
    else:
        top = pd.DataFrame(columns=["explained_class", "token", "mean", "count"])
    top.to_csv(out / "top_tokens.csv", index=False)

    summary = {
        "dataset": ds, "tag": args.tag, "seed": seed, "split": args.split, "n_examples": len(exs),
        "groups": ({f"{k[0]}|{'correct' if k[1] else 'incorrect'}": int(v) for k, v in
                    pd.DataFrame(exs).groupby(["label_name", "correct"]).size().items()} if exs else {}),
        "ig": {"n_steps": args.n_steps, "baseline": "pad tokens (special tokens kept)",
               "target": "predicted class probability", "mean_abs_delta": mean_of(lambda e: e["ig_delta"])},
        "attention_pooling": exs[0]["attention_pooling"] if exs else None,
        "spearman_abs_ig_vs_attention": {k: mean_of(lambda e, k=k: e["spearman_abs_ig_vs_attention"][k])
                                         for k in ("last_layer", "mean_layers", "rollout")},
        "faithfulness": faith_summary,
        "truncated_examples": int(sum(e["truncated"] for e in exs)),
        "notes": ["comprehensiveness: higher = more faithful; sufficiency: lower = more faithful",
                  "an explanation is informative only if it beats the random ranking",
                  "attention is not a faithful explanation by itself (Jain & Wallace, 2019)"],
    }
    save_json(summary, out / "summary.json")
    (out / "examples.html").write_text(page(f"PeerBERT explanations - {ds} seed {seed}", exs[:60]), encoding="utf-8")
    print(f"[xai] {out / 'summary.json'}\n[xai] {out / 'examples.html'}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
# empty package markers
for d, _, _ in os.walk(os.path.join(CODE, "src")):
    open(os.path.join(d, "__init__.py"), "a").close()
sys.path.insert(0, CODE)
print("project code written to", CODE)

## Helpers for running steps and plotting

In [ ]:
import json, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

from src.evaluation.predict import binary_view
from src.evaluation.metrics import binary_metrics

OUT = Path(WORK) / "outputs"
RES = OUT / "results"
FIG = RES / "figures"
FIG.mkdir(parents=True, exist_ok=True)
DD = {"intellabs": "intellabs", "semdetect": "sem_detect"}
NAME = {"intellabs": "IntelLabs", "semdetect": "Sem-Detect"}
TAG = "final"
SEED_ARG = f"--seeds {' '.join(map(str, SEEDS))}" if SEEDS else ""


def run(cmd):
    # Run a pipeline script, streaming its output; stops the notebook on failure.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"step failed (exit code {p.returncode}): {cmd}")


def jload(p):
    p = Path(p)
    return json.loads(p.read_text()) if p.exists() else None


def not_run(what):
    display(Markdown(f"> **{what}: no results yet.** Run the experiment cell above first."))


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.show()


def done_seeds(ds, tag=TAG):
    root = OUT / "models" / DD[ds] / tag
    runs = []
    for d in sorted(root.glob("seed*")):
        info = jload(d / "done.json")
        if info:
            runs.append((info["best_val_macro_f1"], int(d.name[4:])))
    return runs


def ref_seed(ds):
    runs = sorted(done_seeds(ds))
    return runs[(len(runs) - 1) // 2][1] if runs else None   # median validation Macro-F1


def peerbert_pred(ds, seed, split, binary=True):
    p = RES / DD[ds] / "peerbert" / TAG / f"seed{seed}" / f"{split}_predictions.csv"
    if not p.exists():
        return None
    df = pd.read_csv(p, dtype={"row_id": str, "paper_key": str})
    return binary_view(df) if (binary and ds == "semdetect") else df


def ms(d, key):
    v = d.get(key) if d else None
    return f"{v['mean']:.4f} ± {v['std']:.4f}" if v else "–"

---
## Experiment 1: Data, paper-level splits and leakage audit
Loads both datasets from the Hub. It checks every file (labels, review pairs, schema), keeps
each repeated human review once, and splits **by paper**. It then runs the leakage audit,
which blocks training if it fails. If the cell stops with `SchemaError` or `audit FAILED`, read
the message: it lists exactly what was found in the data.

In [ ]:
if RUN["prepare"]:
    run("python scripts/00_inspect_datasets.py --include-semdetect")
    for ds in DATASETS:
        run(f"python scripts/02_prepare_splits.py --dataset {ds}")
    run("python scripts/02_prepare_splits.py --verify-only")

### 1.1 Dataset and split statistics

In [ ]:
rows = []
for ds in DATASETS:
    s = jload(OUT / "manifests" / DD[ds] / "summary.json")
    if not s:
        not_run(f"{NAME[ds]} manifests"); continue
    for part, d in s["full_partitions"].items():
        rows.append({"dataset": NAME[ds], "stage": "full (audited)", "partition": part, "rows": d["rows"],
                     "papers": d["papers"], **{f"label {k}": v for k, v in d["label_counts"].items()}})
    for part, d in s["manifests"].items():
        rows.append({"dataset": NAME[ds], "stage": "used (whole-paper caps)", "partition": part, "rows": d["rows"],
                     "papers": d["papers"], **{f"label {k}": v for k, v in d["label_counts"].items()}})
    ex = pd.Series(s["exclusions"], name="rows").rename_axis("partition:reason").reset_index()
    display(Markdown(f"**{NAME[ds]}: excluded rows (every dropped row is logged)**")); display(ex)
if rows:
    display(Markdown("**Rows, papers and labels per partition** (labels: 0 human, 1 AI, 2 rewrite)"))
    display(pd.DataFrame(rows).fillna(0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, ds in zip(axes, DATASETS):
    p = OUT / "manifests" / DD[ds]
    if not (p / "train.parquet").exists():
        ax.set_axis_off(); ax.set_title(f"{NAME[ds]}: not run"); continue
    counts = {part: pd.read_parquet(p / f"{part}.parquet", columns=["label_name"])["label_name"].value_counts()
              for part in ("train", "val", "test")}
    pd.DataFrame(counts).T.plot.bar(ax=ax, rot=0)
    ax.set_title(f"{NAME[ds]}: class counts per split"); ax.set_ylabel("reviews")
save(fig, "data_class_counts")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, ds in zip(axes, DATASETS):
    p = OUT / "manifests" / DD[ds] / "train.parquet"
    if not p.exists():
        ax.set_axis_off(); continue
    df = pd.read_parquet(p, columns=["text", "label_name"])
    df["words"] = df["text"].str.split().str.len()
    for name, g in df.groupby("label_name"):
        ax.hist(g["words"].clip(upper=g["words"].quantile(0.99)), bins=50, alpha=0.5, label=name, density=True)
    ax.set_title(f"{NAME[ds]}: review length (train)"); ax.set_xlabel("words"); ax.legend()
save(fig, "data_review_length")

### 1.2 Leakage audit

In [ ]:
for ds in DATASETS:
    a = jload(OUT / "audit" / DD[ds] / "leakage_audit.json")
    if not a:
        not_run(f"{NAME[ds]} audit"); continue
    display(Markdown(f"**{NAME[ds]}: overall audit status: `{a['status']}`**"))
    display(pd.DataFrame([{"check": c["check"], "status": c["status"]} for c in a["checks"]]))
    chk = {c["check"]: c for c in a["checks"]}
    lr = []
    for part, res in chk["length_only_auroc"]["results"].items():
        for task, v in res.items():
            if task == "per_generator":
                for g, vv in v.items():
                    if vv: lr.append({"partition": part, "task": f"ai vs human [{g}]", **vv})
            elif v:
                lr.append({"partition": part, "task": task, **v})
    display(Markdown("Length-only AUROC (how well review length alone separates the classes; "
                     "0.5 = no shortcut). This is a WARN-level check, not a blocker."))
    display(pd.DataFrame(lr))

In [ ]:
for ds in DATASETS:
    a = jload(OUT / "audit" / DD[ds] / "leakage_audit.json")
    if not a:
        continue
    lo = {c["check"]: c for c in a["checks"]}["length_only_auroc"]
    worst, overlap = lo["worst"], {c["check"]: c for c in a["checks"]}["paper_overlap"]["overlaps"]["composite"]
    failed = [c["check"] for c in a["checks"] if c["status"] == "FAIL"]
    txt = [f"**Analysis ({NAME[ds]}).** The audit verdict is **{a['status']}**"
           + (f" (failed checks: {', '.join(failed)}; training stays blocked until the data issue is fixed)."
              if failed else ".")]
    txt.append("No paper is shared between train, validation and test." if not any(overlap.values()) else
               "Shared papers between splits: " + ", ".join(f"{k} {v}" for k, v in overlap.items() if v) + ".")
    txt.append(f"The strongest length-only separation is max(AUROC, 1-AUROC) = {worst:.3f}.")
    txt.append("Review length alone is a strong signal here. This motivates the per-length-tercile evaluation "
               "in Experiment 3, which checks that PeerBERT does not rely on it." if worst >= 0.75 else
               "Review length alone separates the classes only weakly.")
    display(Markdown(" ".join(txt)))

---
## Experiment 2: PeerBERT training
ModernBERT-base is fine-tuned separately on each dataset.
- **Loss:** class-weighted cross-entropy + R-Drop consistency.
- **Selection:** early stopping on validation Macro-F1; the best checkpoint is kept.
- **Seeds:** 3 per dataset, reported as mean ± std.
- **Restart-safe:** each finished seed is skipped on re-run, and an interrupted seed resumes from
  its last epoch.

In [ ]:
if RUN["smoke"]:
    run("python scripts/03_train_intellabs.py --smoke")      # about 5 minutes: checks the GPU path end to end
if RUN["train"]:
    if "intellabs" in DATASETS:
        run(f"python scripts/03_train_intellabs.py {SEED_ARG}")
    if "semdetect" in DATASETS:
        run(f"python scripts/04_train_semdetect.py {SEED_ARG}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), squeeze=False)
any_run = False
for col, ds in enumerate(DATASETS):
    for d in sorted((OUT / "models" / DD[ds] / TAG).glob("seed*")):
        h = jload(d / "history.json")
        if not h:
            continue
        any_run = True
        h = pd.DataFrame(h)
        axes[0][col].plot(h["epoch"], h["train_loss"], marker="o", label=d.name)
        axes[1][col].plot(h["epoch"], h["val_macro_f1"], marker="o", label=d.name)
    axes[0][col].set_title(f"{NAME[ds]}: training loss"); axes[1][col].set_title(f"{NAME[ds]}: validation Macro-F1")
    axes[1][col].set_xlabel("epoch")
    for r in (0, 1):
        axes[r][col].legend()
        axes[r][col].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
if any_run:
    save(fig, "train_curves")
else:
    plt.close(fig); not_run("Training")

In [ ]:
rows = []
for ds in DATASETS:
    for d in sorted((OUT / "models" / DD[ds] / TAG).glob("seed*")):
        i = jload(d / "run_info.json")
        if i:
            rows.append({"dataset": NAME[ds], "seed": i["seed"], "best val Macro-F1": i["best_val_macro_f1"],
                         "epochs": i["epochs_run"], "minutes": i["minutes"], "AMP": i["amp"],
                         "dropout": i["model_info"]["dropout_after"].get("attention_dropout"),
                         "R-Drop passes differ": i["rdrop_passes_differ"],
                         "train truncated %": 100 * i["truncation"]["train"]["truncated_fraction"],
                         "median tokens": i["truncation"]["train"]["tokens_median"]})
if rows:
    display(pd.DataFrame(rows))
else:
    not_run("Training")

---
## Experiment 3: PeerBERT evaluation
Evaluated on the official test split (whole-paper capped) and on the shared baseline subset.
- **Thresholds:** 0.5, plus a threshold calibrated on validation.
- **Sem-Detect:** reported natively as 3-class and as a human-vs-AI binary view.
- **Confidence intervals:** 95%, resampling whole papers.

In [ ]:
if RUN["evaluate"]:
    for ds in DATASETS:
        run(f"python scripts/05_evaluate.py --dataset {ds} --n-boot {N_BOOT} {SEED_ARG}")

### 3.1 Main results (mean ± std over seeds)

In [ ]:
rows = []
for ds in DATASETS:
    s = jload(RES / DD[ds] / "peerbert" / TAG / "metrics_summary.json")
    if not s:
        not_run(f"{NAME[ds]} evaluation"); continue
    for split, block in s["per_split"].items():
        for kind in ("binary@0.5", "binary@val_threshold"):
            b = block[kind]
            rows.append({"dataset": NAME[ds], "split": split, "view": kind, "seeds": b["n_seeds"],
                         "Accuracy": ms(b, "accuracy"), "Macro-F1": ms(b, "macro_f1"), "F1 (AI)": ms(b, "f1"),
                         "ROC-AUC": ms(b, "roc_auc"), "PR-AUC": ms(b, "pr_auc")})
        if "native_3class" in block:
            b = block["native_3class"]
            rows.append({"dataset": NAME[ds], "split": split, "view": "3-class (native)", "seeds": b["n_seeds"],
                         "Accuracy": ms(b, "accuracy"), "Macro-F1": ms(b, "macro_f1"),
                         "ROC-AUC": ms(b, "macro_ovr_roc_auc")})
if rows:
    display(pd.DataFrame(rows).fillna("–"))

In [ ]:
rows = []
for ds in DATASETS:
    seed = ref_seed(ds)
    m = jload(RES / DD[ds] / "peerbert" / TAG / f"seed{seed}" / "metrics.json") if seed is not None else None
    if not m or "test" not in m["splits"]:
        continue
    t = m["splits"]["test"]
    ci = t.get("cluster_bootstrap_ci", {})
    b = t["binary@0.5"]
    rows.append({"dataset": NAME[ds], "seed": seed, "n": b["n"],
                 "ROC-AUC [95% CI]": f"{ci['roc_auc']['estimate']:.4f} [{ci['roc_auc']['ci_low']:.4f}, {ci['roc_auc']['ci_high']:.4f}]" if ci else "–",
                 "Macro-F1 [95% CI]": f"{ci['macro_f1@0.5']['estimate']:.4f} [{ci['macro_f1@0.5']['ci_low']:.4f}, {ci['macro_f1@0.5']['ci_high']:.4f}]" if ci else "–",
                 **{f"TPR@{k} FPR": (f"{v['tpr']:.4f}" if v["supported"] else f"n/a (n_neg={v['n_neg']})")
                    for k, v in b["tpr_at_fpr"].items()},
                 "FPR on human": b["ai_fpr"]})
if rows:
    display(Markdown("**Test set: paper-level cluster-bootstrap CIs and detection rates at low false-positive rates** "
                     "(reference seed = median validation Macro-F1)"))
    display(pd.DataFrame(rows))

### 3.2 Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
k = 0
for ds in DATASETS:
    seed = ref_seed(ds)
    m = jload(RES / DD[ds] / "peerbert" / TAG / f"seed{seed}" / "metrics.json") if seed is not None else None
    if not m:
        continue
    views = [("binary@0.5", "binary")] + ([("native_3class", "3-class")] if ds == "semdetect" else [])
    for key, lab in views:
        if k >= 3:
            break
        cm = m["splits"]["test"][key]["confusion_matrix"]
        M = np.array(cm["matrix"]); Mn = M / M.sum(1, keepdims=True).clip(min=1)
        ax = axes[k]; k += 1
        ax.imshow(Mn, cmap="Blues", vmin=0, vmax=1)
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, f"{M[i, j]}\n({Mn[i, j]:.1%})", ha="center", va="center",
                        color="white" if Mn[i, j] > 0.5 else "black", fontsize=9)
        ax.set_xticks(range(len(cm["labels"])), cm["labels"]); ax.set_yticks(range(len(cm["labels"])), cm["labels"])
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{NAME[ds]} ({lab}), seed {seed}")
for ax in axes[k:]:
    ax.set_axis_off()
if k:
    save(fig, "peerbert_confusion")
else:
    plt.close(fig); not_run("Evaluation")

### 3.3 ROC, precision-recall and calibration

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
shown = False
for ds in DATASETS:
    seed = ref_seed(ds)
    p = peerbert_pred(ds, seed, "test") if seed is not None else None
    if p is None or p["label"].nunique() < 2:
        continue
    shown = True
    fpr, tpr, _ = roc_curve(p["label"], p["score"]); axes[0].plot(fpr, tpr, label=NAME[ds])
    pr, rc, _ = precision_recall_curve(p["label"], p["score"]); axes[1].plot(rc, pr, label=NAME[ds])
    fp, mp = calibration_curve(p["label"], p["score"], n_bins=10, strategy="quantile")
    axes[2].plot(mp, fp, marker="o", label=NAME[ds])
axes[0].plot([0, 1], [0, 1], ":", c="grey"); axes[0].set_xscale("log"); axes[0].set_xlim(1e-3, 1)
axes[0].set_xlabel("false positive rate (log)"); axes[0].set_ylabel("true positive rate"); axes[0].set_title("ROC (test)")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].set_title("Precision-recall (test)")
axes[2].plot([0, 1], [0, 1], ":", c="grey"); axes[2].set_xlabel("predicted P(AI)"); axes[2].set_ylabel("observed AI rate")
axes[2].set_title("Calibration (test)")
for ax in axes:
    ax.legend()
if shown:
    save(fig, "peerbert_roc_pr_calibration")
else:
    plt.close(fig); not_run("Evaluation")

### 3.4 Where does PeerBERT work? Generators, venues and review length

In [ ]:
def breakdown_plot(key, title, fname, datasets):
    fig, axes = plt.subplots(1, len(datasets), figsize=(6.5 * len(datasets), 3.8), squeeze=False)
    shown = False
    for ax, ds in zip(axes[0], datasets):
        seed = ref_seed(ds)
        m = jload(RES / DD[ds] / "peerbert" / TAG / f"seed{seed}" / "metrics.json") if seed is not None else None
        if not m or key not in m["splits"]["test"]:
            ax.set_axis_off(); continue
        d = pd.DataFrame(m["splits"]["test"][key]).T
        d[["macro_f1", "roc_auc"]].astype(float).plot.bar(ax=ax, rot=30, ylim=(0, 1.02))
        ax.set_title(f"{NAME[ds]}: {title}"); shown = True
        display(Markdown(f"**{NAME[ds]}: {title}**")); display(d[["n", "n_pos", "n_neg", "macro_f1", "roc_auc", "ai_fpr"]])
    if shown:
        save(fig, fname)
    else:
        plt.close(fig)

breakdown_plot("by_generator@0.5", "per AI generator", "peerbert_by_generator", ["intellabs"])
breakdown_plot("by_conf_year@0.5", "per venue-year", "peerbert_by_venue", DATASETS)
breakdown_plot("by_length_tercile@0.5", "per review-length tercile", "peerbert_by_length", DATASETS)

In [ ]:
for ds in DATASETS:
    seed = ref_seed(ds)
    m = jload(RES / DD[ds] / "peerbert" / TAG / f"seed{seed}" / "metrics.json") if seed is not None else None
    if not m:
        continue
    t = m["splits"]["test"]
    lines = [f"**Analysis ({NAME[ds]}, seed {seed}).** On the test set, PeerBERT reaches Macro-F1 "
             f"{t['binary@0.5']['macro_f1']:.4f} and ROC-AUC {t['binary@0.5']['roc_auc']:.4f} (human vs AI)."]
    lt = pd.DataFrame(t.get("by_length_tercile@0.5", {})).T
    if len(lt):
        spread = lt["macro_f1"].astype(float).max() - lt["macro_f1"].astype(float).min()
        lines.append(f"Across review-length terciles, Macro-F1 varies by {spread:.4f}"
                     + (": performance is stable across lengths." if spread < 0.03
                        else ": performance depends on review length, which is worth discussing."))
    if "by_generator@0.5" in t:
        g = pd.DataFrame(t["by_generator@0.5"]).T["macro_f1"].astype(float)
        lines.append(f"Hardest generator: {g.idxmin()} (Macro-F1 {g.min():.4f}); easiest: {g.idxmax()} ({g.max():.4f}).")
    if "native_3class" in t:
        pc = t["native_3class"]["per_class"]
        lines.append("3-class F1: " + ", ".join(f"{k} {v['f1']:.4f}" for k, v in pc.items()) + ".")
    display(Markdown(" ".join(lines)))

### 3.5 Error analysis (most confident mistakes)

In [ ]:
for ds in DATASETS:
    seed = ref_seed(ds)
    d = RES / DD[ds] / "peerbert" / TAG / f"seed{seed}"
    for name in ("test_binary_false_positives", "test_binary_false_negatives"):
        p = d / f"{name}.csv"
        if p.exists():
            t = pd.read_csv(p)
            display(Markdown(f"**{NAME[ds]}: {name.replace('_', ' ')}** ({len(t)} shown)"))
            display(t.head(5))

---
## Experiment 4: Baselines
Eight detectors run on the **same shared test subset** as PeerBERT. Their thresholds are
calibrated on validation data only.
- **Zero-shot detectors:** LogRank, DetectLLM-LRR, Fast-DetectGPT, Binoculars, DNA-GPT.
- **Feature-based trained classifiers:** GLTR, MFD (fit on the training split only).
- **Pretrained detector:** RADAR.
- **Sem-Detect:** baselines are scored on human/ai reviews only.

The groups are run one after another, and GPU memory is freed between them.

In [ ]:
if RUN["baselines"]:
    groups = ["logrank gltr detectllm_lrr mfd fast_detectgpt", "radar binoculars", "dnagpt"]
    for ds in DATASETS:
        for g in groups:
            run(f"python scripts/06_run_baselines.py --dataset {ds} --only {g}")

In [ ]:
def comparison(ds):
    p = RES / DD[ds] / "comparison_table.csv"
    if not p.exists():
        return None
    t = pd.read_csv(p)
    t = t[t["split"] == "test_baseline"]
    # one PeerBERT row: the reference seed
    seed = ref_seed(ds)
    keep = ~(t["method"].str.startswith("PeerBERT")) | (t["method"] == f"PeerBERT[{TAG}] seed{seed}")
    return t[keep]

for ds in DATASETS:
    t = comparison(ds)
    if t is None:
        not_run(f"{NAME[ds]} comparison"); continue
    cols = ["type", "method", "threshold_kind", "n", "accuracy", "macro_f1", "f1", "roc_auc", "pr_auc",
            "tpr@1%fpr", "tpr@5%fpr", "tpr@10%fpr", "ai_fpr"]
    v = t[t["threshold_kind"] == "val_calibrated"][[c for c in cols if c in t.columns]]
    display(Markdown(f"**{NAME[ds]}: all methods on the shared test subset (validation-calibrated thresholds)**"))
    display(v.sort_values(["type", "macro_f1"], ascending=[True, False]).reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(7 * len(DATASETS), 4.5), squeeze=False)
shown = False
colors = {"trained classifier": "#6B46C1", "zero-shot detector": "#2C7A7B", "pretrained detector": "#C05621"}
for ax, ds in zip(axes[0], DATASETS):
    t = comparison(ds)
    if t is None:
        ax.set_axis_off(); continue
    v = t[t["threshold_kind"] == "val_calibrated"].sort_values("macro_f1")
    ax.barh(v["method"].str.replace(f"PeerBERT[{TAG}]", "PeerBERT", regex=False), v["macro_f1"],
            color=[colors.get(x, "grey") for x in v["type"]])
    for y, (f1, auc) in enumerate(zip(v["macro_f1"], v["roc_auc"])):
        ax.text(f1 + 0.01, y, f"F1 {f1:.3f} | AUC {auc:.3f}", va="center", fontsize=8)
    ax.set_xlim(0, 1.25); ax.set_xlabel("Macro-F1 (validation-calibrated threshold)")
    ax.set_title(f"{NAME[ds]}: shared test subset"); shown = True
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
fig.legend(handles, colors.keys(), loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.04))
if shown:
    save(fig, "baselines_macro_f1")
else:
    plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6.5 * len(DATASETS), 5), squeeze=False)
shown = False
for ax, ds in zip(axes[0], DATASETS):
    seed = ref_seed(ds)
    p = peerbert_pred(ds, seed, "test_baseline") if seed is not None else None
    curves = [("PeerBERT", p)] if p is not None else []
    for d in sorted((RES / DD[ds] / "baselines").glob("*/predictions_test_baseline.csv")):
        curves.append((jload(d.parent / "metrics.json")["display"], pd.read_csv(d, dtype={"row_id": str})))
    for name, c in curves:
        if c["label"].nunique() == 2:
            fpr, tpr, _ = roc_curve(c["label"], c["score"])
            ax.plot(fpr, tpr, lw=2.4 if name == "PeerBERT" else 1.2, label=name); shown = True
    ax.plot([0, 1], [0, 1], ":", c="grey"); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
    ax.set_title(f"{NAME[ds]}: ROC on the shared test subset"); ax.legend(fontsize=8, loc="lower right")
if shown:
    save(fig, "baselines_roc")
else:
    plt.close(fig)

In [ ]:
rows = []
for ds in DATASETS:
    for d in sorted((RES / DD[ds] / "baselines").glob("*/metrics.json")):
        m = jload(d)
        r = {"dataset": NAME[ds], "baseline": m["display"], "type": m["type"], "val threshold": m["val_threshold"]}
        if "orientation" in m["info"]:
            r["note"] = f"RADAR AI output chosen on validation: index {m['info']['orientation']['used_ai_index']}"
        if "rewrite_fraction_flagged_ai@val_threshold" in m:
            rw = m["rewrite_fraction_flagged_ai@val_threshold"]
            r["rewrite flagged as AI"] = f"{rw['fraction']:.1%} of {rw['n_rewrite']}"
        rows.append(r)
if rows:
    display(Markdown("**Baseline details** (Sem-Detect rewrite rows are reported only as the share flagged as AI)"))
    display(pd.DataFrame(rows).fillna(""))

---
## Experiment 5: Ablation
One seed per variant; variants are selected on **validation** Macro-F1 only. The variants are:
- unweighted CE, class-weighted CE, label smoothing, focal loss
- R-Drop α ∈ {0, 0.5, 1}, and dropout off
- the cost-sensitive margin loss with a cosine head (an idea taken from CoCoDet, ACL 2026)

Confirm the selected configuration with all 3 seeds before claiming an improvement.

In [ ]:
if RUN["ablation"]:
    for ds in DATASETS:
        run(f"python scripts/07_run_ablation.py --dataset {ds}")

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6.5 * len(DATASETS), 4), squeeze=False)
shown = False
for ax, ds in zip(axes[0], DATASETS):
    a = jload(RES / DD[ds] / "ablation.json")
    if not a:
        ax.set_axis_off(); not_run(f"{NAME[ds]} ablation"); continue
    d = pd.DataFrame(a["rows"]).sort_values("val_macro_f1")
    ax.barh(d["variant"], d["val_macro_f1"], color=["#D69E2E" if v == a["selected_variant"] else "#A0AEC0"
                                                    for v in d["variant"]])
    lo = d["val_macro_f1"].min()
    ax.set_xlim(max(0, lo - 0.05), 1.0); ax.set_xlabel("validation Macro-F1")
    ax.set_title(f"{NAME[ds]}: ablation (selected: {a['selected_variant']})"); shown = True
    display(Markdown(f"**{NAME[ds]}**")); display(d[::-1].reset_index(drop=True))
if shown:
    save(fig, "ablation")
else:
    plt.close(fig)

---
## Experiment 6: Cross-dataset transfer
Each PeerBERT model is applied to the **other** dataset without retraining (human vs AI only;
Sem-Detect rewrite excluded). Thresholds come from the source validation set. The datasets differ
in generators, label definitions and venues, so a drop is expected; its size is the finding.

In [ ]:
if RUN["cross"]:
    for src in DATASETS:
        run(f"python scripts/08_cross_dataset.py --source {src} {SEED_ARG}")

In [ ]:
rows = []
for src in DATASETS:
    tgt = "semdetect" if src == "intellabs" else "intellabs"
    s_in = jload(RES / DD[src] / "peerbert" / TAG / "metrics_summary.json")
    c = jload(RES / DD[tgt] / f"cross_from_{src}" / "metrics.json")
    if s_in:
        b = s_in["per_split"]["test"]["binary@val_threshold"]
        rows.append({"train on": NAME[src], "test on": NAME[src] + " (in-domain)",
                     "Macro-F1": b["macro_f1"]["mean"], "ROC-AUC": b["roc_auc"]["mean"]})
    if c:
        b = c["mean_std@source_val_threshold"]
        rows.append({"train on": NAME[src], "test on": NAME[tgt] + " (transfer)",
                     "Macro-F1": b["macro_f1"]["mean"], "ROC-AUC": b["roc_auc"]["mean"]})
if rows:
    d = pd.DataFrame(rows); display(d)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    x = np.arange(len(d)); ax.bar(x - 0.18, d["Macro-F1"], 0.36, label="Macro-F1"); ax.bar(x + 0.18, d["ROC-AUC"], 0.36, label="ROC-AUC")
    ax.set_xticks(x, [f"{a}\n-> {b}" for a, b in zip(d["train on"], d["test on"])], fontsize=8); ax.set_ylim(0, 1.05)
    ax.set_title("In-domain vs cross-dataset (mean over seeds)"); ax.legend()
    save(fig, "cross_dataset")
    c = jload(RES / "sem_detect" / "cross_from_intellabs" / "metrics.json") or jload(RES / "intellabs" / "cross_from_semdetect" / "metrics.json")
    if c:
        display(Markdown("**Known mismatches between the datasets:**\n" + "\n".join(f"- {n}" for n in c["mismatch_notes"])))
else:
    not_run("Cross-dataset transfer")

---
## Experiment 7: Explainability
For reviews from the shared test subset (balanced over true class × correct/incorrect), the
model's decision is explained with:
- **Integrated Gradients** on the token embeddings (signed, with a completeness check);
- **attention**: last layer, mean over layers, and rollout.

**Faithfulness** (ERASER): remove the top-k% tokens (comprehensiveness, higher is better) or keep
only them (sufficiency, lower is better). A random ranking is the reference; an explanation is
only useful if it beats random.

In [ ]:
if RUN["xai"]:
    for ds in DATASETS:
        run(f"python scripts/10_explain.py --dataset {ds} --n-per-group {XAI_PER_GROUP}")

In [ ]:
fig, axes = plt.subplots(len(DATASETS), 2, figsize=(12, 3.8 * len(DATASETS)), squeeze=False)
shown = False
for r, ds in enumerate(DATASETS):
    xs = sorted((RES / DD[ds] / "xai" / TAG).glob("seed*/summary.json"))
    if not xs:
        axes[r][0].set_axis_off(); axes[r][1].set_axis_off(); not_run(f"{NAME[ds]} XAI"); continue
    x = jload(xs[0]); shown = True
    rows = []
    for meth, byk in x["faithfulness"].items():
        for k, v in byk.items():
            rows.append({"method": meth, "k": float(k), "comprehensiveness": v["comprehensiveness"]["mean"],
                         "sufficiency": v["sufficiency"]["mean"]})
    f = pd.DataFrame(rows)
    for c, metric in enumerate(("comprehensiveness", "sufficiency")):
        f.pivot(index="method", columns="k", values=metric).plot.bar(ax=axes[r][c], rot=0)
        axes[r][c].set_title(f"{NAME[ds]}: {metric} ({'higher' if c == 0 else 'lower'} = more faithful)")
        axes[r][c].axhline(0, c="grey", lw=0.8)
    display(Markdown(f"**{NAME[ds]}** (seed {x['seed']}, {x['n_examples']} reviews; mean |IG delta| "
                     f"{x['ig']['mean_abs_delta']['mean']:.4f}; Spearman(|IG|, attention): " +
                     ", ".join(f"{k} {v['mean']:.3f}" for k, v in x["spearman_abs_ig_vs_attention"].items() if v) + ")"))
    display(f.sort_values(["k", "comprehensiveness"], ascending=[True, False]).reset_index(drop=True))
    kmin = f["k"].min()
    best = f[f["k"] == kmin].set_index("method")
    if "random" in best.index:
        beats = [m for m in best.index if m != "random" and best.loc[m, "comprehensiveness"] > best.loc["random", "comprehensiveness"]]
        display(Markdown(f"**Analysis.** At k = {kmin:.0%}, methods that beat the random ranking on "
                         f"comprehensiveness: {', '.join(beats) or 'none'}."))
if shown:
    save(fig, "xai_faithfulness")
else:
    plt.close(fig)

In [ ]:
for ds in DATASETS:
    xs = sorted((RES / DD[ds] / "xai" / TAG).glob("seed*/top_tokens.csv"))
    if not xs:
        continue
    t = pd.read_csv(xs[0])
    classes = list(t["explained_class"].unique())
    if not classes:
        continue
    fig, axes = plt.subplots(1, len(classes), figsize=(5 * len(classes), 4.5), squeeze=False)
    for ax, c in zip(axes[0], classes):
        d = t[t["explained_class"] == c].nlargest(15, "mean").iloc[::-1]
        ax.barh(d["token"], d["mean"], color="#B83280"); ax.set_title(f"{NAME[ds]}: tokens pushing to '{c}'")
        ax.set_xlabel("mean Integrated Gradients attribution")
    save(fig, f"xai_top_tokens_{ds}")

In [ ]:
for ds in DATASETS:
    xs = sorted((RES / DD[ds] / "xai" / TAG).glob("seed*/examples.html"))
    if xs:
        display(Markdown(f"**{NAME[ds]}: explained reviews** (red: pushes towards the predicted class, "
                         f"blue: against it). Full page: `{xs[0]}`"))
        html = xs[0].read_text()
        display(HTML(f"<div style='max-height:600px;overflow:auto;border:1px solid #ddd;padding:8px'>{html}</div>"))

---
## 8. Final report and significance tests
This step collects every result into thesis-ready tables (CSV + LaTeX), plus paired tests of
PeerBERT against each baseline on the same reviews. The paired tests use a paper-level bootstrap
on ROC-AUC and McNemar, with Holm correction.

In [ ]:
run(f"python scripts/09_build_report.py --n-boot {N_BOOT}")
for ds in DATASETS:
    p = RES / "report" / f"{ds}_paired_tests.csv"
    if p.exists():
        t = pd.read_csv(p)
        display(Markdown(f"**{NAME[ds]}: PeerBERT vs each baseline (positive difference = PeerBERT better)**"))
        display(t)
        sig = t.loc[t["p_bootstrap_holm"] < 0.05, "baseline"].tolist() if "p_bootstrap_holm" in t else []
        display(Markdown(f"Significant after Holm correction (α = 0.05): {', '.join(sig) or 'none'}."))
display(Markdown((RES / "report" / "report.md").read_text()))
print("\nFigures (PNG + PDF):", FIG)
for f in sorted(FIG.glob("*.png")):
    print("  ", f.name)
print("Tables (CSV + LaTeX):", RES / "report")